# [실습] 80% 구간 포함률 — 주를 나눠 구하고 비교하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 80% 구간을 인력 계획에 믿고 쓸까

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">딥러닝 시계열 · 넷째 날</div>
<h1 style="color:#000000;margin:10px 0">80% 구간 포함률 — 주를 나눠 구하고 비교하기</h1>
<div style="font-size:1.05em;color:#656565">Chronos-2 두 모델의 분위수 열로 80% 구간을 꺼내, 그 포함률과 평균 구간 길이를 공휴일 없는 두 주와 공휴일 있는 세 주로 나눠 구합니다</div>
</div>

**오늘의 질문** 공휴일 열을 입력한 Chronos-2 가 낸 80% 구간을 다음 주 역별 인력 계획에 그대로 써도 될까요?

두 모델은 Chronos-2 제로샷과 공변량을 배울 때 평가표에 MASE 를 적은 Chronos-2 입니다. **제로샷**(평가표의 「Chronos-2 제로샷」)은 가중치를 고치지 않고 사전학습한 그대로 예측한 Chronos-2 입니다. **Chronos-2 + 공휴일**은 공휴일 열을 과거 표 `df` 와 미래 입력 `future_df` 에 함께 포함한 Chronos-2 입니다.

둘 다 원점 전날까지 512일(컨텍스트)을 입력하면 역·날마다 분위수 아홉 열 `"0.1"`～`"0.9"` 를 냅니다. 지금까지는 그 가운데 `"0.5"` 열만 점 예측으로 채점했습니다.

80% 구간은 분위수 0.1 열과 0.9 열 사이입니다. 대상은 서울 지하철 100개 역(2024년까지 하루 평균 승차가 가장 많은 역 100곳)입니다.

**예측 원점(forecast origin)** 하나마다 100개 역 × 7일 = 700점이 있습니다. 그 가운데 실제 승차가 구간 안에 있는 점의 비율이 **포함률(empirical coverage)** 입니다. 채점 7일에 평일 공휴일이 없는 원점 2개를 「공휴일 없는 두 주」, 있는 원점 3개를 「공휴일 있는 세 주」라 부릅니다. 포함률은 이 두 주와 세 주에서 따로 구하고, 평균 구간 길이와 함께 봅니다.

1. 런타임 유형을 GPU(T4)로 두고 맨 위 설치 셀을 한 번 실행합니다. GPU 가 없어도 돌지만 「오늘의 준비」 셀이 약 20초 대신 약 100초 걸립니다.
2. 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 설치 셀 바로 아래의 접힌 `[채점]` 셀을 한 번 ▶ 실행합니다.
3. 「오늘의 준비」 셀을 실행해 두고, 출력을 기다리는 동안 개념 확인 두 문항을 풉니다.

<img src="https://en.wikipedia.org/wiki/Special:FilePath/Seoul_subway_linemap_ko.svg" alt="서울 지하철 노선도. 호선마다 색이 다른 선 위에 역 이름이 이어진 그림" width="560" height="637"/>
<p align="center"><em>▲ 서울 지하철 노선도. 호선마다 색이 다른 선 위에 역이 이어진다. 100개 역은 이 노선도의 역 가운데 2024년까지 평균 승차가 많은 역이다 (출처: Wikimedia Commons, Seoul subway linemap ko.svg)</em></p>

In [ ]:
# 실습 준비 - 패키지 설치·폰트·GPU 확인·Chronos-2 내려받기
# [heavy]
#  Colab 이거나 내 컴퓨터에 두 패키지가 없으면 chronos-forecasting 2.3.1 · holidays 0.103 을 설치합니다.
#  교안의 수를 계산한 버전에 맞춰 고정합니다.
#  numpy · pandas · pyarrow · torch · matplotlib 은 Colab 기본 버전을 그대로 씁니다.
import os, sys, time, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
_missing = [m for m in ("chronos", "holidays") if importlib.util.find_spec(m) is None]
if IN_COLAB or _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "chronos-forecasting==2.3.1", "holidays==0.103"], check=True)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil
if IN_COLAB and shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 기준선=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다
FIGW = FIG_W      # 도우미가 밑줄을 빼고 FIGW 로 옮겨 적는 일이 있어 같은 값을 하나 더 둔다

try:
    import holidays
    _hol_ver = holidays.__version__
except ImportError:
    _hol_ver = "없음"
print("사용 폰트:", FONT, "| 색 팔레트 준비 완료 | holidays", _hol_ver)

# GPU 확인 — Chronos-2 예측은 여기서 정한 DEVICE 에서 실행됩니다
#  정밀도는 32비트 실수(fp32)로 두고, A100·L4 의 TF32 행렬곱을 꺼 GPU 종류에 따른 차이를 줄입니다.
#  오늘 쓰는 Chronos-2 분위수는 Colab T4 와 CPU 에서 원점별 포함률이 같게 나왔습니다.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
if DEVICE == "cuda":
    _gpu = torch.cuda.get_device_properties(0)
    print(f"DEVICE : cuda | GPU {torch.cuda.get_device_name(0)} | 메모리 {_gpu.total_memory / 1e9:.1f} GB"
          f" | torch {torch.__version__}")
else:
    print(f"DEVICE : cpu | GPU 없음 | torch {torch.__version__}")
    print("GPU 가 연결되지 않았습니다. 메뉴 런타임 → 런타임 유형 변경 → GPU 를 고른 뒤 맨 위 셀부터 다시 실행합니다.")
    print("그대로 두면 같은 코드가 CPU 에서 실행되고, 「오늘의 준비」 셀이 약 100초 걸립니다.")

# Chronos-2 가중치(약 480MB)를 여기서 한 번 내려받아 둡니다. 「오늘의 준비」 셀과 미션 준비 셀은 이 사본을 읽습니다.
#  내려받기가 끊기면 10초 쉬고 다시 시도하며, 세 번까지 합니다.
from huggingface_hub import snapshot_download
CHRONOS_ID = "amazon/chronos-2"
for _try in range(1, 4):
    try:
        snapshot_download(CHRONOS_ID, allow_patterns=["config.json", "model.safetensors"])
        print(f"Chronos-2 가중치 준비 완료 : {CHRONOS_ID} ({_try}번째 시도)")
        break
    except Exception as _e:
        print(f"Chronos-2 내려받기 {_try}번째 실패 : {type(_e).__name__}")
        if _try < 3:
            time.sleep(10)
else:   # 받지 못한 채 넘어가면 뒤 셀의 from_pretrained 가 내려받기 오류로 멈춘다. 여기서 멈춘다
    raise RuntimeError("Chronos-2 가중치를 세 번 모두 받지 못했습니다. LMS 면 이 셀을 다시 실행하고, 그래도 같으면"
                       " 강사에게 알립니다. Colab 이면 메뉴 런타임 → 런타임 다시 시작을 누른 뒤 이 셀부터 다시 실행합니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 받아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
_G0 = dict(globals())                     # 이 셀이 만들기 전의 이름 — 맨 아래에서 이 셀이 만든 이름을 가려냅니다
_CANON = "0.4.2"                          # 모든 실습이 함께 쓰는 채점 코드의 판 번호입니다

import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt

#  채점 코드는 아래 별칭만 씁니다 — 학생 셀의 `from numpy import *`(min·max·any·round·sum·abs·bool)와
#  `from pandas import *`(io) 가 같은 이름을 덮어도, 학생이 re·json 같은 이름에 값을 넣어도 채점은 그대로 돕니다.
import base64 as _b64
import builtins as _bi
import io as _io
import json as _json
import re as _re
import sys as _sys
import matplotlib.pyplot as _plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    Colab 에서도 한글이 □ 로 깨진다.
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not _bi.any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        _plt.rcParams["font.family"] = _font
        _plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(_json.loads(_b64.b64decode(s).decode("utf-8")))


#  아래 두 표지 줄 사이만 실습마다 다릅니다 — _ANS · _REVEAL · _HINT (· _CAUSE) 블롭, 튜터 질문 틀(_ASK · _ASK_HEAD ·
#  _ASK_FORM), 빈 PROMPT 에 잠그는 미션 `_blocks(key, reason)`, 권 설정 `_BOOK`(minus · skip · spec · pre · checks ·
#  error_hint), 그 실습에만 있는 함수. 표지 줄 밖은 모든 실습이 같은 채점 코드입니다.
# ━━ 이 실습의 채점 기준과 설정 — 이 줄부터 「채점 기준 끝」 줄까지만 실습마다 다릅니다 ━━
# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
               "eyJtMSI6eyJ2YWx1ZXMiOnsi6rO17Zy07J28IOyXhuuKlCDrkZAg7KO8IO2PrO2VqOuloCI6Wzc2LjAsODMuMCwiJSJdLCLqs7Xtn"
               "LTsnbwg7J6I64qUIOyEuCDso7wg7Y+s7ZWo66WgIjpbNjIuMCw3MS4wLCIlIl19LCJwbG90Ijp7Iuy2leudvOuyqCI6dHJ1ZX19LC"
               "JtMiI6eyJ2YWx1ZXMiOnsiQ2hyb25vcy0yICsg6rO17Zy07J287J2YIOqzte2ctOydvCDsnojripQg7IS4IOyjvCDtj4nqt6Ag6rW"
               "s6rCEIOq4uOydtCI6WzUzMDAsNjIwMCwi66qFIl0sIuqzte2ctOydvCDsnojripQg7IS4IOyjvCDqtazqsIQg6ri47J20IOu5hCI6"
               "WzEuMzgsMS42Niwi67CwIl19LCJwbG90Ijp7Iuy2leudvOuyqCI6dHJ1ZX19fQ==")


_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U"
               "65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg"
               "UV9IT0wsIFlfVEVTVCwgT1JJR0lOUywgUExBSU5fT1JJR0lOUywgSE9MX09SSUdJTlMg7JmAIOyEpOy5mCDshYDsnZggRklHX1cs"
               "IENPTCDsnYAg7J2066+4IOyeiOycvOuLiCDri6Tsi5wg66eM65Ok7KeAIOuniC4gT1JJR0lOUyDsm5DsoJAgNeqwnOuniOuLpCBZ"
               "X1RFU1QKICAgIOqwgCBRX0hPTCDsnZggWy4uLiwgMF0g7J207IOB7J206rOgIFsuLi4sIC0xXSDsnbTtlZjsnbgg7KCQ7J2EIOyE"
               "uOyWtCwg6rWs6rCEIOyViMK37JWE656YwrfsnIQg7KCQIOyImOyZgCDtj6ztlajrpaAoJSnsnYQg7JuQ7KCQ66eI64ukIO2VnCDt"
               "lonslKkg7LCN7Ja0IOykmC4g7JuQ7KCQ67OEIO2PrO2VqOuloOydhCBQTEFJTl9PUklHSU5TIOyZgAogICAgSE9MX09SSUdJTlMg"
               "66GcIOuCmOuIoCDtj4nqt6DtlZjqs6AgIuqzte2ctOydvCDsl4bripQg65GQIOyjvCDtj6ztlajrpaAgPSDqsJIgJSIg7JmAICLq"
               "s7XtnLTsnbwg7J6I64qUIOyEuCDso7wg7Y+s7ZWo66WgID0g6rCSICUiIOulvCDshozsiJgg7LKr7Ke4IOyekOumrOuhnCDtlZwg"
               "7ZaJ7JSpIOy2nOugpe2VtCDspJguIOybkOygkCA16rCc7J2YIO2PrO2VqOuloCDrp4nrjIDrpbwKICAgIOqzte2ctOydvOydtCDs"
               "nojripQg7KO87JmAIOyXhuuKlCDso7zqsIAg64uk66W4IOyDieydtCDrkJjqsowg6re466as6rOgLCA4MCUg7JeQIOqwgOuhnCDs"
               "oJDshKDqs7wg6rCA66Gc7LaVICLsmIjsuKEg7JuQ7KCQIiwg7IS466Gc7LaVICLtj6ztlajrpaAgKCUpIiDsnYQg64us7JWEIOyk"
               "mC4KCiAgICBmcm9tIG1hdHBsb3RsaWIucGF0Y2hlcyBpbXBvcnQgUGF0Y2gKCiAgICBjb3YgPSB7fQogICAgZm9yIG8gaW4gT1JJ"
               "R0lOUzoKICAgICAgICBxLCB5ID0gUV9IT0xbb10sIFlfVEVTVFtvXQogICAgICAgIGxvLCBoaSA9IHFbLi4uLCAwXSwgcVsuLi4s"
               "IC0xXSAgICAgICAgICAgICAgICAgICMgODAlIOq1rOqwhOydmCDrkZAg64GdID0gMC4xIOyXtCDCtyAwLjkg7Je0CiAgICAgICAg"
               "bl9iZWxvdyA9IGludCgoeSA8IGxvKS5zdW0oKSkKICAgICAgICBuX2Fib3ZlID0gaW50KCh5ID4gaGkpLnN1bSgpKQogICAgICAg"
               "IG5faW4gPSB5LnNpemUgLSBuX2JlbG93IC0gbl9hYm92ZSAgICAgICAgICAgICAgICMg64Gd6rO8IOqwmeydgCDqsJLsnYAg6rWs"
               "6rCEIOyViOycvOuhnCDshLzri6QKICAgICAgICBjb3Zbb10gPSBuX2luIC8geS5zaXplICogMTAwCiAgICAgICAgcHJpbnQoZiJ7"
               "b30gwrcgNzAw7KCQIDog6rWs6rCEIOyViCB7bl9pbn3soJAgwrcg7JWE656YIHtuX2JlbG93feygkCDCtyDsnIQge25fYWJvdmV9"
               "7KCQIMK3IO2PrO2VqOuloCB7Y292W29dOi4xZn0gJSIpCgogICAgcGxhaW4gPSBucC5tZWFuKFtjb3Zbb10gZm9yIG8gaW4gUExB"
               "SU5fT1JJR0lOU10pICAgIyDsm5DsoJDrp4jri6QgNzAw7KCQ7J206528IOybkOygkCDtj4nqt6AgPSDsoJDsnYQg7ZWp7LOQIOq1"
               "rO2VnCDqsJIKICAgIGhvbGkgPSBucC5tZWFuKFtjb3Zbb10gZm9yIG8gaW4gSE9MX09SSUdJTlNdKQogICAgcHJpbnQoZiLqs7Xt"
               "nLTsnbwg7JeG64qUIOuRkCDso7wg7Y+s7ZWo66WgID0ge3BsYWluOi4xZn0gJSIpCiAgICBwcmludChmIuqzte2ctOydvCDsnojr"
               "ipQg7IS4IOyjvCDtj6ztlajrpaAgPSB7aG9saTouMWZ9ICUiKQoKICAgIGNvbG9ycyA9IFtDT0xbIuuCqOyDiSJdIGlmIG8gaW4g"
               "UExBSU5fT1JJR0lOUyBlbHNlIENPTFsi7ZWY64qY7IOJIl0gZm9yIG8gaW4gT1JJR0lOU10KICAgIGZpZywgYXggPSBwbHQuc3Vi"
               "cGxvdHMoZmlnc2l6ZT0oRklHX1cgKiAwLjgsIDMuNikpCiAgICBheC5iYXIoT1JJR0lOUywgW2NvdltvXSBmb3IgbyBpbiBPUklH"
               "SU5TXSwgY29sb3I9Y29sb3JzKQogICAgYXguYXhobGluZSg4MCwgY29sb3I9Q09MWyLtmozsg4kiXSwgbHc9MS4wLCBscz0iLS0i"
               "KSAgICAjIOuqheuqqSDtj6ztlajrpaAgODAlCiAgICBheC5zZXRfeWxpbSgwLCAxMTApICAgICAgICAgICAgICAgICAgICAgICAg"
               "ICAgICAgICAgICMgODAlIOyEoCDsnITsl5Ag67KU66GA6rCAIOuTpOyWtOqwiCDqs7XqsITsnYQg64Ko6ri064ukCiAgICBheC5z"
               "ZXRfeGxhYmVsKCLsmIjsuKEg7JuQ7KCQIikKICAgIGF4LnNldF95bGFiZWwoIu2PrO2VqOuloCAoJSkiKQogICAgYXgubGVnZW5k"
               "KGhhbmRsZXM9W1BhdGNoKGNvbG9yPUNPTFsi64Ko7IOJIl0sIGxhYmVsPSLqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wiKSwKICAg"
               "ICAgICAgICAgICAgICAgICAgICBQYXRjaChjb2xvcj1DT0xbIu2VmOuKmOyDiSJdLCBsYWJlbD0i6rO17Zy07J28IOyeiOuKlCDs"
               "hLgg7KO8IildLCBsb2M9InVwcGVyIGNlbnRlciIsIG5jb2w9MiwgZnJhbWVvbj1GYWxzZSkKICAgIHBsdC5zaG93KCkKCu2VtOyE"
               "pAoKLSDsmIjsg4Eg7KCB6riw7J2YIOuLteydgCDshYvsp7jsnoXri4jri6QuIENocm9ub3MtMiArIOqzte2ctOydvOydmCDtj6zt"
               "lajrpaDsnYAg6rO17Zy07J28IOyXhuuKlCDrkZAg7KO8IDc5LjUlLCDqs7XtnLTsnbwg7J6I64qUIOyEuCDso7wgNjYuNCUg7J6F"
               "64uI64ukLiDqs7XtnLTsnbwg7JeG64qUIOuRkCDso7zripQg66qF66qpIDgwJSDsmYAg6rGw7J2YIOqwmeqzoCwg6rO17Zy07J28"
               "IOyeiOuKlCDshLgg7KO864qUIDgwIOKIkiA2Ni40ID0gMTMuNiVwIOuqqOyekOuejeuLiOuLpC4KLSDsm5DsoJDrs4TroZwg67O0"
               "66m0IOqzte2ctOydvCDsnojripQg7IS4IOyjvOuKlCAyMDI1LTA4LTEzIDc4LjklLCAyMDI1LTEwLTAxIDYxLjQlLCAyMDI2LTAy"
               "LTExIDU5LjAlIOyeheuLiOuLpC4g7JuQ7KCQIDIwMjYtMDItMTEo6rO17Zy07J28IOyeiOuKlCDso7wp7J2AIDcwMOygkCDqsIDs"
               "mrTrjbAgMjYy7KCQ7J20IOyVhOuemCDrgZ3rs7Tri6Qg7KCB6rOgIDI17KCQ66eMIOychCDrgZ3rs7Tri6Qg66eO7JWELCDrspfs"
               "lrTrgpwg7KCQ7J20IOyVhOuemOuhnCDrqrDroLjsirXri4jri6QuCi0g6rO17Zy07J28IOyXhuuKlCDrkZAg7KO864qUIDEsNDAw"
               "7KCQIOqwgOyatOuNsCAyODfsoJDsnbQg6rWs6rCEIOuwluydtOyXiOqzoCwg6re4IOqwgOyatOuNsCAyNjXsoJDsnbQg7JyEIOuB"
               "neuztOuLpCDrp47slZjsirXri4jri6Qo7JuQ7KCQIDIwMjUtMTItMTAgMTI47KCQIMK3IDIwMjYtMDYtMjQgMTM37KCQKS4gMC45"
               "IOyXtOyXkCDrp57stpgg7J2466Cl7J20IOuqqOyekOuegCDsoJDsnbQgMjY1IMO3IDEsNDAwIOKJiCAxOC45JSDroZwsIO2AtOym"
               "iCDjgIwwLjkg7Je07JeQIOunnuy2mCDsnbjroKXsnbQg66qo7J6Q656EIO2ZleuloOOAjeydmCAxMCUg67O064ukIOunjuyKteuL"
               "iOuLpC4g7JWE656YIOuBneuztOuLpCDsoIHsnYAg7KCQ7J2AIDIy7KCQ67+Q7J6F64uI64ukLiDqt7jrnpjrj4QgNzkuNSUg64qU"
               "IOuqheuqqSA4MCUg7JmAIOq1rOuzhO2VoCDsiJgg7JeG7Ja0IOq1rOqwhOydgCDqt7jrjIDroZwg7JSB64uI64ukLiAxOC45JSDr"
               "ipQg65GQIOyjvOyXkOyEnOunjCDrgpjsmKgg6rCS7J206528IOychCDrgZ3snYAg67CU6r647KeAIOyViuqzoCDqtazqsIQg7JiG"
               "7JeQIO2VqOq7mCDsoIHsirXri4jri6QuCi0g7Z2U7ZWcIOyLpOyImCDri6TshK86IDAuMiDsl7Tqs7wgMC44IOyXtOuhnCDqtazt"
               "lZjrqbQgNjAuOSUgwrcgNDYuMSUg66GcIDYwJSDqtazqsITsnZgg7Y+s7ZWo66Wg7J20IOuCmOyYteuLiOuLpC4g7JWE656YIOuB"
               "neqzvCDsnIQg64Gd7J2EIOuwlOq/lCDsk7DrqbQgYCZgIOuhnCDrrLbripQg7L2U65Oc64qUIDAlLCDsnIQg7JmE7ISx67O47LKY"
               "65+8IOuwluydmCDsoJDsnYQg67m864qUIOy9lOuTnOuKlCDqtazqsIQg7JWIIOygkOydhCDrkZAg67KIIOu5vOyEnCDiiJI3OS41"
               "JSDqsIAg65Cp64uI64ukLiDrkZAg7KGw6rG07J2EIGB8YCDroZwg66y27Jy866m0IDEwMCUg6rCAIOuQqeuLiOuLpC4g7JuQ7KCQ"
               "IDXqsJzrpbwg7ZWcIOuyiOyXkCDtj4nqt6DtlZjrqbQgMiw1MDggw7cgMyw1MDAgPSA3MS43JSDtlZjrgpjrp4wg64Ko7JWEIDY2"
               "LjQlIOqwgCDrk5zrn6zrgpjsp4Ag7JWK7Iq164uI64ukKOuwmOyYrOumvO2VnCDrkZAg6rCS7Jy866GcIOqzhOyCsO2VmOuptCA3"
               "MS42KS4g7KCc66Gc7IO3IGBRX1pTYCDroZwg6rWs7ZWY66m0IO2RnCAzIOqzvCDqsJnsnYAgODUuMSUgwrcgNDEuNiUg7J6F64uI"
               "64ukLgotIOydtOuyiCDso7zsl5Ag7ZWgIOydvDog7KO86rCEIOyduOugpSDqs4Ttmo3tkZzsnZgg65GQIOuBnSDsmIbsl5DripQg"
               "6rWs6rCE7J2EIOyTsOuKlCDrgqDsnZgg7Y+s7ZWo66Wg7J2EIOyggeyKteuLiOuLpC4g6rO17Zy07J28IOyXhuuKlCDso7zripQg"
               "NzkuNSUg7JmAIOychCDrgZ3rs7Tri6Qg66eO7J2AIOygkCAxOC45JSDrpbwg7KCB7Iq164uI64ukLiDtj4nsnbwg6rO17Zy07J28"
               "7J20IOyeiOuKlCDso7zripQg7IS4IOyjvCAyLDEwMOygkOyXkOyEnCDri7nsnbwgNjAw7KCQ7J2EIOu6gCAxLDUwMOygkCDqsIDs"
               "mrTrjbAgMSwxMTbsoJDsnbQg6rWs6rCEIOyViOydtOudvCA3NC40JSDsnoXri4jri6QuIOyghOuLrO2VoCDqtazqsITsnYAg7JO4"
               "IOyImCDsnojripQg6rWs6rCE64G866asIOu5hOq1kO2VtCDsoJXtlbQsIOygnOuhnOyDtyg0MS42JSnrs7Tri6Qg7Y+s7ZWo66Wg"
               "7J20IOuGkuydgCDsnbQg6rWs6rCE7J2EIO2PrO2VqOuloOqzvCDtlajqu5gg7KCE64us7ZWp64uI64ukLgoK7L2U65Oc6rCAIOuL"
               "teydhCDrp4zrk5zripQg67Cp7IudLiBgUV9IT0xbb11gIOydmCDshYvsp7gg7LCo7JuQ7J2AIDAuMSDsl7TrtoDthLAgMC45IOyX"
               "tOq5jOyngCDslYTtmYkg6rCc6528IGBbLi4uLCAwXWAg7J20IOyVhOuemCDrgZ0sIGBbLi4uLCAtMV1gIOydtCDsnIQg64Gd7J6F"
               "64uI64ukLiDslYTrnpgg64Gd67O064ukIOyggeydgCDsoJDqs7wg7JyEIOuBneuztOuLpCDrp47snYAg7KCQ7J2EIOu5vOuptCDq"
               "tazqsIQg7JWI7J2YIOygkCDsiJjqsIAg64Ko6rOgLCDrgZ3qs7wg6rCZ7J2AIOqwkuydgCDslYjsnLzroZwg7IWJ64uI64ukLiDs"
               "i6TsoJwg7Iq57LCo64qUIOygleyImOydtOqzoCDrtoTsnITsiJjripQg7IaM7IiY6528LCDsnbQg7J6Q66OM7JeQ7ISc64qUIOuB"
               "neqzvCDqsJnsnYAg6rCS7J2EIOuwluycvOuhnCDshLjrj4QoYD5gwrdgPGApIOqwmeydgCDqsJLsnbQg64KY7Ji164uI64ukLiDs"
               "m5DsoJDrp4jri6Qg7KCQ7J20IDcwMOqwnOuhnCDqsJnslYQsIOybkOygkOuzhCDtj6ztlajrpaDsnYQg65GQIOyjvOyZgCDshLgg"
               "7KO87JeQ7IScIOqwgeqwgSDtj4nqt6DtlZwg6rCS7J2AIOq3uCDsoJDsnYQg7ZWp7LOQIOq1rO2VnCDqsJLqs7wg6rCZ7Iq164uI"
               "64ukLgo="),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U"
               "65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg"
               "UV9aUywgUV9IT0wsIFBMQUlOX09SSUdJTlMsIEhPTF9PUklHSU5TIOyZgCDshKTsuZgg7IWA7J2YIEZJR19XLCBDT0wg7J2AIOyd"
               "tOuvuCDsnojsnLzri4gg64uk7IucIOunjOuTpOyngCDrp4guIOuRkCDrqqjrjbjrp4jri6Qg7JuQ7KCQ67OE66GcIHFbLi4uLCAt"
               "MV0gLSBxWy4uLiwgMF0KICAgIOydhCA3MDDsoJDsnLzroZwg7Y+J6reg7ZWY6rOgLCDqt7gg6rCS7J2EIFBMQUlOX09SSUdJTlMg"
               "7JmAIEhPTF9PUklHSU5TIOuhnCDrgpjriKAg7Y+J6reg7ZW07IScIOqzte2ctOydvCDsnKDrrLQgw5cg65GQIOuqqOuNuOydmCDt"
               "j4nqt6Ag6rWs6rCEIOq4uOydtOulvCDtlZwg7ZaJ7JSpIOywjeyWtCDspJguICJDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnZgKICAg"
               "IOqzte2ctOydvCDsnojripQg7IS4IOyjvCDtj4nqt6Ag6rWs6rCEIOq4uOydtCA9IOqwkiDrqoUiIOydhCDsoJXsiJjroZwsIOqz"
               "te2ctOydvCDsnojripQg7IS4IOyjvOyXkOyEnCBDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnYQg7KCc66Gc7IO37Jy866GcIOuCmOuI"
               "iCAi6rO17Zy07J28IOyeiOuKlCDshLgg7KO8IOq1rOqwhCDquLjsnbQg67mEID0g6rCSIOuwsCIg66W8IOyGjOyImCDrkZjsp7gg"
               "7J6Q66as66GcCiAgICDtlZwg7ZaJ7JSpIHByaW50IO2VtCDspJguIOq3uOuemO2UhOuKlCDtlZwg7J6l66eMLCDqs7XtnLTsnbwg"
               "7JeG64qUIOuRkCDso7zsmYAg6rO17Zy07J28IOyeiOuKlCDshLgg7KO866eI64ukIOuRkCDrqqjrjbjsnZgg66eJ64yA66W8IOuC"
               "mOuegO2eiCDqt7jroKQg66eJ64yAIDTqsJzroZwg7ZWY6rOgCiAgICDqsIDroZzstpUg7J2066aEICLssYTsoJDtlZwg7KO8Iiwg"
               "7IS466Gc7LaVIOydtOumhCAi7Y+J6regIOq1rOqwhCDquLjsnbQgKOuqhSkiLCDrspTroYDripQg65GQIOuqqOuNuOuhnCDtlbQg"
               "7KSYLgoKICAgIGRlZiBtZWFuX3dpZHRoKFEsIG9yaWdpbnMpOgogICAgICAgIHJldHVybiBucC5tZWFuKFsoUVtvXVsuLi4sIC0x"
               "XSAtIFFbb11bLi4uLCAwXSkubWVhbigpIGZvciBvIGluIG9yaWdpbnNdKSAgICMgMC45IOyXtCAtIDAuMSDsl7QsIOuqhQoKICAg"
               "IGdyb3VwcyA9IHsi6rO17Zy07J28IOyXhuuKlCDrkZAg7KO8IjogUExBSU5fT1JJR0lOUywgIuqzte2ctOydvCDsnojripQg7IS4"
               "IOyjvCI6IEhPTF9PUklHSU5TfQogICAgdmVyc2lvbnMgPSB7IuygnOuhnOyDtyI6IFFfWlMsICJDaHJvbm9zLTIgKyDqs7XtnLTs"
               "nbwiOiBRX0hPTH0KICAgIGF2ZyA9IHsoZywgdik6IG1lYW5fd2lkdGgoUSwgb3NfKSBmb3IgZywgb3NfIGluIGdyb3Vwcy5pdGVt"
               "cygpIGZvciB2LCBRIGluIHZlcnNpb25zLml0ZW1zKCl9CiAgICBmb3IgKGcsIHYpLCB3IGluIGF2Zy5pdGVtcygpOgogICAgICAg"
               "IHByaW50KGYie2d9IMK3IHt2fSA6IO2Pieq3oCDqtazqsIQg6ri47J20IHt3OiwuMWZ9IOuqhSIpCgogICAgaG9sX3dpZHRoID0g"
               "YXZnWygi6rO17Zy07J28IOyeiOuKlCDshLgg7KO8IiwgIkNocm9ub3MtMiArIOqzte2ctOydvCIpXQogICAgcmF0aW8gPSBob2xf"
               "d2lkdGggLyBhdmdbKCLqs7XtnLTsnbwg7J6I64qUIOyEuCDso7wiLCAi7KCc66Gc7IO3IildCiAgICBwcmludChmIkNocm9ub3Mt"
               "MiArIOqzte2ctOydvOydmCDqs7XtnLTsnbwg7J6I64qUIOyEuCDso7wg7Y+J6regIOq1rOqwhCDquLjsnbQgPSB7aG9sX3dpZHRo"
               "OiwuMGZ9IOuqhSIpCiAgICBwcmludChmIuqzte2ctOydvCDsnojripQg7IS4IOyjvCDqtazqsIQg6ri47J20IOu5hCA9IHtyYXRp"
               "bzouMmZ9IOuwsCIpCiAgICBwcmludChmIuqzte2ctOydvCDsl4bripQg65GQIOyjvOyXkOyEnOuKlCBDaHJvbm9zLTIgKyDqs7Xt"
               "nLTsnbzsnbQg7KCc66Gc7IO37J2YIHthdmdbKCfqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wnLCAnQ2hyb25vcy0yICsg6rO17Zy0"
               "7J28JyldIC8gYXZnWygn6rO17Zy07J28IOyXhuuKlCDrkZAg7KO8JywgJ+ygnOuhnOyDtycpXTouMmZ9IOuwsCIpCgogICAgeCA9"
               "IG5wLmFyYW5nZShsZW4oZ3JvdXBzKSkKICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oRklHX1cgKiAwLjgsIDMu"
               "NikpCiAgICBmb3IgaywgKHYsIGNvbG9yKSBpbiBlbnVtZXJhdGUoWygi7KCc66Gc7IO3IiwgQ09MWyLsl7Dtmozsg4kiXSksICgi"
               "Q2hyb25vcy0yICsg6rO17Zy07J28IiwgQ09MWyLtlZjripjsg4kiXSldKToKICAgICAgICBheC5iYXIoeCArIChrIC0gMC41KSAq"
               "IDAuMzgsIFthdmdbKGcsIHYpXSBmb3IgZyBpbiBncm91cHNdLCB3aWR0aD0wLjM4LCBjb2xvcj1jb2xvciwgbGFiZWw9dikKICAg"
               "IGF4LnNldF94dGlja3MoeCwgbGlzdChncm91cHMpKQogICAgYXguc2V0X3hsYWJlbCgi7LGE7KCQ7ZWcIOyjvCIpCiAgICBheC5z"
               "ZXRfeWxhYmVsKCLtj4nqt6Ag6rWs6rCEIOq4uOydtCAo66qFKSIpCiAgICBheC5sZWdlbmQoZnJhbWVvbj1GYWxzZSkKICAgIHBs"
               "dC5zaG93KCkKCu2VtOyEpAoKLSDsmIjsg4Eg7KCB6riw7J2YIOuLteydgCDssqvsp7jsnoXri4jri6QuIOqzte2ctOydvCDsnojr"
               "ipQg7IS4IOyjvOyXkOyEnCBDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnZgg7Y+J6regIOq1rOqwhCDquLjsnbTripQgNSw3MzTrqoXs"
               "nLzroZwsIOygnOuhnOyDtyAzLDc2OOuqheydmCA1LDczNCDDtyAzLDc2OCA9IDEuNTLrsLDsnoXri4jri6QuIOqzte2ctOydvCDs"
               "l4bripQg65GQIOyjvOyXkOyEnOuKlCAyLDgzOeuqheycvOuhnCDsoJzroZzsg7cgMywwODLrqoXsnZggMC45MuuwsOudvCDsobDq"
               "uIgg7Ken7JWE7KGM7Iq164uI64ukLgotIOq3uOuemOyEnCBDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnYAg7Y+J7J28IOqzte2ctOyd"
               "vOydtCDsnojripQg7KO87JeQ66eMIOq1rOqwhOydhCDrhJPtmJTqs6AsIOuEk+2ejCDqtazqsITsnLzroZzrj4Qg7Y+s7ZWo66Wg"
               "7J2AIOuvuOyFmCAxIOydmCA2Ni40JSDsmIDsirXri4jri6QuIOybkOygkOuzhOuhnOuKlCAyMDI1LTA4LTEzIDEuMTPrsLAsIDIw"
               "MjUtMTAtMDEgMS42M+uwsCwgMjAyNi0wMi0xMSAxLjky67Cw66GcIOuEk+2YlOyKteuLiOuLpC4KLSDtnZTtlZwg7Iuk7IiYIOuE"
               "tzogMC4yIOyXtOqzvCAwLjgg7Je07J2YIOywqOydtOuhnCDsnqzrqbQgQ2hyb25vcy0yICsg6rO17Zy07J287J20IDMsNjAz66qF"
               "LCDruYTqsIAgMS43NOuwsOuhnCDrgpjsmLXri4jri6QuIOu5hOulvCDqsbDqvrjroZwo7KCc66Gc7IO3IMO3IENocm9ub3MtMiAr"
               "IOqzte2ctOydvCkg6rWs7ZWY66m0IDAuNjbrsLDsnoXri4jri6QuIOybkOygkCA16rCc66W8IO2VnCDrsojsl5Ag7Y+J6reg7ZWY"
               "66m0IDQsNTc266qFIMK3IDEuMzHrsLDroZwg65GQIOyjvOyZgCDshLgg7KO87J2YIOqwkuydtCDshJ7snoXri4jri6QuIOyynCDr"
               "qoUg64uo7JyE66GcIOuCmOuIoCA1Ljcg7LKY65+8IOyggeycvOuptCDrqoUg64uo7JyEIOq4sOykgOqzvCDrp57sp4Ag7JWK7Iq1"
               "64uI64ukLgotIOydtOuyiCDso7zsl5Ag7ZWgIOydvDogMTAw6rCcIOyXrSDtj4nqt6Ag6rWs6rCEIOq4uOydtCg1LDczNOuqhSDr"
               "k7Ep64qUIOuRkCDrqqjrjbjsnYQg67mE6rWQ7ZWgIOuVjOunjCDslIHri4jri6QuIO2Pieq3oOycvOuhnOuKlCDtlZwg7Jet7J2Y"
               "IOq1rOqwhOydtCDrhJPsnYDsp4Ag7KKB7J2A7KeAIOyVjCDsiJgg7JeG7Ja0LCDso7zqsIQg7J2466ClIOqzhO2aje2RnOyXkOuK"
               "lCDsl63Ct+uCoOuniOuLpCDrkZAg64Gd6rO8IOq3uOuCoCDqtazqsIQg6ri47J2066W8IOyggeyKteuLiOuLpC4g7J2466Cl7J2A"
               "IOychCDrgZ0oMC45IOyXtCnsl5Ag66ee7LawIOuRkOqzoCwg7JWE656YIOuBneq5jOyngOuKlCDspITsl6zrj4Qg65Cc64uk6rOg"
               "IOuztOuKlCDrspTsnITqsIAg6re464KgIOq1rOqwhCDquLjsnbTsnoXri4jri6QuIO2PieydvCDqs7XtnLTsnbzsnbQg7J6I64qU"
               "IOyjvOyXkOuKlCDri7nsnbzsnYQg67qAIOuCoOydmCDtj6ztlajrpaAgNzQuNCUg66W8IO2VqOq7mCDsoIHsirXri4jri6QuIOyd"
               "tCDqsJLrk6TsnYAgQ29sYWIgVDQg7JmAIENQVSDsl5DshJwg6rCZ6rKMIOuCmOyZlOyKteuLiOuLpC4KCuy9lOuTnOqwgCDri7Xs"
               "nYQg66eM65Oc64qUIOuwqeyLnS4g6rWs6rCEIOq4uOydtOuKlCDsl63Ct+uCoOuniOuLpCBgcVsuLi4sIC0xXSAtIHFbLi4uLCAw"
               "XWAg7J206rOgLCBgLm1lYW4oKWAg7J20IOybkOygkCDtlZjrgpjsnZggNzAw7KCQ7J2EIO2Pieq3oO2VqeuLiOuLpC4gYG1lYW5f"
               "d2lkdGhgIOuKlCDqt7gg6rCS7J2EIO2VtOuLuSDsm5DsoJDrk6TroZwg64uk7IucIO2Pieq3oO2VqeuLiOuLpC4g67mE64qUIOqz"
               "te2ctOydvCDsnojripQg7IS4IOyjvOyXkOyEnCBDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnZgg7Y+J6reg7J2EIOygnOuhnOyDt+yd"
               "mCDtj4nqt6DsnLzroZwg64KY64iIIOqwkuydtOudvCwgMSDrs7Tri6Qg7YGs66m0IENocm9ub3MtMiArIOqzte2ctOydvOydmCDq"
               "tazqsITsnbQg642UIOq5geuLiOuLpC4g7JuQ7KCQ66eI64ukIOu5hOulvCDqtaztlbQg7Y+J6reg7ZWY66m0ICgxLjEzICsgMS42"
               "MyArIDEuOTIpIMO3IDMg4omIIDEuNTYg7Jy866GcIDEuNTIg7JmAIOyhsOq4iCDri6TrpoXri4jri6QuIOydtCDrr7jshZjsnZgg"
               "67mE64qUIOyEuCDso7wg7Y+J6reg64G866asIOuCmOuIiCDqsJLsnoXri4jri6QuCg=="),
}


# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
               "eyJtMSI6eyLqs7XtnLTsnbwg7JeG64qUIOuRkCDso7wg7Y+s7ZWo66WgIjpbIuydveydgCDqsJLsnLzroZwg7Iuk7IiY66W8IOynm"
               "uyKteuLiOuLpC4gODUuMSDsnbTrqbQg7KCc66Gc7IO3IGBRX1pTYCDroZwg6rWs7ZaI6rOgLCA2MC45IOuptCAwLjIg7Je06rO8ID"
               "AuOCDsl7QoYFsuLi4sIDFdYMK3YFsuLi4sIDddYCkg7IKs7J2066GcIOq1rO2WiOyKteuLiOuLpC4gMCDsnbTqsbDrgpgg7J2M7Ii"
               "Y66m0IOyVhOuemCDrgZ3qs7wg7JyEIOuBneydhCDrsJTqv5Qg7I286rOgKOyVhOuemCDrgZ3snYAgMC4xIOyXtCBgcVsuLi4sIDBd"
               "YCksIDEwMCDsnbTrqbQg65GQIOyhsOqxtOydhCBgJmAg64yA7IugIGB8YCDroZwg66y27JeI7Iq164uI64ukLiA3MS43IOydtOupt"
               "CDsm5DsoJAgNeqwnOulvCDtlZwg67KI7JeQIO2Pieq3oO2WiOqzoCgzLDUwMOygkCDqsIDsmrTrjbAgMiw1MDjsoJApLCA2Ni40IO"
               "uptCDrkZAg7KO87JmAIOyEuCDso7zsnZgg6rCS7J2EIOuwlOq/lCDsoIHsl4jsirXri4jri6QuIDEg67O064ukIOyekeycvOuptCA"
               "w772eMSDruYTsnKjsnbTrnbwgMTAwIOydhCDqs7Htlanri4jri6QuIOOAjOqzte2ctOydvCDsl4bripQg65GQIOyjvCDtj6ztlajr"
               "paAgPSDsiKvsnpAgJeOAjSDtlZwg7ZaJ7Jy866GcIOyGjOyImCDssqvsp7gg7J6Q66as6rmM7KeAIOy2nOugpe2VqeuLiOuLpCIsI"
               "uybkOygkOuniOuLpCBgWV9URVNUW+ybkOygkF1gIOydtCBgUV9IT0xb7JuQ7KCQXVsuLi4sIDBdYCDsnbTsg4HsnbTqs6AgYFFfSE"
               "9MW+ybkOygkF1bLi4uLCAtMV1gIOydtO2VmOyduCDsoJDsnZgg67mE7JyoKCUp7J2EIOq1rO2VnCDrkqQsIGBQTEFJTl9PUklHSU5"
               "TYCDrkZAg7JuQ7KCQKDIwMjUtMTItMTAgwrcgMjAyNi0wNi0yNCnsnZgg6rCS66eMIO2Pieq3oO2VqeuLiOuLpC4g7JuQ7KCQ66eI"
               "64ukIOq1rOqwhCDslYjCt+yVhOuemMK37JyEIOygkCDsiJjrpbwg642U7ZWY66m0IDcwMCDsnbQg65CY64qU7KeA64+EIOuztOyEu"
               "OyalC4g44CM6rO17Zy07J28IOyXhuuKlCDrkZAg7KO8IO2PrO2VqOuloCA9IOyIq+yekCAl44CNIO2VnCDtlonsnLzroZwg7Lac66"
               "Cl7ZWY7IS47JqUIiwi64K0IO2PrO2VqOuloOydtCDslrTripAg66qo64247J2YIOyWtOuKkCDrkZAg7Je0IOyCrOydtOuhnCDqtaz"
               "tlZwg6rCS7J206rOgLCDslrTripAg7JuQ7KCQ65Ok7J2EIO2Pieq3oO2WiOuKlOyngCJdLCLqs7XtnLTsnbwg7J6I64qUIOyEuCDs"
               "o7wg7Y+s7ZWo66WgIjpbIuydveydgCDqsJLsnLzroZwg7Iuk7IiY66W8IOynmuyKteuLiOuLpC4gNDEuNiDsnbTrqbQg7KCc66Gc7"
               "IO3IGBRX1pTYCDroZwg6rWs7ZaI6rOgLCA0Ni4xIOydtOuptCAwLjIg7Je06rO8IDAuOCDsl7Qg7IKs7J2066GcIOq1rO2WiOyKte"
               "uLiOuLpC4gMCDsnbTqsbDrgpgg7J2M7IiY66m0IOyVhOuemCDrgZ3qs7wg7JyEIOuBneydhCDrsJTqv5Qg7I286rOgLCAxMDAg7J2"
               "066m0IOuRkCDsobDqsbTsnYQgYCZgIOuMgOyLoCBgfGAg66GcIOustuyXiOyKteuLiOuLpC4gNzEuNyDsnbTrqbQg7JuQ7KCQIDXq"
               "sJzrpbwg7ZWcIOuyiOyXkCDtj4nqt6DtlbQoMyw1MDDsoJAg6rCA7Jq0642wIDIsNTA47KCQKSDrkZAg7KO87JmAIOyEuCDso7zqs"
               "IAg7ISe7JiA6rOgLCA3OS41IOuptCDrkZAg6rCS7J2EIOuwlOq/lCDsoIHsl4jsirXri4jri6QuIDYwLjIg66m0IOybkOygkCAyMD"
               "I1LTA4LTEzKOqzte2ctOydvCDsnojripQg7KO8KeydtCDtj4nqt6Dsl5DshJwg67mg7KGM7Iq164uI64ukLiAxIOuztOuLpCDsnpH"
               "snLzrqbQgMO+9njEg67mE7Jyo7J206528IDEwMCDsnYQg6rOx7ZWp64uI64ukLiDjgIzqs7XtnLTsnbwg7J6I64qUIOyEuCDso7wg"
               "7Y+s7ZWo66WgID0g7Iir7J6QICXjgI0g7ZWcIO2WieycvOuhnCDshozsiJgg7LKr7Ke4IOyekOumrOq5jOyngCDstpzroKXtlanri"
               "4jri6QiLCLqsJnsnYAg67Cp67KV7Jy866GcIGBIT0xfT1JJR0lOU2Ag7IS4IOybkOygkCgyMDI1LTA4LTEzIMK3IDIwMjUtMTAtMD"
               "EgwrcgMjAyNi0wMi0xMSnsnZgg7Y+s7ZWo66Wg7J2EIO2Pieq3oO2VqeuLiOuLpC4g67CY67O166y47J20IGBPUklHSU5TWzE6XWA"
               "g7LKY65+8IOybkOygkOydhCDqsbTrhIjrm7Dsp4Ag7JWK7JWY64qU7KeALCDshLgg7JuQ7KCQ7J20IOuqqOuRkCDtj4nqt6Dsl5Ag"
               "7J6I64qU7KeAIOuztOyEuOyalC4g44CM6rO17Zy07J28IOyeiOuKlCDshLgg7KO8IO2PrO2VqOuloCA9IOyIq+yekCAl44CNIO2Vn"
               "CDtlonsnLzroZwg7Lac66Cl7ZWY7IS47JqUIiwi64K0IOqzte2ctOydvCDsnojripQg7IS4IOyjvCDtj6ztlajrpaDsnbQg66qHIO"
               "qwnCDsm5DsoJDsnZgg7Y+J6reg7J206rOgLCDslrTripAg65GQIOyXtCDsgqzsnbTroZwg6rWs7ZWcIOqwkuyduOyngCJdLCJwbG9"
               "0IjpbIuq3uOuemO2UhOulvCDsnbQg7IWA7JeQ7IScIOq3uOuguOuKlOyngCwg6rCA66Gc7LaV6rO8IOyEuOuhnOy2leyXkCDsnbTr"
               "poQo44CM7JiI7LihIOybkOygkOOAjcK344CM7Y+s7ZWo66WgICglKeOAjSnsnYQg64us7JWY64qU7KeAIOuztOyEuOyalC4g6re46"
               "56Y7ZSE64qUIOq3uOumvOydtCDsnojripTsp4DsmYAg65GQIOy2leydmCDsnbTrpoTrp4wg7LGE7KCQ7ZWp64uI64ukIiwiYGF4Ln"
               "NldF94bGFiZWwoXCLsmIjsuKEg7JuQ7KCQXCIpYCDqs7wgYGF4LnNldF95bGFiZWwoXCLtj6ztlajrpaAgKCUpXCIpYCDsnYQg6re"
               "466as64qUIOy9lOuTnOyXkCDstpTqsIDtlZjrnbzqs6AgUFJPTVBUIOyXkCDsoIHsnLzshLjsmpQuIOunieuMgOydmCDsiJjsmYAg"
               "64aS7J20LCA4MCUg7KCQ7ISg7J2AIOyxhOygkO2VmOyngCDslYrsirXri4jri6QiLCLrgrQg6re4656Y7ZSE7J2YIOqwgOuhnOy2l"
               "cK37IS466Gc7LaVIOydtOumhOydtCDslrTrlJTshJwg67mg7KGM64qU7KeAIl0sImVycm9yIjpbIk5hbWVFcnJvciDrqbQg66eoIO"
               "ychCDshKTsuZgg7IWA6rO8IOuwlOuhnCDsnIQg44CM66+47IWYIDEg7KSA67mEIOyFgOOAjeydhCDrqLzsoIAg7Iuk7ZaJ7ZaI64q"
               "U7KeA67aA7YSwIO2ZleyduO2VmOyEuOyalC4g7Iuk7ZaJ7ZaI64qU642w64+EIOyYpOulmOqwgCDrgpjrqbQg7Jik66WYIOusuOye"
               "pSDssqsg7ZaJ7J2EIOq3uOuMgOuhnCDtipzthLDsl5Ag67O07JesIOyjvOqzoCwg44CM66+47IWYIDEg7KSA67mEIOyFgOOAjeydt"
               "CDrp4zrk6Ag7J2066aEKFFfSE9MIMK3IFlfVEVTVCDCtyBPUklHSU5TIMK3IFBMQUlOX09SSUdJTlMgwrcgSE9MX09SSUdJTlMgwr"
               "cgUVMp6rO8IOyEpOy5mCDshYDsnbQg66eM65OgIOydtOumhCjqt7jrprwg7Y+tIGBGSUdfV2AsIOyDiSDsgqzsoIQgYENPTGAp7J2"
               "AIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq4gOyekCDqt7jrjIDroZwg7JOw65286rOgIOyggeycvOyEuOyalC4gS2V5RXJyb3Ig"
               "7JeQIOybkOygkCDrgqDsp5zqsIAg7LCN7Z6I66m0IOuwmOuzteusuOydtCDsm5DsoJAg7ZWY64KY66W8IOqxtOuEiOubtCDqsoPsn"
               "bTri4gg7JuQ7KCQIDXqsJzrpbwg66qo65GQIOqzhOyCsO2VmOudvOqzoCDsoIHsirXri4jri6QiLCLsnbQg7Jik66WY6rCAIOyWtO"
               "uKkCDtlonsl5DshJwg7JmcIOuCrOuKlOyngCJdfSwibTIiOnsiQ2hyb25vcy0yICsg6rO17Zy07J287J2YIOqzte2ctOydvCDsnoj"
               "ripQg7IS4IOyjvCDtj4nqt6Ag6rWs6rCEIOq4uOydtCI6WyLsnb3snYAg6rCS7Jy866GcIOyLpOyImOulvCDsp5rsirXri4jri6Qu"
               "IDMsNzY4IOq3vOyymOuptCDsoJzroZzsg7cgYFFfWlNgIOulvCDsn4Dqs6AsIDIsODM5IOq3vOyymOuptCDqs7XtnLTsnbwg7JeG6"
               "4qUIOuRkCDso7zrpbwg7J+A7Iq164uI64ukLiA0LDU3NiDqt7zsspjrqbQg7JuQ7KCQIDXqsJzrpbwg7ZWcIOuyiOyXkCDtj4nqt6"
               "DtlbQg65GQIOyjvOyZgCDshLgg7KO87J2YIOqwkuydtCDshJ7smIDqs6AsIDMsNjAzIOq3vOyymOuptCAwLjIg7Je06rO8IDAuOCD"
               "sl7TsnZgg7LCo7J2066W8IOyfgOyKteuLiOuLpC4g7J2M7IiY66m0IOu5vOuKlCDsiJzshJzqsIAg6rGw6r6466Gc7J206rOgKDAu"
               "OSDsl7Qg4oiSIDAuMSDsl7QpLCA1Ljcg7LKY65+8IO2VnCDsnpDrpqwg7IiY66m0IOyynCDrqoUg64uo7JyE66GcIOuCmOuIiCDqs"
               "oPsnbTri4gg66qFIOuLqOychCDqt7jrjIDroZwg7KCB7Iq164uI64ukLiDjgIxDaHJvbm9zLTIgKyDqs7XtnLTsnbzsnZgg6rO17Z"
               "y07J28IOyeiOuKlCDshLgg7KO8IO2Pieq3oCDqtazqsIQg6ri47J20ID0g7Iir7J6QIOuqheOAjSDtlZwg7ZaJ7Jy866GcIOygley"
               "ImOq5jOyngCDstpzroKXtlanri4jri6QiLCLsm5DsoJDrp4jri6QgYFFfSE9MW+ybkOygkF1bLi4uLCAtMV0gLSBRX0hPTFvsm5Ds"
               "oJBdWy4uLiwgMF1gIOydhCA3MDDsoJDsnLzroZwg7Y+J6reg7ZWcIOuSpCBgSE9MX09SSUdJTlNgIOyEuCDsm5DsoJDsnYQg7Y+J6"
               "reg7ZWp64uI64ukLiDshLgg7JuQ7KCQ7J20IOuqqOuRkCDtj4nqt6Dsl5Ag7J6I64qU7KeAIOuztOyEuOyalC4g7JuQ7KCQIDIwMj"
               "UtMDgtMTMo6rO17Zy07J28IOyeiOuKlCDso7wp7J20IOu5oOyngOuptCA2LDMzOSwg7JuQ7KCQIDIwMjUtMTAtMDEo6rO17Zy07J2"
               "8IOyeiOuKlCDso7wp7J20IOu5oOyngOuptCA0LDg2NCDqsIAg64KY7Ji164uI64ukLiDjgIxDaHJvbm9zLTIgKyDqs7XtnLTsnbzs"
               "nZgg6rO17Zy07J28IOyeiOuKlCDshLgg7KO8IO2Pieq3oCDqtazqsIQg6ri47J20ID0g7Iir7J6QIOuqheOAjSDtlZwg7ZaJ7Jy86"
               "6GcIOy2nOugpe2VmOyEuOyalCIsIuuCtCDtj4nqt6Ag6rWs6rCEIOq4uOydtOqwgCDslrTripAg66qo6424IMK3IOyWtOuKkCDso7"
               "wgwrcg7Ja064qQIOuRkCDsl7TsnZgg7LCo7J207J206rOgIOustOyKqCDri6jsnITsnbjsp4AiXSwi6rO17Zy07J28IOyeiOuKlCD"
               "shLgg7KO8IOq1rOqwhCDquLjsnbQg67mEIjpbIuydveydgCDqsJLsnLzroZwg7Iuk7IiY66W8IOynmuyKteuLiOuLpC4gMC42NiDs"
               "nbTrqbQg67aE7J6Q7JmAIOu2hOuqqOqwgCDrkqTrsJTrgIzsl4jqs6AoQ2hyb25vcy0yICsg6rO17Zy07J28IMO3IOygnOuhnOyDt"
               "yksIDAuOTIg66m0IOqzte2ctOydvCDsl4bripQg65GQIOyjvOydmCDruYTsnoXri4jri6QuIDEuMzEg7J2066m0IOybkOygkCA16r"
               "Cc66GcIO2Pieq3oO2VnCDrkZAg6ri47J207J2YIOu5hOydtOqzoCwgMS43NCDrqbQgMC4yIOyXtOqzvCAwLjgg7Je07J2YIOywqOy"
               "dtOuhnCDsn4DqsbDrgpgg7JuQ7KCQIDIwMjUtMDgtMTMo6rO17Zy07J28IOyeiOuKlCDso7wp7J20IOu5oOyhjOyKteuLiOuLpC4g"
               "MS4wMCDsnbTrqbQg6rCZ7J2AIOuqqOuNuOuBvOumrCDrgpjriLTsirXri4jri6QuIDE1MiDsspjrn7wgMTAwIOydhCDrhJjsnLzrq"
               "bQgJSDroZwg7KCB7J2AIOqyg+ydtOudvCAxMDAg7Jy866GcIOuCmOuIoCDrsLDroZwg7KCB7Iq164uI64ukLiDjgIw944CNIOuSpO"
               "yXkOuKlCDruYQg7ZWY64KY66eMIOyggeyKteuLiOuLpCjjgIzigKYgPSA1LDczNCDDtyAzLDc2OCA9IDEuNTLjgI0g7LKY65+8IOy"
               "LneydhCDri6Qg7KCB7Jy866m0IOyyqyDsiKvsnpAgNSw3MzQg66W8IOydveyKteuLiOuLpCkiLCLqsJnsnYAg67Cp67KV7Jy866Gc"
               "IOygnOuhnOyDtyBgUV9aU2Ag7J2YIOqzte2ctOydvCDsnojripQg7IS4IOyjvCDtj4nqt6Ag6rWs6rCEIOq4uOydtOulvCDqtaztl"
               "Zjqs6AsIDHrsojsnZggQ2hyb25vcy0yICsg6rO17Zy07J28IOqwkuydhCDqt7gg6rCS7Jy866GcIOuCmOuIleuLiOuLpC4g65GQIO"
               "qwkiDrqqjrkZAgYEhPTF9PUklHSU5TYCDshLgg7JuQ7KCQ7Jy866Gc66eMIO2Pieq3oO2WiOuKlOyngCDrs7TshLjsmpQuIOOAjOq"
               "zte2ctOydvCDsnojripQg7IS4IOyjvCDqtazqsIQg6ri47J20IOu5hCA9IOyIq+yekCDrsLDjgI0g7ZWcIO2WieycvOuhnCDshozs"
               "iJgg65GY7Ke4IOyekOumrOq5jOyngCDstpzroKXtlZjshLjsmpQiLCLrgrQg67mE6rCAIOustOyXh+ydhCDrrLTsl4fsnLzroZwg6"
               "4KY64iIIOqwkuydtOqzoCwg65GQIOqwkuydtCDslrTripAg7KO87J2YIO2Pieq3oOyduOyngCJdLCJwbG90IjpbIuq3uOuemO2UhO"
               "ulvCDsnbQg7IWA7JeQ7IScIOq3uOuguOuKlOyngCwg6rCA66Gc7LaV6rO8IOyEuOuhnOy2leyXkCDsnbTrpoQo44CM7LGE7KCQ7ZW"
               "cIOyjvOOAjcK344CM7Y+J6regIOq1rOqwhCDquLjsnbQgKOuqhSnjgI0p7J2EIOuLrOyVmOuKlOyngCDrs7TshLjsmpQuIOq3uOue"
               "mO2UhOuKlCDqt7jrprzsnbQg7J6I64qU7KeA7JmAIOuRkCDstpXsnZgg7J2066aE66eMIOyxhOygkO2VqeuLiOuLpCIsImBheC5zZ"
               "XRfeGxhYmVsKFwi7LGE7KCQ7ZWcIOyjvFwiKWAg6rO8IGBheC5zZXRfeWxhYmVsKFwi7Y+J6regIOq1rOqwhCDquLjsnbQgKOuqhS"
               "lcIilgIOydhCDqt7jrpqzripQg7L2U65Oc7JeQIOy2lOqwgO2VmOudvOqzoCBQUk9NUFQg7JeQIOyggeycvOyEuOyalC4g66eJ64y"
               "A7J2YIOyImOyZgCDrhpLsnbTripQg7LGE7KCQ7ZWY7KeAIOyViuyKteuLiOuLpCIsIuuCtCDqt7jrnpjtlITsnZgg6rCA66Gc7LaV"
               "wrfshLjroZzstpUg7J2066aE7J20IOyWtOuUlOyEnCDruaDsoYzripTsp4AiXSwiZXJyb3IiOlsiTmFtZUVycm9yIOuptCDrp6gg7"
               "JyEIOyEpOy5mCDshYDqs7wg67CU66GcIOychCDjgIzrr7jshZggMiDspIDruYQg7IWA44CN7J2EIOuovOyggCDsi6Ttlontlojrip"
               "Tsp4DrtoDthLAg7ZmV7J247ZWY7IS47JqULiDsi6TtlontlojripTrjbDrj4Qg7Jik66WY6rCAIOuCmOuptCDsmKTrpZgg66y47J6"
               "lIOyyqyDtlonsnYQg6re464yA66GcIO2KnO2EsOyXkCDrs7Tsl6wg7KO86rOgLCDjgIzrr7jshZggMiDspIDruYQg7IWA44CN7J20"
               "IOunjOuToCDsnbTrpoQoUV9aUyDCtyBRX0hPTCDCtyBZX1RFU1QgwrcgT1JJR0lOUyDCtyBQTEFJTl9PUklHSU5TIMK3IEhPTF9PU"
               "klHSU5TIMK3IFFTKeqzvCDshKTsuZgg7IWA7J20IOunjOuToCDsnbTrpoQo6re466a8IO2PrSBgRklHX1dgLCDsg4kg7IKs7KCEIG"
               "BDT0xgKeydgCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDquIDsnpAg6re464yA66GcIOyTsOudvOqzoCDsoIHsnLzshLjsmpQuIEt"
               "leUVycm9yIOyXkCDsm5DsoJAg64Kg7Kec6rCAIOywje2eiOuptCDrsJjrs7XrrLjsnbQg7JuQ7KCQIO2VmOuCmOulvCDqsbTrhIjr"
               "m7Qg6rKD7J2064uIIOybkOygkCA16rCc66W8IOuqqOuRkCDsnqzrnbzqs6Ag7KCB7Iq164uI64ukIiwi7J20IOyYpOulmOqwgCDsl"
               "rTripAg7ZaJ7JeQ7IScIOyZnCDrgqzripTsp4AiXX19")


_ASK = {"m1": "full", "m2": "blank"}   # 미션 1 은 그대로 보낼 질문, 미션 2 는 빈칸을 내가 채운다


_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)"}


_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, PROMPT 에 무엇을 더 적으면 될지 한 문장으로 답해 주세요. 코드는 고치지 마세요."}


def _blocks(key, reason):
    """PROMPT 칸이 비었을 때(reason) 값·그래프 채점 전에 잠그는가.

    이 실습에는 함정 미션이 없어 PROMPT 칸이 비면 모든 미션을 잠급니다
    """
    if reason:
        return True
    return False


# 권 설정 — 더할 것이 없습니다(minus · skip · spec · pre · checks 를 쓰지 않습니다)
_BOOK = {}
# ━━ 채점 기준 끝 — 아래는 모든 실습이 함께 쓰는 채점 코드입니다 ━━


# ───────────────────────── 출력에서 값 읽기 ─────────────────────────
#  숫자 하나 — 천 단위 쉼표(8,760) · 지수 표기 · 부호. 영문·숫자·밑줄·점 바로 뒤에 붙은 숫자는
#  이름의 일부라 읽지 않습니다 — 「float64」·「int64」 의 64, 「C23」 의 23, 「v2.3」 의 3. 영문에 붙임표로
#  이은 숫자도 이름입니다 — 「Chronos-2」 의 2, 「t-1」 의 1, 「top-3」 의 3.
_NUM = (r"(?<![A-Za-z_\d.])(?<![A-Za-z]-)[-+]?(?:\d{1,3}(?:,\d{3})+(?!\d)|\d+)(?:\.\d*)?(?:[eE][-+]?\d+)?")
_NUM_RE = _re.compile(_NUM)
_KW_RE = _re.compile(r"[A-Za-z_][A-Za-z0-9_]*=(?!=)")       # 「period=24」 같은 인자 설명
_KW_VAL = _re.compile(r"\s*(?:'[^']*'|\"[^\"]*\"|\([^()]*\)|\[[^\[\]]*\]|[^\s,;)\]]+)")
#  Series 를 통째로 찍은 출력의 끝 줄 — 「Name: coef, dtype: float64」 · 「Freq: h, Name: cnt, Length: 168, dtype: float64」 ·
#  「dtype: int64」. f-string 안에 Series 를 넣으면 뒤의 글자(단위)가 이 줄 끝에 붙으므로(「… dtype: float64 대/도」)
#  dtype 뒤의 꼬리 글자는 가리지 않습니다.
_SERIES_RE = _re.compile(r"^(?:(?:Name|Length|Freq): .*, )?dtype: [A-Za-z]\w*", _re.M)
_GLYPH = {**{c: "-" for c in "—–―‒‐‑-−"}, **{c: "·" for c in "·ㆍ•‧・∙⋅"}}   # 겉보기가 같은 대시·가운뎃점은 같게 봅니다
_SEP = set(",;|") | set("·ㆍ•‧・∙⋅") | set("—–―")          # 값 뒤에서 다른 말로 넘어가는 자리


def _num_tokens(text):
    """출력 텍스트의 숫자 — (찍힌 글자, 값) 목록 (유니코드 마이너스 − 는 - 로 읽습니다)"""
    picked = []
    for m in _NUM_RE.finditer((text or "").replace("−", "-")):
        try:
            picked.append((m.group(0), float(m.group(0).replace(",", ""))))
        except ValueError:
            continue
    return picked


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다"""
    return [v for _, v in _num_tokens(text)]


def _unwrap_np(txt):
    """numpy·pandas 의 값 표기를 숫자로 폅니다 — 화면에는 맞는 수가 보이는데 채점이 못 읽는 일이 없게.

    np.float64(40.2) · np.int64(3) · float64(40.2) → 40.2 · 3 (numpy 2 가 dict·list 출력에 남기는 꼴),
    array([40.2]) · [40.2] · [80.] → 40.2 · 80. (원소 하나인 배열 — `.values` 뒤 `[0]` 을 빠뜨린 꼴).
    원소가 둘 이상인 배열은 펴지 않습니다 — 어느 값인지 모릅니다(채점 표가 「숫자 하나로」 를 안내합니다).
    """
    txt = _re.sub(r"\b(?:np|numpy)\.[A-Za-z_]\w*\(\s*([^()]*?)\s*\)", r"\1", txt or "")
    txt = _re.sub(r"\b(?:float|int|uint|complex)(?:8|16|32|64|128)\(\s*([^()]*?)\s*\)", r"\1", txt)
    txt = _re.sub(r"\barray\(\s*(\[[^\[\]()]*\])(?:\s*,\s*dtype=[\w.]+)?\s*\)", r"\1", txt)
    return _re.sub(r"\[\s*([-−+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?)\s*\]", r" \1 ", txt)


def _bare_gap(gap):
    """이름과 값(또는 「=」) 사이가 따옴표·굵게 표시(**)·표의 세로줄·빈칸·괄호 설명·줄표뿐인가 — 그러면 그 이름의 값 줄입니다"""
    gap = _re.sub(r"\([^()]*\)|\[[^\[\]]*\]", "", gap or "")
    return not _re.sub(r"[\s'\"`*」』”’|│┃—–―\-:=,·：＝]", "", gap)


def _paren_open(t, unit):
    """괄호 밖에 숫자가 하나도 없는 꼬리에서 괄호 안의 값 하나만 드러낸 글 — 그런 꼴이 아니면 None.

    「명사 + 용언 1위 = 하다 (18.74 %)」 처럼 낱말 뒤 괄호에 비율을 넣거나 `print("이름 =", (낱말, 비율))` 로 튜플을 그대로
    찍으면 값이 괄호 안에만 있습니다. 괄호 안은 설명으로 보고 건너뛰므로 맞는 출력이 「숫자를 찾지 못했습니다」 가 됩니다.
    괄호 밖에서 값을 하나도 못 읽었을 때만 봅니다 — 읽은 값이 있으면 그 값이 먼저입니다(「확신 = 26 (전체 168 시간 가운데)」).

    - 그릇: 값 표지(또는 이름) 바로 뒤가 여는 괄호이고, 안이 쉼표로 가른 낱말(숫자 없는 글)과 숫자(짧은 단위까지)뿐이며
      숫자가 하나면 그 숫자입니다 — `('하다', 18.74)` · `(하다, 18.74 %)` · `(18.74 %)`. 숫자가 둘 이상인 그릇은 어느 값인지
      모르므로 읽지 않습니다(`[0.57 0.09]` · `(1, 0, 1)`).
    - 기준 단위: 괄호 안에 기준 단위가 바로 붙은 수가 있으면 그 수입니다 — 「하다 (18.74 %)」 · 「하다(18.74%)」 ·
      「하다 (문서 비율 18.74 %)」. 단위가 없는 칸의 괄호 안 숫자(「(기준: 2 미만이면 비슷)」)는 지금처럼 설명입니다.
    """
    groups = _spans(t)
    if not groups:
        return None
    pick = None
    a, b = groups[0]
    before = t[:a].rstrip()
    if not before or before[-1] in "=:→：＝":            # 그릇 — 표지(또는 이름) 바로 뒤의 괄호
        at, spots, clean = a + 1, [], True
        for piece in t[a + 1:b].split(","):
            body = _re.sub(r"[()\[\]'\"`\s]", "", piece)
            m = _NUM_RE.match(body)
            if m and len(body) - m.end() <= 4 and not _re.search(r"\d", body[m.end():]):
                spots.append((at, at + len(piece)))        # 숫자 하나(짧은 단위까지)
            elif _re.search(r"[\d=:→：＝]", body):
                clean = False                              # 숫자가 섞인 말 · 다른 표지 — 그릇이 아니라 설명입니다
            at += len(piece) + 1
        if clean and len(spots) == 1:
            m = _NUM_RE.search(t, spots[0][0], spots[0][1])
            end = spots[0][1]
            while m and end > m.end() and t[end - 1] in ")]'\"` \t":
                end -= 1
            pick = (m.start(), end) if m else None
    if pick is None and unit:
        for a, b in groups:
            for m in _NUM_RE.finditer(t, a + 1, b):
                rest = t[m.end():b]
                gap = len(rest) - len(rest.lstrip())
                if rest[gap:].startswith(unit) or (unit.startswith("%") and rest[gap:].startswith("%")):
                    pick = (m.start(), m.end() + gap + (len(unit) if rest[gap:].startswith(unit) else 1))
                    break
            if pick is not None:
                break
    if pick is None:
        return None
    hide = {i for a, b in groups for i in range(a, _bi.min(b, len(t) - 1) + 1)} - set(range(pick[0], pick[1]))
    return "".join(" " if i in hide else ch for i, ch in enumerate(t))


def _read_value(tail, series=False, inner=0, unit=""):
    """이름 뒤 꼬리에서 그 이름의 값 하나 — (값, 찍힌 글자, 값 뒤 글자, 이름 바로 뒤 값인가, 꼴, 읽은 길) · 없으면 None.

    꼴은 "plain"·"series"(Series 의 인덱스를 건너뛴 값)·"kw"(인자 설명의 값), 읽은 길은 "marker"(값 표지 뒤)·
    "bare"(표지 없이 이름 뒤 첫 숫자)·"kw" 입니다.

    - 괄호·대괄호 안(설명 · 「(기준: 2 미만이면 비슷)」 · 「(m=7)」)은 건너뜁니다.
    - 「period=24, robust=True」 같은 인자 설명은 값이 아닙니다. 그 뒤의 「→」·「=」 뒤를 읽고, 인자 설명뿐이면 마지막 인자 값입니다.
    - 값 표지(「=」·「:」·「→」, 전각 「＝」·「：」)가 숫자보다 먼저 나오면 그 뒤 첫 숫자가 값입니다. 시각 23:00 의 「:」 는 표지가 아닙니다.
    - 숫자가 표지보다 먼저 나오면 — 그 숫자와 표지 사이에 쉼표·가운뎃점·줄표가 있으면 그 숫자가 값이고
      (「기온 계수 -40.20 대/도, 크기 = 40.20」 의 -40.20), 없으면 숫자는 이름의 설명이고 표지 뒤가 값입니다
      (「…, 원점 5 11-16 23:00 까지 = 385.06」 의 385.06). 「→」 는 앞에 숫자가 있으면 해석으로 넘어가는 자리입니다.
      이름 바로 뒤(빈칸·괄호 설명만 사이에 둔)의 소수는 구분 기호가 없어도 값입니다 — 「이름 4.837 대 기준값 = 0.271」 의
      4.837. 이름 바로 뒤의 정수(「이름 2개 원점 평균 = 0.271」 의 2)는 개수를 적은 설명일 수 있어 표지 뒤를 읽습니다.
    - series=True(출력에 「dtype:」 행이 있음)면 「= 24    277.46」 의 인덱스 24 대신 값 277.46 을 읽습니다.
    - inner 는 이름이 괄호 안에서 잡혔을 때의 괄호 깊이입니다 — 그 괄호가 닫히기 전의 숫자는 읽지 않습니다.
    - 괄호 밖에서 값을 하나도 못 읽었으면 괄호 안의 값 하나를 읽습니다 — 값이 그릇(튜플)에 담겼거나, 괄호 안에 기준 단위
      (unit)가 붙은 수가 있을 때입니다(`_paren_open`). 이렇게 읽은 줄은 약한 줄입니다(`_read_paren` · `_value_use`, 0.4.2).
    """
    return _read_paren(tail, series, inner, unit)[0]


def _read_paren(tail, series=False, inner=0, unit=""):
    """`_read_value` 의 값과 그 값을 괄호 안에서만 읽었는가 — (값 또는 None, 괄호 안에서만 읽음).

    괄호 안에서만 읽은 값은 약한 줄의 값입니다 — 「이름 (제로샷 41.6 % 보다 높음)」 처럼 덧붙인 설명 줄의 수가 맨 이름
    「이름 = 값」 줄과 부딪치지 않게, 다른 값 줄이 있으면 `_value_use` 가 쓰지 않습니다(그런 줄뿐이면 0.4 의 53번대로 읽습니다).
    """
    t = "(" * inner + _unwrap_np(tail).replace("−", "-")
    got = _scan_value(t, 0, series, True)
    if got is None and not inner and t.count("(") + t.count("[") > t.count(")") + t.count("]"):
        got = _scan_value(_re.sub(r"[()\[\]]", " ", t), 0, series, True)   # 닫히지 않은 괄호 — 괄호 기호만 떼고 다시
    if got is None and not inner:
        opened = _paren_open(t, unit)                     # 「하다 (18.74 %)」 · `('하다', 18.74)` — 값이 괄호 안에만 있습니다
        if opened is not None:
            got = _scan_value(opened, 0, series, True)
            return got, got is not None
    return got, False


def _scan_value(t, i, series, top):
    """`_read_value` 의 한 단계 — top 이면 이름 바로 뒤, 아니면 값 표지 뒤에서 읽습니다"""
    depth, cand, sep, kws, start, n = 0, None, False, [], i, len(t)
    while i < n:
        ch = t[i]
        if ch in "([":
            depth += 1
            i += 1
            continue
        if ch in ")]":
            depth = _bi.max(0, depth - 1)
            i += 1
            continue
        if depth:
            i += 1
            continue
        kw = _KW_RE.match(t, i)
        if kw and (i == 0 or not (t[i - 1].isalnum() or t[i - 1] == "_")):
            val = _KW_VAL.match(t, kw.end())
            toks = _num_tokens(val.group(0)) if val else []
            if toks:
                kws.append(toks[0])
            i = val.end() if val else kw.end()
            continue
        if ch in "=:→：＝" or t.startswith("->", i):
            step = 2 if t.startswith("->", i) else 1
            if ch == ":" and 0 < i < n - 1 and t[i - 1].isdigit() and t[i + 1].isdigit():
                i += 1                                            # 시각·비율(23:00 · 1:2)
                continue
            if ch == "=" and t[i + 1:i + 2] == "=":
                i += 2
                continue
            arrow = ch == "→" or step == 2
            if cand is not None and (sep or arrow or not top or (cand[3] and "." in cand[1])):
                return cand                                       # 값 뒤의 다른 말(「크기 = …」·「→ 해석」)
            got = _scan_value(t, i + step, series, False)
            if got is not None:
                if top:
                    got = got[:3] + (cand is None and _bare_gap(t[start:i]),) + got[4:5] + ("marker",)
                return got
            if cand is not None:
                return cand
            i += step
            continue
        m = _NUM_RE.match(t, i)
        if m:
            if cand is None:
                raw, end, kind = m.group(0), m.end(), "plain"
                if series:                                        # 「= 24    277.46」 — Series 의 인덱스와 값
                    pair = _re.match(r"(-?\d+)\s{2,}(" + _NUM + ")", t[i:])
                    if pair:
                        raw, end, kind = pair.group(2), i + pair.end(), "series"
                try:
                    cand = (float(raw.replace(",", "")), raw, t[end:], top and _bare_gap(t[start:i]), kind,
                            "bare" if top else "marker")
                except ValueError:
                    cand = None
                if cand is not None and not top:
                    return cand
            i = m.end()
            continue
        if cand is not None and (ch in _SEP or (ch == "-" and t[i - 1:i] == " " and t[i + 1:i + 2] == " ")):
            sep = True
        i += 1
    if cand is not None:
        return cand
    if kws:                                                       # 인자 설명뿐 — 마지막 인자의 값(「fs_hour=0.5715」)
        return (kws[-1][1], kws[-1][0], "", False, "kw", "kw")
    return None


def _depth(txt):
    """txt 를 지난 뒤의 괄호 깊이"""
    depth = 0
    for ch in txt or "":
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
    return depth


def _spans(txt):
    """맨 바깥 괄호·대괄호 묶음의 (여는 자리, 닫는 자리) 목록 — 닫히지 않은 묶음은 글 끝까지입니다"""
    out, depth, start = [], 0, 0
    for i, ch in enumerate(txt or ""):
        if ch in "([":
            if depth == 0:
                start = i
            depth += 1
        elif ch in ")]" and depth:
            depth -= 1
            if depth == 0:
                out.append((start, i))
    if depth:
        out.append((start, len(txt)))
    return out


def _squeeze(txt):
    """빈칸과 괄호 기호를 걷고 대시·가운뎃점 글리프를 하나로 맞춘 글자"""
    return "".join(_GLYPH.get(ch, ch) for ch in (txt or "") if not ch.isspace() and ch not in "()[]")


def _tighten(txt, own=None):
    """빈칸과 괄호 기호를 걷어낸 글자 · 원래 자리 — 대시·가운뎃점 글리프는 하나로 맞춥니다.

    own 이 None 이면 괄호 안의 말은 둡니다(기호만 걷음). own 이 집합이면 안의 말이 own 에 있는 묶음만 두고 나머지 묶음은
    안의 말까지 걷습니다. 줄에 끼어든 괄호 설명(「…입력(512일)에 있는…」)은 이름의 일부가 아니지만, 이름에 원래 있는
    괄호 안의 말(「…MASE(Chronos-2 + 공휴일)」)은 이름을 가르는 부분이라 걷지 않습니다.
    """
    txt = txt or ""
    drop = set()
    if own is not None:
        for a, b in _spans(txt):
            if _squeeze(txt[a + 1:b]) not in own:
                drop.update(range(a, b + 1))
    keep = [i for i, ch in enumerate(txt) if i not in drop and not ch.isspace() and ch not in "()[]"]
    return "".join(_GLYPH.get(txt[i], txt[i]) for i in keep), keep


def _canon(txt):
    """숫자 표기를 맞춥니다 — 5.0 → 5 · 3.50 → 3.5 · 1,234 → 1234"""
    def _one(m):
        s = m.group(0).replace(",", "")
        return s.rstrip("0").rstrip(".") if "." in s else s
    return _re.sub(r"\d[\d,]*(?:\.\d+)?", _one, txt or "")


def _label_spot(line, label):
    """줄에서 이름이 있는 자리 (시작, 끝, 느슨하게 찾았는가) — 없으면 None.

    차례로 찾습니다 — 정확히 같은 글자(괄호 밖 자리 먼저) → 빈칸과 줄에 끼어든 괄호 설명을 걷어낸 꼴(대시 — – ― - 와
    가운뎃점 · ㆍ • ‧ ・ 는 같게) → 괄호 기호만 뗀 꼴(「횟수(D)」 와 「횟수 D」) → 숫자 표기까지 맞춘 꼴은 부르는 쪽이 봅니다.

    이름에 괄호가 있으면 그 안의 말도 이름입니다. 「공휴일 있는 세 주 MASE(Chronos-2 + 공휴일)」 을 찾을 때
    「공휴일 있는 세 주 MASE(Chronos-2 제로샷) = 2.763」 은 다른 이름의 줄입니다 — 괄호 안까지 걷고 찾으면 나란히 찍은
    다른 모델의 줄이 같은 이름 줄이 되어 맞는 코드가 「줄이 2개이고 값이 서로 다릅니다」 로 막힙니다. 괄호 묶음을 아예
    적지 않은 줄(「공휴일 있는 세 주 MASE = 1.022」)만 느슨하게 찾은 것으로 돌려주고, 부르는 쪽은 제대로 찾은 줄이 하나라도
    있으면 느슨하게 찾은 줄을 쓰지 않습니다(`_occurrences`).
    """
    if label and label in line:
        spots = [m.start() for m in _re.finditer(_re.escape(label), line)]
        top = [p for p in spots if _depth(line[:p]) == 0]
        # 「{'상태별 리뷰 수': 178, '리뷰 수': 200}」 — 앞에 다른 낱말이 붙은 자리(더 긴 이름)보다 이름으로 시작하는 자리가 먼저입니다
        at = next((p for p in top if not line[:p].rstrip()[-1:].isalpha()), top[0] if top else spots[0])
        return at, at + len(label), False
    full = _tighten(label)[0]
    if not full:
        return None
    groups = [_squeeze(label[a + 1:b]) for a, b in _spans(label)]
    for own in (set(groups), None):
        tight_line, keep = _tighten(line, own)
        at = tight_line.find(full)
        if at >= 0:
            return keep[at], keep[at + len(full) - 1] + 1, False
    if not groups:
        return None
    # 이름의 괄호 묶음을 적지 않은 줄 — 뼈대(괄호 묶음을 뺀 이름)만으로 찾습니다. 이름의 괄호 자리에 다른 말의 괄호가
    # 있으면(「MASE(Chronos-2 제로샷)」) 이 이름의 줄이 아닙니다
    base, base_keep = _tighten(label, set())
    base_line, keep = _tighten(line, set())
    at = base_line.find(base) if base else -1
    if at < 0:
        return None
    spot_of = {_bi.sum(1 for k in base_keep if k < a): g for (a, _b), g in zip(_spans(label), groups)}
    for a, b in _spans(line):
        rel = _bi.sum(1 for k in keep if k < a) - at
        if rel in spot_of and _squeeze(line[a + 1:b]) != spot_of[rel]:
            return None
    return keep[at], keep[at + len(base) - 1] + 1, True


_OP_SURE = set("−+×÷*")                    # 식의 연산 기호
_OP_MAYBE = set("-–‒/><≥≤→")               # 붙임표 · 빗금 · 부등호 · 화살표 — 식일 수도, 말을 가르는 자리일 수도 있습니다


def _operand_head(head):
    """이름 앞의 글이 「다른 이름 + 연산 기호」 로 끝나는가 — "expr"(이름이 식의 뒤 항) · "maybe"(식인지 모름) · "".

    「명사만 1위 − 명사 + 용언 1위 : −7.82 %p」 에서 「명사 + 용언 1위」 는 뺄셈의 뒤 항이고 −7.82 는 두 값의 차입니다.
    이 줄을 뒤 이름의 값 줄로 읽으면 제 값 줄(「명사 + 용언 1위 = 하다 18.74 %」)과 값이 달라 맞는 출력이
    「줄이 2개이고 값이 서로 다릅니다」 가 됩니다. 이름에 원래 든 기호(「명사 + 용언 1위」 의 +)는 이름의 일부라 보지 않습니다.
    기호 앞의 토막이 숫자로 시작하면(「A = 0.271 / 이름 = 0.281」 · 「A = 40.2 대/도 - 이름 = …」) 그 기호는 항목을 가르는
    자리이고, 기호 앞에 말이 없으면(「- 이름 = 값」 · 「* 이름」 · 「> 이름」) 글머리표입니다 — 둘 다 식이 아닙니다.
    붙임표 · 빗금 · 부등호 · 화살표(「A - 이름」 · 「A / 이름」 · 「A > 이름 : a > b」 · 「A → 이름 : a → b」)는 말을 가르는 자리로도
    쓰여 식인지 알 수 없습니다 — "maybe" 로 돌려주고, 부르는 쪽이 이름으로 시작하는 줄을 먼저 씁니다.
    """
    h = (head or "").rstrip()
    op = h[-1:]
    if not op or (op not in _OP_SURE and op not in _OP_MAYBE):
        return ""
    before = h[:-1].rstrip()
    if not before or not (before[-1].isalnum() or before[-1] in ")]%\0」』”’\"'"):
        return ""
    if op in "-–‒" and h[-2:-1].strip():           # 앞말에 붙은 붙임표는 낱말을 잇는 글자입니다
        return ""
    cut = _bi.max(before.rfind(ch) for ch in "=:→：＝,;|")
    seg = before[cut + 1:].strip()
    if not seg or _NUM_RE.match(seg.replace("−", "-")):
        return ""
    return "expr" if op in _OP_SURE else "maybe"


def _mark_at(tail):
    """꼬리에서 괄호 밖 첫 값 표지(「=」·「:」, 전각 포함)의 자리 — 없으면 -1. 시각 23:00 의 「:」 는 표지가 아닙니다"""
    depth, n = 0, len(tail or "")
    for i, ch in enumerate(tail or ""):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
        elif depth == 0 and ch in "=:：＝":
            if ch == ":" and 0 < i < n - 1 and tail[i - 1].isdigit() and tail[i + 1].isdigit():
                continue
            return i
    return -1


#  강한 값 줄(0.4.2) — 이름이 줄 머리에 있고, 이름 바로 뒤가 값 표지(「=」·「:」·「→」, 전각 포함)이며 표지 바로 뒤가 숫자인 줄.
#  줄 머리: 이름 앞에는 빈칸 · 글머리표 · 번호(「1.」·「1)」·「(1)」·「①」) · 여는 따옴표 · 굵게 표시 · 사전의 「{」 만 옵니다
_LEAD_OK = _re.compile(r"(?:\s|[-*•‧·+>▪◦▶■□○●※→]|\d{1,3}[.)](?=\s)|\(\d{1,3}\)|[①-⑳⑴-⒇]|['\"`「『“‘{]|[(\[](?=['\"]))*")
#  이름과 표지 사이에는 닫는 따옴표 · 굵게 표시 · 빈칸만 — 숫자가 numpy 꼴(np.float64(…) · [ … ] · array([…]))로 싸여도 됩니다
_STRONG_TAIL = _re.compile(r"[\s'\"`*」』”’]*(?:=(?!=)|:|：|＝|→|->)\s*"
                           r"(?:(?:np|numpy)\.[A-Za-z_]\w*\(\s*|(?:float|int|uint)(?:8|16|32|64|128)\(\s*|array\(\s*\[\s*|\[\s*)?"
                           r"[-+−]?\.?\d")
#  한 줄에 값 짝을 이어 적는 구분 기호 — 쉼표(숫자 사이의 천 단위 쉼표는 아님) · 쌍반점 · 세로줄 · 빗금 · 가운뎃점 · 줄표 · 화살표 · 빈칸 둘 이상
_CHAIN_SEP = _re.compile(r"(?<!\d),|,(?!\d)|[;|/·ㆍ•‧・∙⋅—–―→]|->|\s-\s|\s{2,}")
_NAME_RE = {}                              # 이름 → 빈칸 · 대시 · 가운뎃점 글리프를 가리지 않고 찾는 정규식(한 번만 짓습니다)


def _name_len_at(text, i, names):
    """text 의 i 자리에서 시작하는 이름(names 가운데 가장 긴 것)의 글자 수 — 없으면 0. 가려 둔 더 긴 이름(\\0)도 이름입니다"""
    if text[i:i + 1] == "\0":
        j = i
        while j < len(text) and text[j] == "\0":
            j += 1
        return j - i
    best = 0
    for name in names:
        if not name:
            continue
        if text.startswith(name, i):
            best = _bi.max(best, len(name))
            continue
        pat = _NAME_RE.get(name)
        if pat is None:
            same = {}
            for k, v in _GLYPH.items():
                same.setdefault(v, set()).add(k)
            pat = _NAME_RE[name] = _re.compile(r"\s*".join(
                "[" + _re.escape("".join(sorted(same[_GLYPH[ch]] | {_GLYPH[ch]}))) + "]" if ch in _GLYPH else _re.escape(ch)
                for ch in name if not ch.isspace()))
        m = pat.match(text, i)
        if m:
            best = _bi.max(best, m.end() - i)
    return best


def _chain_head(lead, names):
    """이름 앞의 글(lead)이 「이름 = 숫자」 짝을 구분 기호로 이어 적은 값 줄의 앞부분인가 — 「확신 = 26 시간 · 보류 = 142 시간」 의 「보류」.

    맨 앞 짝은 줄 머리에서 시작하고, 짝마다 이름(값 칸의 이름) 바로 뒤가 값 표지이고 표지 바로 뒤가 숫자이며, 짝 사이에는 값 표지가 없는
    글과 구분 기호만 있습니다. 「(참고) 제로샷: A = 85.1 % · B = 41.6 %」 의 B 는 맨 앞 짝이 줄 머리에서 시작하지 않아 아닙니다 —
    비교하려고 덧붙인 줄이라 그 줄의 수가 맨 이름 「B = 값」 줄과 부딪치지 않게 합니다.
    """
    m = _LEAD_OK.match(lead)
    i = m.end() if m else 0
    while True:
        k = _name_len_at(lead, i, names)
        if not k:
            return False
        mt = _STRONG_TAIL.match(lead, i + k)
        if not mt:
            return False
        j, nxt = mt.end(), None
        for sm in _CHAIN_SEP.finditer(lead, j):
            if _re.search(r"[=＝：]|(?<!\d):|:(?!\d)", lead[j:sm.start()]):
                return False                                  # 짝 사이의 다른 값 표지 — 이름을 모르는 짝입니다
            q = sm.end()
            while q < len(lead) and lead[q] in " \t'\"`*「『“‘":
                q += 1
            if q >= len(lead):
                return True                                   # 지금 이름 바로 앞의 구분 기호
            if _name_len_at(lead, q, names):
                nxt = q
                break
        if nxt is None:
            return False
        i = nxt


def _lead_kind(lead):
    """이름 앞 글(lead)의 꼴 — "top"(줄 머리) · "num"(날짜 · 숫자 · % 뒤) · "sep"(구분 기호 뒤) · ""(그 밖)"""
    if _LEAD_OK.fullmatch(lead or ""):
        return "top"
    body = (lead or "").rstrip(" \t'\"`*「『“‘")
    last = body[-1:]
    if last.isdigit() or last == "%":
        return "num"
    if last and (last in ",;|/·ㆍ•‧・∙⋅—–―" or (last in "-–‒" and body[-2:-1].isspace())):
        return "sep"
    return ""


def _occurrences(lines, label, others=(), series=False, unit=""):
    """이름이 있는 줄마다 하나씩 — {"tail": 이름 뒤 꼬리, "seg": 이 항목의 토막, "val": `_read_value` 결과, "boxed": 배열인가}.

    값줄의 숫자를 한 주머니에 담아 기준 범위와 대조하면 **다른 항목 줄의 값**이 이 항목을 헛통과시킵니다.
    그래서 이름이 있는 줄의 이름 뒤만 봅니다. 꼬리에 다른 항목의 이름(`others`)이 나오면 거기서 끊고
    (표 머리 줄에서 옆 칸 이름 안의 「365일」·「168」 을 이 항목의 값으로 읽지 않게), 이 이름을 품은 더 긴
    이름(「공식 차수」 에 대한 「공식 차수 AIC」)이 있는 자리는 이 이름으로 치지 않습니다.
    이름이 괄호 안에서만 잡히면(「… (명목 포함률 80% 기준)」) 그 괄호가 닫히기 전의 숫자는 읽지 않습니다. 따옴표로
    싼 이름(튜플·리스트를 그대로 찍은 `('이름', 0.281)`)은 괄호가 값을 담은 그릇이라 그 안의 값을 읽습니다.
    "seg" 는 이 이름부터 다른 항목 이름 앞까지입니다 — 줄의 첫 이름이면 이름 앞의 말까지 넣습니다.
    "line" 은 줄 원문, "loose" 는 이름의 괄호 묶음을 적지 않은 줄인가입니다 — 이름을 제대로 적은 줄이 하나라도 있으면
    느슨하게 찾은 줄은 돌려주지 않습니다(`_label_spot`). "after" 는 다른 항목 이름에서 끊지 않은 이름 뒤의 글입니다
    (값이 낱말 · 문장인 항목을 보는 권 검사가 씁니다 — `_text_tails`). "pre" 는 이름 바로 앞에 다른 낱말이 붙어 있는가입니다
    (「상태별 리뷰 수 : …」 의 「리뷰 수」 — 더 긴 이름의 줄일 수 있어 `_value_use` 가 이름으로 시작하는 줄을 먼저 씁니다.
    붙임표 · 빗금 뒤의 이름도 여기에 넣습니다).
    이름 앞에 「다른 이름 + 연산 기호」 가 있는 줄은 이름이 식의 뒤 항이라 이 이름의 줄이 아닙니다. 기호가 붙임표 · 빗금이면
    식인지 알 수 없어 느슨하게 찾은 줄로 둡니다(`_operand_head`).
    "tag" 는 이름 앞에 꼬리표가 붙은 줄인가입니다 — 「(전임 코드) 이름 = …」 · 「고치기 전: 이름 = …」 처럼 이름 앞의 말이
    닫는 괄호나 쌍점으로 끝납니다. "noted" 는 이름과 값 표지 사이에 괄호 설명이 낀 줄인가입니다 — 「이름(50만 건 전체, 이론) = …」.
    둘 다 비교하려고 덧붙인 줄일 수 있어, 맨 이름 「이름 = 값」 줄이 따로 있으면 `_value_use` 가 쓰지 않습니다(그런 줄뿐이면 그 값을 읽습니다).
    "strong" 은 강한 값 줄인가입니다(0.4.2) — 이름이 줄 머리에 있고(앞에는 글머리표 · 번호 · 따옴표만, `_LEAD_OK`) 이름 바로 뒤가
    값 표지이며 표지 바로 뒤가 숫자인 줄(`_STRONG_TAIL`), 그리고 그런 「이름 = 숫자」 짝을 구분 기호로 이어 적은 줄의 뒤 짝(`_chain_head`)입니다.
    강한 값 줄이 하나라도 있으면 `_value_use` 가 그 줄끼리만 값을 읽고 부딪침을 봅니다. "aside" 는 이름 앞이 날짜 · 숫자로 끝나거나
    (「2025-08-13 이름: 78.9%」) 구분 기호 바로 뒤에 이름이 온 줄(「명목 80% 와의 차: A -0.5 %p · 이름 -13.6 %p」), 꼬리표 뒤의 이름에
    표지 없이 숫자가 붙은 줄(같은 줄의 「명목 80% 와의 차: 이름 -0.5 %p」)입니다 — 강한 값 줄이 없을 때 이름으로 시작하는 줄이 따로
    있으면 `_value_use` 가 쓰지 않습니다. "paren" 은 값을 괄호 안에서만 읽은 줄입니다(`_read_paren`) — 다른 값 줄이 있으면 쓰지 않습니다.
    unit 은 이 칸의 기준 단위입니다 — 값이 괄호 안에만 있는 줄에서 기준 단위가 붙은 수를 읽습니다(`_paren_open`).
    """
    rivals = [o for o in others if o and o != label]
    longer = [o for o in rivals if label in o]
    # 빠른 거르기 — 이름 뼈대의 글자(한글·영문)가 다 있지 않은 줄에는 이름이 없습니다(출력이 수만 줄일 때 채점이 늦지 않게)
    need = {ch for ch in _tighten(label, set())[0] if ch.isalpha()}
    out = []
    for line in lines:
        if not need.issubset(line):
            continue
        masked = line
        for o in longer:
            masked = masked.replace(o, "\0" * len(o))
        spot = _label_spot(masked, label)
        plain = line
        if spot is None:
            masked = _canon(masked)
            spot = _label_spot(masked, _canon(label))
            plain = masked
        if spot is None:
            continue
        start, end, loose = spot
        role = _operand_head(masked[:start])
        if role == "expr":
            continue                                              # 「다른 이름 − 이름 : 차」 — 이 이름의 값 줄이 아닙니다
        loose = loose or role == "maybe"
        tail = masked[end:]
        stops = []
        for o in rivals:
            at = tail.find(o) if o not in longer else -1
            while at >= 0 and _depth(tail[:at]):                  # 괄호 설명 안에 나온 다른 항목 이름에서는 끊지 않습니다 —
                at = tail.find(o, at + 1)                         # 「이상치 비율 (분모 N) = 37.68 %」 의 「분모」
            if at >= 0:
                stops.append(at)
        if stops:
            tail = tail[:_bi.min(stops)]
        inner = _depth(masked[:start])
        quote = masked[start - 1:start]
        if inner and quote in ("'", '"') and masked[end:end + 1] == quote:
            inner = 0                                             # ('이름', 값) · ['이름', 값] — 그릇 안의 값
        head = masked[:end]
        seg = (tail if _bi.any(o in head for o in rivals if o not in longer) else head + tail).replace("\0", "")
        val, in_paren = _read_paren(tail, series, inner, unit)
        if val is not None and inner:
            val = val[:3] + (False,) + val[4:]
        boxed = val is None and _bi.bool(_re.search(r"\[[^\[\]]*\d[^\[\]]*\]", _unwrap_np(tail)))
        lead = masked[:start].rstrip()
        cut = _mark_at(tail)
        kind = _lead_kind(masked[:start])                         # 줄 머리 · 날짜와 숫자 뒤 · 구분 기호 뒤
        chain = kind != "top" and not inner and _chain_head(masked[:start], [label] + rivals)
        tag = lead[-1:] in tuple(":：)]」』") and _bi.any(ch.isalpha() for ch in lead)
        out.append({"tail": tail.replace("\0", ""), "seg": seg, "val": val, "boxed": boxed, "line": line, "loose": loose,
                    "after": plain[end:].replace("\0", ""),
                    "pre": role == "maybe" or lead[-1:].isalpha(),
                    "tag": tag,
                    "noted": cut >= 0 and _bi.bool(_re.search(r"[(\[]", tail[:cut])) and _bare_gap(tail[:cut]),
                    "strong": (val is not None and val[3] and val[5] == "marker" and not inner and not in_paren
                               and (kind == "top" or chain) and _bi.bool(_STRONG_TAIL.match(tail))),
                    "paren": in_paren,                            # 값을 괄호 안에서만 읽은 줄(약한 줄 — 0.4.2)
                    # 이름 앞이 날짜 · 숫자 · 구분 기호로 끝나는 줄(값 짝을 이은 줄은 아님), 꼬리표 뒤 이름에 표지 없이 숫자가 붙은 줄
                    # (「명목 80% 와의 차: 이름 -0.5 %p」) — 보조 줄일 수 있습니다
                    "aside": (kind in ("num", "sep") and not chain) or (tag and val is not None and val[5] == "bare")})
    if _bi.any(not o["loose"] for o in out):
        out = [o for o in out if not o["loose"]]
    return out


def _unit_value(val, unit):
    """「k 건 (r %)」 꼴에서 기준 단위가 붙은 괄호 안의 수 — 그런 꼴이 아니면 읽은 값 그대로.

    기준 단위가 % 인 칸에 「이름 = 14 건 (1.4 %)」 로 찍으면 표지 뒤 첫 숫자는 건수 14 이고 비율은 괄호 안에 있습니다.
    읽은 값 바로 뒤에 기준 단위가 없고, 값(과 짧은 단위 낱말) 바로 뒤의 괄호 안에 기준 단위가 붙은 수가 있으면 그 수가 값입니다.
    값과 괄호 사이에 분모를 적은 꼴(「14 건 / 1,000 건 (1.4 %)」 · 「14 건 ÷ 1,000 건 (1.4 %)」)도 같습니다.
    칸마다 켜는 규칙입니다(기준의 `paren_unit`) — 「확신 = 26 (전체 168 시간 가운데)」 처럼 괄호 안의 수가 설명인 칸에도
    켜면 설명의 수를 값으로 읽습니다.
    """
    after = val[2]
    if not unit or after.lstrip().startswith(unit):
        return val
    word = r"[^\s\d()\[\],;:=→·|/÷]{0,6}"                          # 값 뒤의 짧은 단위 낱말(건 · 개 · 명)
    m = _re.match(r"\s*" + word + r"\s*(?:[/÷]\s*[\d,]+(?:\.\d+)?\s*" + word + r"\s*)?\(", after)   # (분모) 여는 괄호
    if not m:
        return val
    close = after.find(")", m.end())
    close = len(after) if close < 0 else close
    inner = after[m.end():close]
    for tok in _NUM_RE.finditer(inner):
        if inner[tok.end():].lstrip().startswith(unit):
            try:
                return (float(tok.group(0).replace(",", "")), tok.group(0), inner[tok.end():] + after[close:]) + val[3:]
            except ValueError:
                continue
    return val


def _value_use(occ, val_lines):
    """이름이 있는 줄들(`_occurrences`) 가운데 채점에 쓸 값 줄.

    같은 이름 줄이 여럿이면 이름 바로 뒤 값 줄만 봅니다. 바로 뒤 값 줄은 「이름 = 값」·「이름: 값」(어느 행이든), 그리고
    단위·「=」 가 붙은 행(val_lines)의 「이름 값」 입니다. 표지도 단위도 없는 행의 「이름 숫자」(「결과 요약: 이름 A, 이름 B 2개 값」
    의 2)는 값 줄이 달리 없을 때만 씁니다. 이름 바로 앞에 다른 낱말이 붙은 줄은 더 긴 이름의 줄일 수 있습니다 — 도우미가 찍는
    「상태별 리뷰 수 : 통과 178 · …」 을 「리뷰 수」 의 값 줄로 읽으면 맞는 출력이 「줄이 2개이고 값이 서로 다릅니다」 가 됩니다.
    이름으로 시작하는 값 줄이 따로 있으면 그런 줄은 쓰지 않고, 그런 줄뿐이면 지금처럼 그 값을 읽습니다.
    비교하려고 덧붙인 줄도 같습니다 — 이름 앞에 꼬리표가 붙은 줄(「(전임 코드) 이름 = 892 건」)과 이름 뒤 괄호에 설명을 넣은 줄
    (「이름(50만 건 전체, 이론) = 17.39 %」)은 맨 이름 「이름 = 값」 줄(이름 바로 뒤 값 줄)이 따로 있으면 쓰지 않습니다. 설명을 이름
    앞에 붙인 「원본 이름 = …」 은 통과하고 이름 뒤 괄호에 넣은 줄만 「줄이 2개이고 값이 서로 다릅니다」 가 되지 않게 합니다.

    강한 값 줄(「이름 = 숫자」 — 이름이 줄 머리에 있고 표지 바로 뒤가 숫자인 줄)이 하나라도 있으면 그 줄끼리만 씁니다(0.4.2).
    원점별 행 · 명목 대비 줄 · 제로샷 비교 줄 · 날짜가 앞에 붙은 줄처럼 같은 이름이 든 보조 줄의 수가 맨 이름 줄과 부딪쳐 맞는 출력이
    「줄이 2개이고 값이 서로 다릅니다」 가 되지 않게 합니다. 옛 print 와 새 print 를 함께 남긴 셀, 전임 코드의 줄은 둘 다 강한 값 줄이라
    지금처럼 부딪침으로 잡힙니다. 강한 값 줄이 없으면 아래 차례(0.4.1 과 같고, 이름 앞에 날짜 · 숫자가 오거나 구분 기호 바로 뒤에 이름이 온
    줄과 꼬리표 뒤 이름에 표지 없이 숫자가 붙은 줄 — "aside" — 은 이름으로 시작하는 줄이 따로 있을 때, 값을 괄호 안에서만 읽은 줄 —
    "paren" — 은 다른 값 줄이 있을 때 쓰지 않습니다)로 고릅니다.
    """
    found = [o for o in occ if o["val"]]
    strong = [o for o in found if o.get("strong")]
    if strong:                                              # 강한 값 줄끼리만 — 보조 줄의 수는 이 항목의 값으로 읽지 않습니다
        return strong
    if _bi.any(not o["pre"] for o in found):               # 이름으로 시작하는 줄(구분 기호 뒤의 이름 포함)이 있으면, 이름 앞에
        found = [o for o in found if not o["pre"]]          # 다른 낱말이 붙은 줄(「상태별 리뷰 수 : 통과 178 …」)은 쓰지 않습니다
    if _bi.any(not o.get("aside") for o in found):         # 이름 앞이 날짜 · 숫자이거나 구분 기호 바로 뒤에 이름이 온 줄도 같습니다
        found = [o for o in found if not o.get("aside")]
    if _bi.any(not o.get("paren") for o in found):         # 값을 괄호 안에서만 읽은 줄(「이름 (제로샷 41.6 % 보다 높음)」)은
        found = [o for o in found if not o.get("paren")]    # 다른 값 줄이 있으면 쓰지 않습니다 — 그런 줄뿐이면 0.4 의 53번대로 읽습니다
    # 비교하려고 덧붙인 줄(꼬리표 · 이름 뒤 괄호 설명)은 맨 이름 「이름 = 값」 줄 — 이름 바로 뒤 값 줄 — 이 따로 있을 때만 뺍니다.
    # 문장 속의 이름이나 다른 줄의 이름(「… (분모 2,000건)」 · 「결론 : … 이름은 9월의 0.933배 …」)이 있다고 괄호 설명 줄을 버리면
    # 「분모(보낸 리뷰 수) = 2,000 건」 처럼 설명을 붙인 맞는 값 줄을 잃습니다
    if _bi.any(o["val"][3] and not o["tag"] and not o["noted"] for o in found):
        found = [o for o in found if not o["tag"] and not o["noted"]]
    direct = [o for o in found if o["val"][3]]
    return ([o for o in direct if o["line"] in val_lines or o["val"][5] != "bare"]
            or [o for o in found if o["line"] in val_lines] or direct or found)


def _text_tails(lines, label, others=()):
    """이름이 있는 줄마다 값 표지(「=」·「:」) 뒤의 글 — 값이 낱말 · 문장인 항목(권 검사)이 읽습니다.

    이름 바로 뒤가 표지인 줄(「이름 = …」 · 「이름 (설명): …」)이 있으면 그 줄만, 없으면 이름이 있는 모든 줄입니다
    (표지가 없는 줄은 이름 뒤 전체). 글은 다른 항목 이름에서 끊지 않습니다 — 내가 적은 문장에 다른 항목의 이름이 나와도 그대로입니다.
    others 는 다른 항목의 이름입니다 — 이 이름을 품은 더 긴 이름(「1위 토픽」 에 대한 「1위 토픽 비율」)의 줄을 가려냅니다.
    이름 앞에 다른 낱말이 붙은 줄(「전임 판단 : …」 — 다른 이름의 줄)과 꼬리표가 붙은 줄(「(전임 코드) 판단 : …」)은
    같은 급의 줄 가운데 이름으로 시작하는 줄이 따로 있으면 쓰지 않습니다(값 칸의 `_value_use` 와 같은 규칙). 「고른 품목 = 헬스장」
    처럼 그런 줄뿐이면 그 줄을 읽습니다 — 표지 없는 줄(「… · 품목 민원 4,000건」)이 있다고 버리지 않습니다.
    """
    whole, rest = [], []
    for o in _occurrences(lines, label, others):
        cut = _mark_at(o["after"])
        if cut >= 0 and _bare_gap(o["after"][:cut]):
            whole.append((o["after"][cut + 1:], o))
        else:
            rest.append((o["after"][cut + 1:] if cut >= 0 else o["after"], o))
    pick = whole or rest
    for mark in ("pre", "tag"):
        if _bi.any(not o[mark] for _, o in pick):
            pick = [(t, o) for t, o in pick if not o[mark]]
    return [t for t, _ in pick]


def _value_lines(txt, units):
    """값 줄 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    이름은 출력의 모든 행에서 찾습니다(「이름: 값」 꼴 포함). 이름 바로 뒤 값 줄이 하나도 없을 때만 이 행들의 값을
    먼저 씁니다 — 이 행들에서 값이 읽혔다고 「이름: 값」 줄을 보지 않으면, 틀린 값을 「:」 로 찍고 맞는 값을 「=」 로 찍은
    출력의 충돌을 놓칩니다. 이름이 있는 행이 아예 없으면 숫자를 읽지 않고 ❌ 로 알립니다(아래 `judge`).
    """
    return [line for line in (txt or "").splitlines()
            if line.strip() and ("=" in line or _bi.any(unit and unit in line for unit in units))]


def _short_digits(raw, value, lo, hi):
    """찍힌 자릿수가 모자라 범위 밖으로 읽혔으면 그 자릿수, 아니면 None.

    「0.10」 · 「205」 처럼 찍힌 자리로 반올림하면 기준 범위와 겹치는데 값 자체는 범위 밖이면, 계산이 아니라
    출력 자릿수를 짚어야 학생이 엉뚱한 원인을 찾지 않습니다. 소수 한 자리(「0.1」)는 자릿수를 줄인 값인지
    조회표 끝의 p 값인지 가를 수 없어 보지 않습니다.

    겹침은 경계를 빼고 봅니다 — 반올림 구간의 끝이 기준 범위의 끝에 닿기만 한 값은 자릿수 문제가 아닙니다.
    개수처럼 정수인 칸은 기준 범위를 n±0.5 로 잡는데, 바로 옆 정수(n−1 · n+1)의 반올림 구간이 그 끝에 딱 닿습니다.
    닿은 것까지 겹침으로 치면 틀린 개수(167 · 169)가 「자릿수 모자람」 이 되고 같은 이름 줄의 충돌에서도 빠집니다.
    """
    raw = (raw or "").replace(",", "").lstrip("+-")
    if not raw or "e" in raw.lower() or lo <= value <= hi:
        return None
    places = len(raw.split(".", 1)[1]) if "." in raw else 0
    if places == 1:
        return None
    half = 0.5 * 10 ** (-places)
    edge = half * 1e-6                      # 부동소수 오차로 「닿음」 이 「겹침」 으로 읽히지 않게 두는 여유
    return places if value - half < hi - edge and value + half > lo + edge else None


# ───────────────────────── 문구 · 축 이름 ─────────────────────────
_NEG_TAIL = ("하지못", "하지않", "하지는못", "하지는않", "되지않", "되지못", "할수없", "못", "아님", "아니",
             "안되", "안됐", "안돼", "안됨", "안됩", "안함", "안했", "불가", "에실패", "실패",
             "에이르지못", "에이르지않", "까지는못", "까지못")
_NEG_MARK = r"(?:False|false|FALSE|No|no|NO|N|X|x|×|✗|✘|❌)(?![A-Za-z])"
_YES_MARK = r"[=:]?(?:True|true|TRUE|Yes|yes|YES|O|예|네|있음|함)(?![A-Za-z])"


def _flat(txt):
    """빈칸을 걷고 겉보기가 같은 대시·가운뎃점을 맞춘 글자"""
    return "".join(_GLYPH.get(ch, ch) for ch in _re.sub(r"\s+", "", txt or ""))


def _word_in(row, word):
    """빈칸을 걷은 줄 `row` 에 낱말이 **긍정으로** 있는가.

    「비기각」·「미기각」·「기각하지 못함」·「기각 안 됨」·「기각에 이르지 못함」·「기각 X」·「기각 = False」 는
    「기각」 을 글자로 품지만 뜻은 반대입니다. 「기각 여부」 뒤에 답(True·예)이 없으면 무엇도 말하지 않은 것이고,
    「비정상 쪽 증거 없음」 도 반대입니다. 그 꼴을 지운 뒤에도 낱말이 남아야 채운 것으로 봅니다
    (「비정상 쪽」 안의 「정상 쪽」 도 같은 방식으로 지웁니다). 금지 문구(forbid)는 이 규칙을 쓰지 않습니다 —
    머리말·옛 이름에 낱말용 부정 꼴을 대면 문구째 지워집니다(`_forbid_in`).
    """
    w = _flat(word)
    if not w:
        return True
    e = _re.escape(w)
    neg = "|".join((
        "(?:비|미|불)" + e,                                            # 비기각 · 미기각 · 불기각
        e + "(?:" + "|".join(_NEG_TAIL) + ")",                          # 기각하지 못함 · 기각 안 됨 · 기각에 이르지 못함
        e + r"(?:여부)?[=:]?" + _NEG_MARK,                              # 기각 X · 기각 = False · 기각 여부 = False
        e + r"의?증거(?:는|가|도)?(?:없|부족|불충분|아님|아니|못)",           # 비정상 쪽 증거 없음
        e + r"여부(?!" + _YES_MARK + ")",                               # 「기각 여부」 뒤에 답이 없다
    ))
    return w in _re.sub(neg, "", _flat(row))


#  금지 문구(forbid) 뒤에서 그 문구를 서술로 부정한 꼴 — 「…넣어도 되지 않습니다」·「운영 가능하지 않다」·「…이 아니다」
_FORBID_NOT = (r"(?:은|는|이|가|도|고|라고|다고)?(?:(?:하|되)지(?:는|도)?(?:않|못)|할수(?:는|도)?없|안(?:되|된|됩|돼|됨|됐)"
               r"|아니|아님|아닙)")


#  줄 머리말(「판단 :」) 바로 뒤에서 그 줄을 지웠다고 알리는 말 — 「판단 : 행은 삭제했습니다」·「판단 : 줄은 지웠습니다」
_FORBID_GONE = (r"(?:행|줄|문장|결론)(?:은|을|도|이|만)?(?:모두|전부|이제)?"
                r"(?:지웠|지움|지우|지운|삭제|뺐|뺌|빼고|뺀|없앴|없앰|없애|제거|남기지않|출력하지않|찍지않)")


def _forbid_in(row, phrase):
    """빈칸을 걷은 줄 `row` 에 금지 문구가 남아 있는가 — 함정 미션의 `forbid`.

    「판단 :」 처럼 「:」 로 끝나는 문구는 줄 머리말입니다. 줄의 맨 앞이나 쉼표·화살표 같은 구분 기호 바로 뒤에 올 때만
    남은 줄입니다 — 「누수 여부 판단: …」 처럼 다른 말 뒤에 이어 쓴 것, 따옴표 안에 옮겨 적은 것은 머리말이 아닙니다.
    머리말 뒤에 오는 말은 가리지 않습니다. 「판단 : 불가」·「판단 : X」·「판단 : 아니다」 도 판단 줄이고, 전각 「：」 도
    같은 표지입니다. 하나만 가립니다 — 머리말 바로 뒤가 그 줄을 지웠다는 말(「(판단 : 행은 삭제했습니다)」)이면 PROMPT 의
    「판단 : 행은 지워」 를 되받아 알린 글이라 남은 줄이 아닙니다(`_FORBID_GONE`).

    그 밖의 문구(옛 이름 · 전임 담당자의 결론)는 글자로 대조합니다. 문구 바로 뒤에서 서술로 부정한 인용(「…넣어도 되지
    않습니다」)과 답 없는 「… 여부」 만 남은 줄이 아닙니다. 낱말용 부정 꼴(`_word_in` 의 비·미·불 접두, 못·불가·실패,
    X·No·N)은 쓰지 않습니다 — 「계절 나이브 대비 제로샷 평균 MASE」 의 「비」, 「제로샷 평균 MASE: N/A」 의 「N」,
    「판단 : 불가」 의 「불가」 가 문구를 지워 남은 줄이 통과하던 자리입니다.
    """
    w = _flat(phrase).replace("：", ":")
    if not w:
        return False
    row = _re.sub(r"[*`]", "", row or "").replace("：", ":")
    e = _re.escape(w)
    if w.endswith(":") and len(w) > 1:
        # 「(판단 : 행은 삭제했습니다)」 — 머리말 바로 뒤가 그 줄을 지웠다는 말이면 남은 줄이 아니라 지웠다고 알리는 글입니다
        return _bi.bool(_re.search(r"(?:^|[^0-9A-Za-z가-힣_'\"「『“‘])" + e + "(?!" + _FORBID_GONE + ")", row))
    return w in _re.sub(e + "(?:" + _FORBID_NOT + "|여부(?!" + _YES_MARK + "))", "", row)


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _sup_label(ax, which):
    """그림 수준 축 이름(fig.supxlabel · fig.supylabel)이 있는가 — 패널이 여럿인 그림에서 축마다 이름을 다는 대신 씁니다"""
    fig, seen = getattr(ax, "figure", None), set()
    while fig is not None and id(fig) not in seen:   # SubFigure 면 바깥 그림까지 올라갑니다
        seen.add(id(fig))
        text = getattr(fig, "_sup" + which + "label", None)
        try:
            if text is not None and text.get_text().strip():
                return True
        except Exception:
            pass
        fig = getattr(fig, "figure", None)
    return False


def _mate_label(ax, which):
    """가로축(which="x")·세로축("y")을 나눠 쓰는 다른 축에 그 이름이 있는가.

    `ax.twinx()` 로 겹친 두 축은 가로축 이름을 한쪽에, 세로축 이름을 다른 쪽에 다는 일이 흔하고, `sharex=True` 로 쌓은
    그림은 맨 아래 축에만 가로축 이름을 답니다. 화면에는 두 이름이 다 보이므로 짝의 이름도 이 축의 이름으로 봅니다.
    나눠 쓰지 않는 옆 칸의 이름(`subplots(1, 2)` 의 한 칸에 가로축 이름, 다른 칸에 세로축 이름)은 보지 않습니다.
    """
    mates = []
    for find in (lambda: getattr(ax, "get_shared_" + which + "_axes")().get_siblings(ax),
                 lambda: ax._twinned_axes.get_siblings(ax)):
        try:
            mates += [a for a in find() if a is not ax]
        except Exception:
            pass
    for a in mates:
        try:
            if getattr(a, "get_" + which + "label")().strip():
                return True
        except Exception:
            pass
    return False


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과).

    그래프 종류(막대·가로 막대·선·점, pandas·seaborn 으로 그린 것)는 가리지 않고 가로축·세로축 이름만 봅니다.
    `fig.supxlabel`·`fig.supylabel` 로 그림 전체에 단 이름, 축을 나눠 쓰는 짝(twinx · sharex)에 단 이름도 축 이름입니다.
    기준이 `가로축라벨` 이면(세로 눈금에 막대 이름이 붙는 가로 막대 그림) 가로축 이름만 봅니다.
    """
    x_name = ax.get_xlabel().strip() or _sup_label(ax, "x") or _mate_label(ax, "x")
    y_name = ax.get_ylabel().strip() or _sup_label(ax, "y") or _mate_label(ax, "y")
    if spec.get("축라벨") and not (x_name and y_name):
        return ["가로축·세로축 이름 가운데 빈 것이 있습니다"]
    if spec.get("가로축라벨") and not x_name:
        return ["가로축 이름이 비어 있습니다"]
    return []


def _order_of(tail):
    """「공식 차수」 줄 꼬리의 비계절 차수 — ((p, d, q), 이름 바로 뒤 값 줄인가) · 못 읽으면 None.

    값 표지(괄호 밖 첫 「=」·「:」) 뒤에서 **정수 3개가 든 첫 괄호 묶음**을 먼저 찾습니다(계절 차수까지 한 묶음에 적은
    7개도). 「= 1위 조합 (1,0,1)(0,1,0,24)」·「= AIC 1위 (1, 0, 1) x (0, 1, 0, 24)」 처럼 괄호 앞에 숫자가 든 말이 와도
    괄호 안을 읽습니다. 그런 묶음이 없으면 표지 뒤 앞 숫자 3개입니다 — 「= 1, 0, 1」. (1,0,1) · (1.0, 0, 1.0) ·
    SARIMA(1,0,1)(…) 를 같게 읽고, 이름 뒤 괄호 설명(「(p=1, d=0, q=1 로 적합)」)은 건너뜁니다.
    진행 막대 줄(「17%|█▋ | 1/6 [00:00<…」)은 차수 줄이 아닙니다.
    """
    t, depth, cut = _unwrap_np(tail), 0, -1
    if _re.search(r"\d+%\|", t):
        return None
    for i, ch in enumerate(t):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
        elif ch in "=:：＝" and depth == 0:
            cut = i
            break
    rest = t[cut + 1:] if cut >= 0 else t
    direct = cut >= 0 and _bare_gap(t[:cut])
    num = r"(?<![A-Za-z_\d.])-?\d+(?:\.\d+)?"
    picks = [_re.findall(num, a or b) for a, b in _re.findall(r"\(([^()\[\]]*)\)|\[([^()\[\]]*)\]", rest)]
    nums = next((p for p in picks if len(p) in (3, 7)), None) or _re.findall(num, rest)
    nums = nums[:3]
    if len(nums) < 3 or _bi.any(float(x) != int(float(x)) for x in nums):
        return None
    return tuple(int(float(x)) for x in nums), direct


# ───────────────────────── 채점 상태 · 그림 ─────────────────────────
_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set(), "src": "", "prompt": None, "pub": None,
          "lost": None, "imgs": 0}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트·기준 범위·빠진 낱말이 이 수를 봅니다
_orig_figure = getattr(_plt.figure, "_orig", _plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, _orig=_orig_figure, _figs=_FIGS, **k):
    # 학생 코드가 그림을 만들 때 불립니다 — 채점 코드의 이름을 기본값으로 묶어 둡니다(학생 셀이 같은 이름을 만들어도 그대로)
    f = _orig(*a, **k)
    _figs.append(f)
    return f


_figure._orig = _orig_figure
_plt.figure = _figure                      # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _device_spec(spec):
    """실행 경로(CPU · GPU)마다 기준이 다른 미션 — 기준의 `by_device` 에서 지금 DEVICE 의 한 벌을 덮어씁니다.

    준비 셀이 DEVICE 를 다시 정해도 채점 시점의 값을 따릅니다("cuda" 로 시작하면 GPU 기준, 그 밖은 CPU 기준).
    """
    per = (spec or {}).get("by_device")
    if not per:
        return spec
    dev = "cuda" if str(_get("DEVICE") or "cpu").startswith("cuda") else "cpu"
    return {**{k: v for k, v in spec.items() if k != "by_device"}, **dict(per.get(dev, {}))}


def _spec_of(key):
    """이 미션의 채점 기준 — 실행 경로별 기준을 맞추고(`_device_spec`), 권 설정의 `spec` 으로 이 세션의 기준을 다시 계산합니다"""
    spec = _device_spec(_ANS.get(key))
    hook = _BOOK.get("spec")
    if hook is not None and spec is not None:
        try:
            spec = hook(key, spec) or spec      # 이 세션에서 다시 계산한 기준(권 설정)
        except Exception:
            pass
    return spec


def _takes_ctx(fn):
    """권 검사 함수가 셋째 인자(출력 글 · 그림 · 실행 상태)를 받는 꼴인가 — `fn(key, spec, ctx)`. 옛 꼴은 `fn(key, spec)` 입니다"""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    n = code.co_argcount - (1 if getattr(fn, "__self__", None) is not None else 0)
    return n >= 3 or _bi.bool(code.co_flags & 0x04)


def _ai_lines(src):
    """셀 원문에서 「여기부터 AI 코드」 표시선 아래의 코드 줄 — 주석은 뗍니다(표시선이 없으면 빈 목록).

    셀이 표시선 줄에서 끝나면(줄바꿈 없이 — 코드를 받기 전의 PROMPT 셀) 표시선 줄의 나머지 글자(「 ---」)를 코드 줄로 돌려주지 않습니다.
    """
    src = str(src or "")
    if "여기부터 AI 코드" not in src:
        return []
    rest = src.split("여기부터 AI 코드", 1)[1]
    body = rest.split("\n", 1)[1] if "\n" in rest else ""
    return [ln.split("#", 1)[0] for ln in body.splitlines()]


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [_plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in _plt.get_fignums()]
    except Exception:
        return []


def _num_txt(v, fmt=",.6g"):
    """채점 문장에 찍을 수 — 음수 부호는 권 설정(`minus`)을 따릅니다(본문이 − U+2212 를 쓰는 권)"""
    return format(v, fmt).replace("-", _BOOK.get("minus", "-"))


def _val_txt(val):
    """읽은 값(`_read_value` 결과)을 채점 문장에 찍을 글자 — 찍힌 자리까지 보여 줍니다.

    유효숫자 6자리로만 찍으면 학생이 출력한 「25,209.34」 가 「25,209.3」 으로, 「1,052,176.7」 이 「1.05218e+06」 으로 보여
    자리가 잘린 값으로 읽힙니다. 6자리로 줄였을 때 찍힌 자리의 숫자가 사라지면 찍힌 소수 자릿수대로 찍습니다.
    끝의 0 은 뗍니다(91.0 → 91). 소수 7자리 넘게 찍힌 값(반올림하지 않은 0.2812250064155164)은 지금처럼 유효숫자 6자리이고
    (지수 표기가 되는 큰 수만 소수 6자리), 지수 표기로 찍힌 값은 지수 표기입니다.
    """
    v, raw = val[0], str(val[1] or "")
    short = _num_txt(v)
    places = len(raw.split(".", 1)[1]) if "." in raw else 0
    if "e" in raw.lower() or v != v or v in (float("inf"), float("-inf")) or (places > 6 and "e" not in short):
        return short
    places = _bi.min(places, 6)
    full = format(v, ",." + str(places) + "f")
    full = full.rstrip("0").rstrip(".") if "." in full else full
    try:
        same = float(format(v, ",.6g").replace(",", "")) == float(full.replace(",", ""))
    except ValueError:
        return short
    return short if same else full.replace("-", _BOOK.get("minus", "-"))


_ANSI_RE = _re.compile(r"\x1b\[[0-9;?]*[ -/]*[@-~]")       # 굵게·색을 켜고 끄는 글자 — 화면에는 보이지 않습니다


def _clean_out(text, prompt=""):
    """채점이 읽을 출력 — 화면에 보이지 않는 글자를 지우고 PROMPT 를 그대로 찍은 글을 뺍니다.

    굵게 찍으려고 넣은 ANSI 코드(ESC[1m · ESC[0m)는 「[」 가 있어 열린 괄호로 읽히고, `print(PROMPT)` 로 찍은 지시문의
    「"이름 = 값" 한 행으로 출력하고, Q_ZS 의 0.5 분위수로 …」 는 이름 줄이 되어 0.5 가 값으로 읽힙니다.
    PROMPT 글은 값 줄이 아닙니다.
    """
    txt = _ANSI_RE.sub("", text or "")
    echo = str(prompt or "").strip()
    if len(echo) >= _PROMPT_MIN and echo in txt:
        txt = txt.replace(echo, "\n")
    return txt


# ───────────────────────── PROMPT 칸 ─────────────────────────
def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다.

    문법 오류는 임시 파일 이름 대신 셀의 몇째 줄인지 적고, 예외의 문장을 읽다가 다시 오류가 나도
    채점 표가 닫히도록 그 자리는 안내 문구로 채웁니다.
    """
    try:
        if isinstance(e, SyntaxError) and getattr(e, "msg", None):
            txt = str(e.msg) + (f" (셀 {e.lineno}번째 줄)" if getattr(e, "lineno", None) else "")
        else:
            txt = str(e)
    except Exception:
        txt = "(오류 문장을 읽지 못했습니다)"
    txt = txt.replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt_in_cell(src):
    """셀 원문의 PROMPT 따옴표 안 글자 — 셀을 실행하기 전에 읽어 둡니다.

    이름 공간의 PROMPT 를 읽으면 PROMPT 블록을 지운 셀이 앞 미션에서 채운 PROMPT 로 통과하고, 문법 오류로
    셀이 돌지 않은 날은 채운 PROMPT 도 앞 값으로 보입니다. 셀에 PROMPT 대입이 없으면 "" 이고, 따옴표 꼴이 아니라
    읽지 못하면 None 입니다. 셀이 돌았으면 따옴표 안 글자 대신 실행된 PROMPT 값을 씁니다(`_post`).
    """
    src = str(src or "")
    if not _re.search(r"(?m)^[ \t]*PROMPT[ \t]*=(?!=)", src):
        return ""
    lit = _re.search(r"^[ \t]*PROMPT[ \t]*=[ \t]*[rRuU]?(\"\"\"|'''|\"|')(.*?)\1", src, _re.M | _re.S)
    return lit.group(2) if lit else None


def _prompt_now(given=None):
    """채점할 PROMPT 글자 — 셀 원문에서 읽은 것(given)이 있으면 그것, 없으면 이름 공간의 PROMPT"""
    if given is not None:
        return str(given)
    try:
        return str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "")
    except Exception:
        return str(_get("PROMPT") or "")


def _prompt1(n=70, given=None):
    """지금 셀의 지시 첫 줄 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in _prompt_now(given).split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                        "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return _re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing(given=None):
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    txt = _prompt_now(given).strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return "PROMPT 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return "PROMPT 입력란이 처음 적혀 있던 견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 영역이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


# ───────────────────────── 오류 · 흔한 실수 ─────────────────────────
def _error_hint(key, error):
    """오류에 맞는 (힌트, 물을 것).

    문법 오류(SyntaxError·IndentationError)는 셀이 한 줄도 돌지 않은 것이라 붙여 넣은 글자를 짚습니다.
    노트북에 있는 이름을 잘못 옮겨 적은 NameError 면 그 이름을 짚고, 권 설정의 `error_hint(key, error)` 가 (힌트, 물을 것)을
    돌려주면 그것, 권의 힌트에 오류 종류별 힌트(`error_kinds`)가 있으면 그것, 없으면 미션 공통 힌트입니다
    (준비 셀·설치 셀이 만든 이름과 다시 실행할 셀).
    """
    kind, msg = type(error).__name__, _one_line(error, 400)
    tbl = _HINT.get(key, {})
    if isinstance(error, SyntaxError):
        line_no = getattr(error, "lineno", None)
        at = f"셀 {line_no}번째 줄" if line_no else "오류 문장이 가리키는 줄"
        head = "문법 오류라 이 셀의 코드가 한 줄도 실행되지 않았습니다. "
        if "```" in (_STATE.get("src") or ""):
            return (head + "AI 답에서 코드를 옮길 때 「```python」·「```」 같은 코드 블록 표시 행(펜스 줄)이 함께 들어왔습니다."
                    " 표시 행을 지우고 그 안쪽 코드만 남긴 뒤 다시 실행하세요",
                    "붙여 넣은 코드의 첫 행과 끝 행에 파이썬이 아닌 행이 있는지")
        if isinstance(error, (IndentationError, TabError)):
            return (head + f"들여쓰기가 어긋났습니다 — {at}과 바로 위 행의 들여쓰기 칸 수를 비교하고,"
                    " 같은 블록의 행은 같은 칸 수로 맞추세요",
                    "이 행의 들여쓰기가 위 행과 어떻게 어긋났는지")
        return (head + f"{at}에서 괄호·따옴표가 닫혔는지, 코드가 아닌 설명 문장이 섞였는지,"
                ' PROMPT 에 적을 문장을 PROMPT = """ """ 따옴표 밖에 두지 않았는지 보세요',
                "이 문법 오류가 어느 행에서 왜 났는지")
    if kind == "NameError":                # 노트북에 있는 이름을 잘못 옮겨 적은 경우(FIG_W → FGW)는 그 이름을 짚습니다
        import difflib
        bad = _re.search(r"name '(\w+)'", msg)
        near = bad and difflib.get_close_matches(
            bad.group(1), [n for n in globals() if not n.startswith("_") and n != "FIGW"], n=1, cutoff=0.7)
        if near:
            return (f"「{bad.group(1)}」 이름은 이 노트북에 없습니다. 비슷한 이름은 「{near[0]}」 입니다."
                    f" 잘못 옮겨 적었다면 코드에서 「{bad.group(1)}」 를 「{near[0]}」 로 고쳐 다시 실행하세요",
                    "이 오류가 어느 행에서 왜 났는지")
    hook = _BOOK.get("error_hint")          # 권 설정 — 셀 위쪽에 내가 적은 값의 꼴이 어긋나 난 오류처럼 권만 아는 까닭
    if hook is not None:
        try:
            own = hook(key, error)
        except Exception:
            own = None
        if own and own[0]:
            return own[0], (own[1] if len(own) > 1 and own[1] else "이 오류가 어느 행에서 왜 났는지")
    for rule in tbl.get("error_kinds", ()):
        if _bi.any(kind == k and _re.search(pat, msg) for k, pat in rule.get("match", ())):
            return rule.get("hint", ""), rule.get("ask") or "이 오류가 어느 행에서 왜 났는지"
    hint, question = tuple(tbl.get("error", ("", "이 오류가 어느 행에서 왜 났는지")))[:2]
    return hint, question


def _note_for(rules, values, tail_text):
    """값이 흔한 실수 하나의 값과 같으면 그 원인 문장 — 규칙은 (하한, 상한, 문장) · ("nan", 문장) (`_CAUSE` 블롭)"""
    for rule in rules or ():
        if rule[0] == "nan":
            if "nan" in (tail_text or "").lower():
                return rule[1]
        elif _bi.any(float(rule[0]) <= v <= float(rule[1]) for v in values):
            return rule[2]
    return ""


def _stale_code(rule, src, lo, hi):
    """값은 기준 범위 안인데 그 값을 코드에 손으로 적었고 shift 인자는 전임 코드의 틀린 값뿐이면 그 사실 한 문장.

    함정 미션에서 전임 코드의 `shift(24)` 를 그대로 둔 채 기준 범위의 숫자를 코드에 적어 출력하면 값만 맞고
    고친 곳이 없습니다. 그 꼴만 거릅니다(`_CAUSE["code"]` 규칙이 있는 미션만).
    """
    if not rule or not src:
        return ""
    body = src.split("여기부터 AI 코드", 1)[-1]           # PROMPT 안의 말은 보지 않습니다
    body = "\n".join(l.split("#", 1)[0] for l in body.splitlines())
    args = [_re.sub(r"\s+", "", a) for a in _re.findall(r"\.shift\(([^()]*)\)", body)]
    if not (args and _bi.all(a in rule[0] for a in args)):
        return ""
    if not _bi.any(lo <= v <= hi for v in _nums(body)):
        return ""
    return rule[1]


def _ask_line(form, name, prompt, question):
    """튜터에 보낼 질문 한 줄 — 권의 `_ASK_FORM` 틀에 채웁니다"""
    tmpl = _ASK_FORM.get(form) or _ASK_FORM.get("full") or "{name} — {ask}"
    return "  " + tmpl.format(name=name, prompt=_prompt1(given=prompt), ask=question)


_FORM_HINT = (                              # 이름·숫자를 못 찾은 ❌ — 값 계산이 아니라 출력 꼴을 짚습니다
    "값 계산보다 출력 꼴이 먼저입니다 — 이름과 값을 「이름 = 숫자」 꼴로 한 줄에 찍고, 이름은 「만들 것」 의 굵은 이름과"
    " 글자까지 같게 적도록 PROMPT 의 출력 형식에 적으세요. 표(DataFrame)·딕셔너리·배열로 띄웠다면 항목마다 print 한 줄로 바꿉니다",
    "이 값을 출력하는 행이 이름과 값을 어떤 꼴로 찍는가")
_DIGITS_HINT = (                            # 반올림하면 기준 범위에 닿는 값 — 계산보다 자릿수를 짚습니다
    "값은 맞았을 수 있습니다 — 출력 형식의 자릿수(:.2f 같은 서식)가 「만들 것」 보다 적어 반올림된 값이 찍혔습니다."
    " PROMPT 의 출력 형식에 「만들 것」 의 소수점 아래 자릿수를 적고 다시 실행하세요",
    "이 값을 소수점 아래 몇 자리로 출력하도록 코드가 정했는가")
_DUP_HINT = (                               # 같은 이름 줄이 여럿이고 값이 다르면 — 어느 줄이 최종 답인지 모릅니다
    "같은 이름으로 값을 두 번 이상 찍었습니다 — 전임 코드나 시험 삼아 찍은 옛 print 를 지우고, 최종 값 한 줄만 남기도록"
    " PROMPT 에 적으세요",                  # 함정 미션의 힌트(전임 담당자 코드가 있는 미션)
    "같은 이름을 찍는 print 가 코드 어디 어디에 있는가")
_DUP_HINT_PLAIN = (                         # 함정이 아닌 미션 — 비교 · 중간 결과를 출력하며 같은 이름을 쓴 경우를 짚습니다(0.4.2, 분야 낱말 없이)
    "비교 결과나 중간 결과를 출력할 때도 이 이름을 쓰면, 함께 출력한 수도 이 항목의 값으로 읽습니다. 이름은 최종 값을 출력할 때만"
    " 쓰도록 PROMPT 에 적으세요",
    _DUP_HINT[1])


def _hint_key(key, kind, label):
    """❌ 의 종류에 맞는 힌트 열쇠 — 권의 힌트(`_HINT`)에 그 종류의 열쇠가 있으면 그것, 없으면 공통 힌트"""
    own = _HINT.get(key, {})
    pick = {"form": ("이름_형식", "label"), "dup": ("줄_중복", "dup"), "digits": ("round",), "unit": ("unit",)}
    for cand in pick.get(kind, ()):
        if cand in own:
            return cand
    return {"form": "_form", "dup": "_dup", "digits": "_digits"}.get(kind, label)


def _is_trap(key):
    """함정 미션인가 — 빈 PROMPT 에도 잠그지 않고 표시선 아래 전임 담당자 코드를 진단하는 미션(권 토막의 `_blocks`)"""
    try:
        return not _blocks(key, "PROMPT 입력란이 비어 있습니다")
    except Exception:
        return False


def _hint_parts(tiers):
    """권의 힌트 한 칸 → (힌트 단 목록, 튜터에 물을 것) — 원소가 둘이면 (힌트, 물을 것), 셋 이상이면 마지막이 물을 것"""
    tiers = (tiers,) if isinstance(tiers, str) else tuple(tiers or ())
    steps = tiers[:-1] if len(tiers) > 2 else tiers[:1]
    return [t for t in steps if t], (tiers[-1] if len(tiers) > 1 else "")


# ───────────────────────── 채점 표 ─────────────────────────
def judge(key, text="", figs=(), error=None, prompt=None, shown=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다.

    prompt 는 셀 원문에서 읽은 PROMPT(없으면 이름 공간의 PROMPT), shown 은 그림이 화면에 떴는지
    (True·False, 알 수 없으면 None — inline 백엔드에서만 셉니다)입니다.
    """
    spec = _spec_of(key)                   # 실행 경로별 기준(by_device) · 이 세션에서 다시 계산한 기준(권 설정)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    if spec.get("path_note"):              # 실행 경로(CPU · GPU)마다 기준이 다른 미션 — 어느 기준으로 채점하는지 먼저 알립니다
        print(f"ⓘ 채점 기준: {spec['path_note']}")
    reason = _prompt_missing(prompt)                  # 값·그래프보다 먼저 — 내 말로 시킨 지시가 있는가
    _locked = _bi.bool(reason)
    if _blocks(key, reason):                       # 권 토막의 `_blocks` — 함정 미션은 빈 PROMPT 에도 진단합니다
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print('   AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적었는지 확인하세요')
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션은 PROMPT 가 비어도 표시선 아래 코드를 진단합니다(지시서 §9-10 · ⓘ 한 줄)
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다. 완성본 풀이는 PROMPT 를 채운 뒤에 열립니다")
        print("   (표시선 아래를 이미 고쳤다면 아래는 AI 코드 표시선 아래 코드의 진단입니다)"
              ' 고칠 지시는 AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적습니다')
    form = _ASK.get(key, "full")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        kind = type(error).__name__                  # 괄호가 안 닫힌 셀은 3.13 에서 _IncompleteInputError 입니다
        kind = "SyntaxError" if isinstance(error, SyntaxError) and kind.startswith("_") else kind
        msg = _one_line(error)
        print(f"❌ 실행 오류 — {kind}" + (f": {msg}" if msg else ""))
        hint, question = _error_hint(key, error)          # 비슷한 이름이 있으면 그 이름, 없으면 미션 공통 힌트
        if hint:
            print(f"   힌트 : {hint}")
        print(_ASK_HEAD.get(form, ""))
        print(_ask_line(form, "실행 오류", prompt, question))
        print("─" * 54)
        return False
    txt = _clean_out(text, _prompt_now(prompt))
    trap = _is_trap(key)
    cell_src = str(_STATE.get("src") or "")
    if ("여기부터 AI 코드" in cell_src and not _bi.any(l.strip() for l in _ai_lines(cell_src))
            and not txt.strip() and not _bi.any(_axes(f) for f in figs)):
        # PROMPT 만 적고 ▶ 한 셀 — 틀린 것이 아니라 아직 코드를 받지 않은 것입니다. 값 · 그래프 ❌ 와 힌트를 늘어놓지 않고
        # 다음 할 일 한 줄만 알립니다(❌ 횟수에 넣지 않습니다)
        print("ⓘ 출력이 없습니다 — 「여기부터 AI 코드」 표시선 아래에 코드가 없습니다. "
              + ("전임 담당자 코드를 지웠다면 실행 취소로 되돌린 뒤, " if trap else "")
              + 'AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 하면 그 코드의 출력과 그림을 채점합니다')
        print("─" * 54)
        return False
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    values = spec.get("values", {})
    units = [str(ref[2]) for ref in values.values() if len(ref) > 2]
    val_lines = set(_value_lines(txt, units))               # 단위·「=」 가 붙은 행 — 이름 바로 뒤 값 줄이 없을 때 먼저 씁니다
    all_lines = [l for l in txt.splitlines() if l.strip()]
    series = _bi.bool(_SERIES_RE.search(txt))               # Series 를 통째로 찍은 흔적
    need_words = spec.get("need_words", {})        # 함정 미션 — {이름: [낱말, …]} 그 항목의 토막에 꼭 있어야 하는 낱말
    cause = _get("_CAUSE") or {}
    fail_notes = cause.get("fail", {}).get(key, {})            # 흔한 실수의 값 → 원인 문장
    pass_notes = cause.get("pass", {}).get(key, {})
    code_notes = cause.get("code", {}).get(key, {})
    find = {l: (r[3] if len(r) > 3 and r[3] else l) for l, r in values.items()}   # 출력에서 찾을 이름(짧은 이름 포함)
    names = list(find.values()) + (["공식 차수"] if spec.get("차수_표기") else [])
    skip = _BOOK.get("skip", {})
    own_too = []                                   # own_too: 출력 꼴 힌트에 그 항목의 권 힌트를 함께 낼 이름(아래 `not occ`)
    form_only = spec.get("form_only")              # 이름 줄이 없는 칸에 출력 꼴 힌트만 내는 미션(권 힌트 첫 단이 읽은 값을 전제로 쓴 글)
    forms = spec.get("forms", {})                  # {이름: "낱말 숫자 %"} 값과 낱말을 함께 찍는 칸의 출력 꼴
    paren_unit = spec.get("paren_unit", ())        # 「k 건 (r %)」 꼴로 찍는 칸 — 괄호 안의 기준 단위 수를 읽습니다(`_unit_value`)
    traps = spec.get("traps", {})                  # {이름: [(하한, 상한, 문장), …]} 흔한 실수의 값이면 그 실수를 짚는 힌트
    seen = {}                                      # 값 칸에서 읽은 값 — 권 검사가 값끼리 맞물리는지 볼 때 씁니다(ctx["values"])
    made = "고칠 것" if trap else "만들 것"          # 채점할 이름이 적힌 본문 소제목 — 함정 미션은 「고칠 것」, 다른 미션은 「만들 것」
    for label, ref in values.items():
        skip_if, skip_msg = skip.get((key, label), (None, ""))
        if skip_if is not None and skip_if():           # 대체 탐색을 쓴 날의 AIC 처럼 채점하지 않는 칸
            print(f"ⓘ {label} — {skip_msg}")
            continue
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        name = find[label]
        want_form = f"「{name} = {forms.get(label) or ('숫자' + ((' ' + unit) if unit else ''))}」"
        where = "이 이름이 있는 줄" if name == label else f"이 이름(「{name}」)이 있는 줄"
        # 이 이름이 있는 줄에서 이름 뒤의 값 하나만 읽습니다. 이름이 있는 줄이 없을 때 출력 전체의 숫자로
        # 물러나면 다른 행의 값(부호를 뒤집은 차, 다른 모델의 값)으로 틀린 값이 ✅ 가 됩니다
        occ = _occurrences(all_lines, name, names, series, unit)   # 「이름 = 값」·「이름: 값」·「이름 값」 — 모든 행에서
        # 같은 이름 줄이 여럿이면 이름 바로 뒤 값 줄만 봅니다(`_value_use`). 그 줄들의 값이 모두 맞아야 통과입니다 — 하나만
        # 맞아도 통과시키면 옛 print 를 남긴 채 덧붙인 줄·부호를 뒤집은 줄이 헛통과합니다.
        use = _value_use(occ, val_lines)
        vals = [o["val"] for o in use]
        if label in paren_unit:                         # 「이름 = 14 건 (1.4 %)」 — 괄호 안의 기준 단위 수가 값입니다
            vals = [_unit_value(v, unit) for v in vals]
        in_range = [v for v in vals if lo <= v[0] <= hi]
        clash = [v for v in vals if not (lo <= v[0] <= hi) and _short_digits(v[1], v[0], lo, hi) is None]
        # 이름 바로 뒤 값 줄이 하나도 없어 「이름 = 값」 꼴이 아닌 줄에서 읽은 값 — 이름 앞에 다른 말이 붙은 줄(더 긴 이름일 수 있는 줄:
        # 「상태별 리뷰 수 : 통과 178 …」)이나, 값 표지 없이 이름과 숫자 사이에 다른 말이 있는 줄(결론 문장 「…하루 평균은 9월의
        # 0.933배로…」 의 9). 범위 안이면 그 값을 받고(「전체 리뷰 수 = 200 건」 · 「리뷰 수는 200 건입니다」), 범위 밖이면 이 이름의
        # 값으로 읽지 않습니다 — 다른 항목의 값이나 문장 속의 수로 「읽은 값은 … 범위 밖」 과 계산 힌트를 내면 거짓 원인이 됩니다.
        # 이름과 값 표지 사이에 설명이 낀 줄(「이름 — period=24, robust=False → 0.8115」)은 지금처럼 그 값을 읽습니다
        weak = _bi.bool(use) and (use[0]["pre"] or (not use[0]["val"][3] and use[0]["val"][5] == "bare"))
        foreign = use[0]["line"].strip() if weak and not in_range else ""
        if vals and not foreign:
            seen[label] = (in_range or vals)[0][0]
        # 개수 자리에 찍은 비율 — 기준 단위가 % 로 시작하는 칸(% · %p)은 값 뒤의 % 가 제 단위입니다
        pct = (next((v for v in in_range if v[2].lstrip().startswith("%")), None)
               if unit and not unit.startswith("%") else None)
        stale = _stale_code(code_notes.get(label), _STATE.get("src"), lo, hi) if in_range else ""
        words = need_words.get(label, ())
        words = (words,) if isinstance(words, str) else tuple(words)
        missing = []
        if in_range and words:          # 값은 기준 범위 안 — 그 항목의 토막에 필요한 낱말이 있는가
            # 값 칸 하나만 고치고 전임자의 틀린 결론 문구(「→ 정상」)를 그대로 둔 줄이 통과하지 않게 합니다.
            # 줄 전체가 아니라 이 항목의 토막 — 같은 줄의 다른 항목 낱말은 보지 않습니다
            rows = []
            for o in _occurrences(all_lines, name, names, series):
                flat = _re.sub(r"\s+", "", o["seg"])
                rows += [flat, _re.sub(r"\([^()]*\)", "", flat)]
            missing = [w for w in words if not _bi.any(_word_in(row, w) for row in rows)]
        if in_range and pct is not None:              # 개수 자리에 비율을 찍은 판 — 범위가 넓어 수만으로는 못 거릅니다
            passed = False
            failed.append((label, _hint_key(key, "unit", label)))
            print(f"❌ {label} — {where}에서 읽은 값 {_val_txt(pct)} 뒤에 % 가 붙어 있습니다."
                  f" 비율이 아니라 {want_form} 처럼 개수로 출력하세요")
        elif in_range and clash:                      # 같은 이름, 다른 값
            passed = False
            failed.append((label, _hint_key(key, "dup", label)))
            shown_vals = " · ".join(_val_txt(v) for v in vals)
            print(f"❌ {label} — {where}이 {len(vals)}개이고 값이 서로 다릅니다({shown_vals})."
                  f" 틀린 값을 찍는 옛 줄을 지우고 {want_form} 한 줄만 남기세요")
        elif in_range and stale:                      # 값만 손으로 적고 코드는 그대로 둔 판
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 출력에서 {_val_txt(in_range[0])} 를 찾았지만 {stale}")
        elif in_range and missing:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 첫 ❌ 에는 빠진 낱말을 감춥니다
            print(f"❌ {label} — 값 {_val_txt(in_range[0])} 는 기준 범위 안이지만, 이 이름이 있는 줄의 문구가 기준과 다릅니다"
                  + (" (그 줄에 있어야 할 낱말 : " + " · ".join(f"「{w}」" for w in missing) + ")"
                     if _prev_fail else " — 무엇을 기각했고 어느 쪽 증거인지 같은 줄에 적었는지 보세요"))
        elif in_range:
            print(f"✅ {label} — 출력에서 찾은 값: {_val_txt(in_range[0])}")
            note = _note_for(pass_notes.get(label, ()), [in_range[0][0]], "")
            if note:
                print(f"   ⓘ {note}")
            if in_range[0][4] == "series":           # f-string 안의 Series 는 뒤에 적은 단위가 dtype 줄 끝으로 밀립니다 —
                # 단위 ⓘ 까지 겹치면 적은 단위를 안 적었다고 읽히므로, 출력 꼴을 고치라는 ⓘ 하나만 냅니다
                print("   ⓘ 표(Series)를 그대로 넣어 인덱스와 값이 함께 찍혔습니다 — 값만 읽었습니다."
                      " float(…) 로 숫자 하나만 출력하면 보고서에 그대로 옮길 수 있습니다")
            elif unit and unit not in " ".join(o["tail"] for o in use):   # 단위는 그 이름 줄에서 봅니다
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif not occ or foreign:
            passed = False
            # 계산 힌트가 아니라 출력 꼴 힌트 — 함정 미션의 낱말 항목은 행 이름·문구를 함께 안내하는 항목 힌트 그대로
            form_key = _hint_key(key, "form", label)
            failed.append((label, label if words else form_key))
            # 함정 미션의 전임 담당자 코드가 이 이름의 줄을 아예 찍지 않는 권이 있습니다 — 공통 출력 꼴 힌트만 내면 그 함정을
            # 짚는 항목 힌트와 튜터에 물을 것이 사라지므로, 권이 그 항목에 적어 둔 힌트 첫 단과 물을 것을 함께 냅니다.
            # 두 경우는 지금처럼 출력 꼴 힌트만 냅니다 — 권에 출력 꼴 힌트(이름_형식 · label)가 따로 있으면 그 힌트와 물을
            # 것이 이미 권의 것이고, 낱말을 채점하는 함정 미션은 전임 담당자 코드가 이름 줄을 이미 찍으며 낱말 없는 항목의
            # 힌트가 그 줄의 결론 문구를 짚는 글이라 이름 줄이 없을 때 내면 사실과 다를 수 있습니다
            # 권 힌트 첫 단이 읽은 값을 전제로 쓴 글이면(기준값을 예로 든 글) 이름 줄이 없는 화면에 내지 않습니다 — 기준의 `form_only`
            if (trap and not need_words and not form_only and form_key == "_form"
                    and _hint_parts(_HINT.get(key, {}).get(label))[0]):
                own_too.append(label)
            print(f"❌ {label} — {where}을 출력에서 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
            if foreign:
                print(f"   ⓘ 「{foreign[:60]}」 은 「이름 = 값」 꼴의 줄이 아니라(이름 앞뒤에 다른 말이 있습니다) 이 이름의 값으로 읽지 않았습니다")
        elif not vals:
            passed = False
            note = _note_for(fail_notes.get(label, ()), [], " ".join(o["tail"] for o in occ))   # 「= nan」 같은 흔한 실수
            failed.append((label, label if words or note else _hint_key(key, "form", label)))
            if _bi.any(o["boxed"] for o in occ):     # 배열로 찍힌 값 — 값이 아니라 출력 꼴의 문제입니다
                print(f"❌ {label} — {where}에서 숫자를 찾지 못했습니다 — 값이 대괄호 [ ] 안의 배열이라 숫자 하나로 읽지 못했습니다")
                print(f"   ⓘ 값이 배열·Series 꼴로 찍혔습니다 — float(값) 이나 .item() 으로 꺼낸 숫자 하나를"
                      f" {want_form} 한 줄로 출력하세요")
            else:
                print(f"❌ {label} — {where}에서 숫자를 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
                if _bi.any(_re.search(r"[(\[][^()\[\]]*\d", o["tail"]) for o in occ):
                    print("   ⓘ 이 줄의 숫자는 괄호 안에만 있습니다 — 괄호 안은 설명으로 보고 값으로 읽지 않습니다."
                          f" 값을 괄호 밖에 두고 {want_form} 꼴로 출력하세요")
            if note:
                print(f"   원인 : {note}")
        else:
            passed = False
            first = vals[0]
            _prev_fail = _TRY.get(key, 0)              # 첫 ❌ 에는 기준 범위를 감춥니다
            short = _short_digits(first[1], first[0], lo, hi)
            if short is not None:
                done = "정수로만" if short == 0 else f"소수점 아래 {short}자리까지만"
                got_txt = (f"{where}에서 읽은 값은 {_val_txt(first)} 인데 기준 범위 밖입니다 — 반올림한 자리가 모자라"
                           f" 범위 밖으로 읽혔습니다({done} 찍힘). 「{made}」 에 적힌 소수점 아래 자릿수대로 출력하세요")
            else:
                got_txt = f"{where}에서 읽은 값은 {_val_txt(first)} 인데 기준 범위 밖입니다"
            ref_txt = spec.get("range_text", {}).get(label)      # 한쪽이 열린 기준(「0 보다 큼」)은 범위 대신 말로 적습니다
            band = (f"기준: {ref_txt}" if ref_txt else f"기준 {_num_txt(lo, 'g')}" if lo == hi
                    else f"기준 범위 {_num_txt(lo, 'g')} ~ {_num_txt(hi, 'g')}")
            print(f"❌ {label} — {got_txt}" + (f" ({band})" if _prev_fail else ""))
            note = "" if short is not None else _note_for(fail_notes.get(label, ()), [v[0] for v in vals],
                                                          " ".join(o["tail"] for o in use))
            if note:                                 # 흔한 실수 하나와 같은 값이면 그 원인을 바로 말합니다
                print(f"   원인 : {note}")
            trap_note = "" if short is not None else _note_for(traps.get(label, ()), [v[0] for v in vals], "")
            if trap_note:                            # 기준에 적어 둔 흔한 실수의 값(traps)이면 그 실수를 짚습니다
                print(f"   힌트 : {trap_note}")
            if short is None and _bi.any(_re.search(r"\[|\barray\(|dtype|\bnp\.\w+\(", o["tail"]) for o in use):
                print("   ⓘ 값이 배열·Series 꼴로 찍혔습니다 — float(값) 이나 .item() 으로 숫자 하나만 출력하세요")
            elif short is None and first[5] == "bare":
                print(f"   ⓘ 이 줄은 이름 바로 뒤에 「=」·「:」 가 없어 이름 뒤의 숫자를 값으로 읽었습니다."
                      f" {want_form} 꼴로 출력하면 「=」 뒤 값을 읽습니다")
            elif short is None and first[5] == "marker" and _bi.any(lo <= v <= hi for v in _nums(first[2])):
                # 「= 182/200 = 91.0%」·「= 512일 가운데 0 일」·「= 2개 원점 평균 0.271」 — 값은 같은 줄 뒤쪽에 있습니다.
                # 계산이 아니라 출력 꼴의 문제일 수 있다는 것을 알려 계산부터 의심하지 않게 합니다. 뒤 숫자가 우연히 범위 안일
                # 수도 있어(「차 = -0.210 (MSE 0.536, MAE 0.326)」 의 0.326) 「값이 그 뒤에 있다면」 으로 적습니다
                print(f"   ⓘ 이 줄은 「=」·「:」 뒤의 첫 숫자({first[1]})를 값으로 읽었습니다 — 같은 줄의 뒤 숫자는 읽지 않습니다."
                      f" 값이 그 뒤에 있다면 계산식·설명에 든 숫자는 빼고 {want_form} 꼴로 값 하나만 출력하세요")
            failed.append((label, _hint_key(key, "digits", label) if short is not None else label))
    order_tag = spec.get("차수_표기")               # 적합만 고치고 출력 문자열을 그대로 두면 ✅ 가 열리던 자리
    if order_tag:
        order_label = "공식 차수"                   # 이 이름이 있는 줄의 「=」·「:」 뒤에서 고친 차수 표기를 찾습니다
        order_occ = _occurrences(all_lines, order_label, names)   # 「공식 차수 AIC」 줄은 건너뜁니다
        want_order = tuple(int(x) for x in _re.findall(r"\d+", order_tag))
        got = [r for r in (_order_of(o["tail"]) for o in order_occ) if r]
        # 이름 바로 뒤 값 줄(「공식 차수 = …」)이 있으면 그 줄만 — 「공식 차수 탐색: …」 같은 줄의 숫자는 차수가 아닙니다
        read = [r[0] for r in ([r for r in got if r[1]] or got)]
        many = list(dict.fromkeys(read))
        order_form = f"「{order_label} = (p,d,q)(P,D,Q,m)」 한 줄로 출력하세요"
        if len(many) == 1 and many[0] == want_order:
            print(f"✅ 차수 표기 — 「{order_label}」 줄이 고친 차수를 적고 있습니다")
        else:
            passed = False
            if len(many) > 1:
                failed.append(("차수 표기", _hint_key(key, "dup", "차수 표기")))
                print(f"❌ 차수 표기 — 「{order_label}」 줄이 {len(read)}개이고 적힌 차수가 서로 다릅니다"
                      f" ({' · '.join(str(o) for o in many)}). 최종 차수 한 줄만 남기세요")
            elif many:
                failed.append(("차수 표기", "차수_표기"))
                print(f"❌ 차수 표기 — 「{order_label}」 줄에 적힌 비계절 차수 {many[0]} 가 고친 차수와 다릅니다")
            else:
                failed.append(("차수 표기", _hint_key(key, "form", "차수 표기")))
                print("❌ 차수 표기 — " + (f"「{order_label}」 줄에서 숫자 3개로 된 차수를 읽지 못했습니다. {order_form}"
                                        if order_occ else f"이 이름이 있는 줄을 출력에서 찾지 못했습니다. {order_form}"))
    flat_lines = [_flat(l) for l in all_lines]
    forbid_all = [str(p) for p in spec.get("forbid", ())]
    forbid_seen = set()
    for phrase in forbid_all:                        # 함정 미션 — 전임자의 결론 줄이 남으면 통과시키지 않습니다
        # 빈칸만 다른 문구(「판단 :」·「판단:」)는 빈칸을 걷으면 같은 문구입니다 — 한 번만 알립니다
        same = _flat(phrase)
        if not same or same in forbid_seen:
            continue
        # 「…넣어도 되지 않습니다」 처럼 서술로 부정해 인용한 줄은 남은 줄이 아닙니다(`_forbid_in`)
        if _bi.any(_forbid_in(row, phrase) for row in flat_lines):
            forbid_seen.add(same)
            passed = False
            if ("남은 줄", "forbid") not in failed:
                failed.append(("남은 줄", "forbid"))
            # 알리는 글자는 출력에 찍힌 꼴 — 같은 문구의 여러 꼴 가운데 출력에 글자 그대로 있는 것, 없으면 먼저 적힌 것
            shown_as = next((p for p in forbid_all if _flat(p) == same and p in txt), phrase)
            print(f"❌ 남은 줄 — 출력에 「{shown_as}」 문구가 남아 있습니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다."
                  " 채점은 matplotlib 으로 그린 그림을 읽습니다)")
        else:
            gaps = None
            for ax in axs:                 # 축이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림의 축 이름을 읽지 못했습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            axis_words = "가로축·세로축 이름이" if plot_spec.get("축라벨") or not plot_spec.get("가로축라벨") else "가로축 이름이"
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            elif shown is False:                    # inline 백엔드인데 이 셀이 그림을 한 장도 화면에 내보내지 않았습니다
                print(f"✅ 그래프 — 그림 객체에 {axis_words} 있습니다")
                print("   ⓘ 그림이 화면에 뜨지 않았습니다 — plt.close() 나 savefig 로만 끝냈다면 plt.show() 로 띄워"
                      " 눈으로도 확인하세요")
            else:                                   # 그래프 종류는 가리지 않고 그림과 축 이름만 봅니다
                print(f"✅ 그래프 — 그림이 있고 {axis_words} 있습니다")
    ctx, notices = None, []
    for check in _BOOK.get("checks", ()):            # 권 설정의 검사(계획표의 열 · 낱말 · 저장한 파일처럼 값·그림 밖의 결과물)
        try:
            if _takes_ctx(check):                    # fn(key, spec, ctx) — 출력 글 · 그림 · 실행 상태를 받는 꼴
                if ctx is None:
                    ctx = {"text": txt, "lines": all_lines, "figs": list(figs), "axes": [a for f in figs for a in _axes(f)],
                           "values": seen, "failed": failed, "attempt": _TRY.get(key, 0), "locked": _locked, "trap": trap,
                           "src": str(_STATE.get("src") or ""), "code": _ai_lines(_STATE.get("src")),
                           "prompt": _prompt_now(prompt), "shown": shown,
                           "read": lambda name, others=(): [o["val"][0] for o in _value_use(
                               _occurrences(all_lines, name, names + list(others), series), val_lines)],
                           "tails": lambda name, others=(): _text_tails(all_lines, name, names + list(others))}
                rows = list(check(key, spec, ctx) or ())
            else:                                    # fn(key, spec) — 옛 꼴
                rows = list(check(key, spec) or ())
        except Exception as e:                       # 검사가 예외로 끝나도 채점 표는 끝까지 찍습니다
            rows = [("채점 검사", False, f"이 미션의 검사 하나를 마치지 못했습니다 ({type(e).__name__}: {_one_line(e, 80)})."
                     " 이 미션의 준비 셀을 실행했는지 보고, 그래도 같으면 맨 위 「[채점] 이 셀을 먼저 실행하세요」 셀을 다시 실행한 뒤"
                     " 이 셀을 다시 실행하세요", "_check")]
        for row in rows:
            name, ok, why, hkey = row[:4]
            more = row[4] if len(row) > 4 else ()
            if ok is None:                           # 알림 — 틀린 것이 아니라 다음 할 일(❌ 로 세지 않고 풀이는 열지 않습니다)
                notices.append(why if not name else f"ⓘ {name} — {why}")
                continue
            if ok:
                print(f"✅ {name} — {why}")
            else:
                passed = False
                if (name, hkey) not in failed:       # 같은 항목이 두 번 ❌ 여도 힌트는 한 번만
                    failed.append((name, hkey))
                print(f"❌ {name} — {why}")
            for line in ((more,) if isinstance(more, str) else more or ()):   # 그 줄에 덧붙이는 줄(ⓘ 참고 · 힌트)
                print(f"   {line}")
    tbl = dict(_HINT.get(key, {}))
    tbl.setdefault("_form", tuple(t.replace("「만들 것」", f"「{made}」") for t in _FORM_HINT))     # 공통 힌트가 가리키는
    tbl.setdefault("_digits", tuple(t.replace("「만들 것」", f"「{made}」") for t in _DIGITS_HINT))  # 소제목을 이 미션의 것으로
    tbl.setdefault("_dup", _DUP_HINT if trap else _DUP_HINT_PLAIN)
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    groups = {}                                    # 같은 힌트(출력 꼴 등)는 한 번만 — 그 힌트가 걸린 항목 이름을 모아 찍습니다
    for name, hkey in failed:
        groups.setdefault(hkey, []).append(name)
    for hkey, members in groups.items():
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[_bi.min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({'출력 꼴' if hkey == '_form' else ' · '.join(members)}) : {hint}")
        if hkey == "_form":                        # 함정 미션 — 출력 꼴 힌트 뒤에 그 항목의 권 힌트 첫 단
            for name in members:
                if name in own_too:
                    print(f"   힌트({name}) : {_hint_parts(tbl.get(name))[0][0]}")
    if failed:
        print(_ASK_HEAD.get(form, ""))
        tmpl = _ASK_FORM.get(form) or ""
        for hkey, members in groups.items():
            # 함정 미션에서 이름 줄이 없던 항목은 권이 그 항목에 적어 둔 물을 것을 씁니다(없는 줄의 출력 꼴을 묻지 않게)
            own_ask = [m for m in members if hkey == "_form" and m in own_too and _hint_parts(tbl.get(m))[1]]
            rest = [m for m in members if m not in own_ask]
            if rest:
                question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                    or "이 항목이 왜 ❌ 로 남는지"
                joined = "」·「".join(rest) if "「{name}」" in tmpl else " · ".join(rest)
                print(_ask_line(form, joined, prompt, question))
            for name in own_ask:
                print(_ask_line(form, name, prompt, _hint_parts(tbl.get(name))[1]))
    elif notices:                                  # 틀린 것은 없고 다음 할 일이 남았습니다(두 번에 나눠 푸는 미션의 1회차)
        for line in notices:
            print(line)
    elif _locked:                                  # 값·그림은 기준에 맞지만 PROMPT 칸이 아직 비었습니다
        print('ⓘ 값과 그림은 기준에 맞습니다 — 이 셀의 PROMPT = """ """ 따옴표 안에 고칠 지시를 내 말로 쓰고'
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 비교해 보세요")
        _reveal_show(_b64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked and not notices


# ───────────────────────── PROMPT 셀 훅 ─────────────────────────
class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """머리 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다.

    머리 행은 비어 있지 않은 앞 3행 가운데 처음 나오는 `# 미션`·`# 과제` 행입니다 — 맨 앞에 빈 줄이나
    import 한 줄이 끼어도 채점 표가 찍히게 합니다. 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다.
    """
    lines = [l for l in (src or "").split("\n") if l.strip()]
    if lines and lines[0].lstrip().startswith("#@title"):
        return None
    for head_line in lines[:3]:
        m = _re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
        if not m:
            continue
        if "프롬프트" not in head_line:
            return None
        return "m" + m.group(2) if m.group(2) else "과제"
    return None


def _lost_tag(src):
    """머리줄이 사라진 「PROMPT 셀」 — 미션 이름이 적힌 코드면 그 미션, PROMPT·표시선만 있으면 "?", 아니면 None.

    AI 답 전체를 셀에 덮어쓰면 첫 행의 머리줄이 사라져 채점이 아무것도 찍지 않습니다. 조용히 지나가지 않고
    「머리줄이 없어 채점하지 못했다」 고 채점 표로 알립니다(`_lost_notice`).
    """
    first = next((l for l in (src or "").split("\n") if l.strip()), "")
    if first.lstrip().startswith("#@title") or _re.match(r"\s*#\s*(미션\s*\d+|과제)\s*·\s*준비", first):
        return None                                       # 채점 셀 · 준비 셀
    flat = _flat(src)
    hits = {k: _bi.sum(_flat(l) in flat for l in (_device_spec(s) or {}).get("values", {})) for k, s in _ANS.items()}
    best = _bi.max(hits.values()) if hits else 0
    keys = [k for k, n in hits.items() if n == best and n > 0]
    if len(keys) == 1:
        return keys[0]
    if "여기부터 AI 코드" in (src or "") or _re.search(r"^[ \t]*PROMPT[ \t]*=", src or "", _re.M):
        return "?"
    return None


def _lost_notice(key):
    """머리줄이 없어 채점하지 못한 셀 — 채점 표 꼴로 알리고 닫습니다"""
    if key == "?":
        head_label, header = "채점하지 못한 셀", "「# 미션 N — 프롬프트」"
    else:
        head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
        trap = _is_trap(key)                               # 빈 PROMPT 에도 잠그지 않는 미션이 함정 미션입니다
        header = f"「# {head_label} — 프롬프트{' (함정)' if trap else ''}」"
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과" if key != "?" else f"◼ {head_label}")
    print(f"❌ 머리줄 — 셀 첫 행에 {header} 행이 없어 이 셀을 채점하지 못했습니다")
    print("   힌트 : AI 답을 붙여 넣으며 셀 전체를 덮어쓰면 첫 행의 머리줄, PROMPT 입력란, 「# --- 여기부터 AI 코드 ---」"
          " 표시선이 함께 사라집니다. 셀을 되돌리거나(실행 취소) 그 세 가지를 셀 맨 위에 다시 적고,"
          " AI 코드는 표시선 아래에만 붙여 넣은 뒤 다시 실행하세요")
    print("─" * 54)


def _shown_text(data):
    """display 출력 묶음에서 화면에 보이는 글자 — text/plain 이 자리표시면 markdown · html 의 글자.

    자리표시는 `<IPython.core.display.Markdown object>` · `<pandas.io.formats.style.Styler at 0x…>` 같은 객체 표기입니다
    (`display(표.style…)` 의 text/plain 에는 「object」 낱말이 없습니다). HTML 표는 칸마다 줄이 바뀌어 적혀 있으므로
    한 행을 한 줄로 모으고, 화면에 보이지 않는 `<style>`·`<script>` 는 뺍니다.
    """
    import html as _html
    plain = str(data.get("text/plain", ""))
    if plain and not _re.fullmatch(r"\s*<[\w.]+(?: object)?(?: at 0x[0-9a-fA-F]+)?>\s*", plain):
        return plain
    if "text/markdown" in data:                  # **굵게** · `코드` 표시 기호는 화면에 안 보입니다
        return _re.sub(r"\*\*|__|`", "", str(data["text/markdown"]))
    if "text/html" in data:
        page = _re.sub(r"(?is)<(style|script)\b.*?</\1\s*>", "", str(data["text/html"]))
        page = _re.sub(r"(?is)<table\b.*?</table\s*>", lambda m: _re.sub(r"\s+", " ", m.group(0)), page)
        page = _re.sub(r"(?i)</t[dh]\s*>", "\t", page)
        page = _re.sub(r"(?i)<br\s*/?>|</(?:p|div|tr|li|h\d)\s*>", "\n", page)
        return _html.unescape(_re.sub(r"<[^>]+>", "", page))
    return plain


def _pub_hook():
    """display() 로 띄운 표·값도 채점이 읽게 — Colab·LMS(ipykernel)는 display 출력을 stdout 이 아닌 채널로 보냅니다.

    「PROMPT 셀」이 도는 동안만 display_pub.publish 를 감싸, 화면에 보이는 글자(text/plain · Markdown · HTML)를
    채점 버퍼에 적고 화면에 뜬 그림(image/*) 수를 셉니다. 터미널처럼 표시가 stdout 에도 쓰이는 자리에서 두 번
    담기지 않게, 원래 표시를 부르는 동안은 화면으로만 내보냅니다. `_post` 가 원래 publish 로 되돌립니다.
    """
    _pub_unhook()
    try:
        dp = _ip.display_pub
    except Exception:
        return
    had = "publish" in vars(dp)
    orig = dp.publish
    state, sysm = _STATE, _sys                 # 학생 코드가 도는 동안 불립니다 — 이름을 여기서 묶어 둡니다

    def note(d):
        if _bi.any(str(m).startswith("image/") for m in d):
            state["imgs"] = state.get("imgs", 0) + 1
        elif d and state.get("buf") is not None:
            state["buf"].write("\n" + _shown_text(d) + "\n")

    own = _get("_OWN")
    if isinstance(own, dict):                  # 학생 셀이 채점 코드와 같은 이름을 만든 뒤 display 를 불러도 읽게
        note = _guard(note, own, globals())

    def publish(data=None, *a, **k):
        try:
            note(data if isinstance(data, dict) else {})
        except Exception:
            pass
        cur = sysm.stdout
        sysm.stdout = state.get("out") or cur
        try:
            return orig(data, *a, **k)
        finally:
            sysm.stdout = cur

    dp.publish = publish
    _STATE["pub"] = (dp, had, orig, publish)


def _pub_unhook():
    """`_pub_hook` 이 감싼 publish 를 되돌립니다"""
    saved, _STATE["pub"] = _STATE.get("pub"), None
    if not saved:
        return
    dp, had, orig, mine = saved
    try:
        if vars(dp).get("publish") is mine:
            if had:
                dp.publish = orig
            else:
                del dp.publish
    except Exception:
        pass


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        src = str(getattr(info, "raw_cell", "") or "")
        _STATE.update(tag=None, lost=None, src=src, prompt=None, imgs=0)
        tag = _tag(src)
        hook = _BOOK.get("pre")
        if hook is not None:
            try:
                hook(tag if tag in _ANS else None)     # 권 설정 — 셀 실행 전 상태를 적어 둡니다
            except Exception:
                pass
        if tag is None:
            _STATE["lost"] = _lost_tag(src)
            return
        if tag not in _ANS:
            return
        _STATE["tag"] = tag
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["prompt"] = _prompt_in_cell(src)    # 앞 미션의 PROMPT 가 남아 있어도 이 셀 것만
        _STATE["buf"], _STATE["out"] = _io.StringIO(), _sys.stdout
        _sys.stdout = _Tee(_sys.stdout, _STATE["buf"])
        _pub_hook()
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다 — 무슨 일이 있어도 채점 표를 닫습니다"""
    tag, buf, out, prompt = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out"), _STATE.get("prompt")
    _STATE.update(tag=None, buf=None, out=None, prompt=None)
    _pub_unhook()
    if out is not None:
        _sys.stdout = out
    lost, _STATE["lost"] = _STATE.get("lost"), None
    if lost:
        try:
            _lost_notice(lost)
        except Exception:
            print("─" * 54)
        return
    if tag is None or tag not in _ANS:
        return
    try:
        seen = {id(f) for f in _FIGS}
        figs_all = list(_FIGS) + [f for f in _open_figs()
                                  if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
        txt = buf.getvalue() if buf is not None else ""
        last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
        if last_val is not None:
            try:
                txt = txt + "\n" + str(last_val)
            except Exception:
                pass
        # 문법 오류(SyntaxError·IndentationError)는 실행 전에 나서 error_before_exec 에만 남습니다 — 이것도 실행 오류입니다
        before = getattr(result, "error_before_exec", None)
        error = getattr(result, "error_in_exec", None) or before
        if before is None and prompt:
            prompt = None                                # 셀이 돌았으면 PROMPT 는 방금 실행된 값을 읽습니다
        elif before is not None and prompt is None:
            prompt = ""                                  # 돌지 않은 셀의 PROMPT 를 따옴표 꼴로 못 읽었습니다
        try:
            import matplotlib as _mpl
            shown = (_STATE.get("imgs", 0) > 0) if "inline" in _mpl.get_backend().lower() else None
        except Exception:
            shown = None
        judge(tag, txt, figs_all, error=error, prompt=prompt, shown=shown)
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")
        print("   맨 위 「[채점] 이 셀을 먼저 실행하세요」 셀을 다시 실행한 뒤 이 셀을 다시 실행하세요")
        print("─" * 54)


def _own_names(before):
    """이 셀이 만든 이름 → 값 — 밑줄로 시작하는 이름과 judge. IPython 이 쓰는 이름(_ · _i · _oh · _3 …)은 뺍니다.

    셀을 실행하기 전과 값이 달라졌거나 새로 생긴 이름입니다. 이 셀을 다시 실행하면 모듈 별칭처럼 값이 그대로인 이름은
    달라진 것으로 보이지 않으므로, 앞서 실행 때 적어 둔 이름도 함께 넣습니다.
    """
    prev, g = _get("_OWN"), globals()
    prev = prev if isinstance(prev, dict) else {}
    return {n: v for n, v in list(g.items())
            if (n[:1] == "_" or n == "judge") and n[:2] != "__" and n != "_G0"
            and not _re.fullmatch(r"_(?:i{0,3}|ih|oh|dh|i?\d+|exit_code)", n)
            and (n not in before or before[n] is not v or n in prev)}


def _guard(fn, own, g):
    """훅(`_pre`·`_post`)이 도는 동안만 채점 코드의 이름을 제자리에 둡니다.

    채점 코드의 도우미(`_axes`·`_flat`·`_nums`·`_get` …)는 학생 셀과 같은 이름 공간에 있습니다. 학생 코드가
    `_fig, _axes = plt.subplots(1, 2)` 처럼 같은 이름을 만들면 채점이 그 값을 함수로 부르다 멎고, 그 뒤 모든 미션이
    「채점을 마치지 못했습니다」 로 끝납니다. 그래서 훅에 들어갈 때 채점 코드의 이름을 되돌려 놓고, 나올 때 학생이 만든
    값을 돌려놓습니다 — 채점도, 그 이름을 쓰는 학생의 다음 셀도 그대로 돕니다.
    """
    def run(*a, **k):
        mine = {n: g[n] for n in own if n in g and g[n] is not own[n]}     # 학생 셀이 같은 이름에 넣어 둔 값
        g.update(own)
        try:
            return fn(*a, **k)
        finally:
            own.update({n: g[n] for n in own if n in g and g[n] is not own[n]})   # 채점 코드가 바꿔 단 값은 그대로
            g.update(mine)

    run.__name__ = fn.__name__
    return run


try:
    _OWN = _own_names(_G0)
    del _G0
    _ip = get_ipython()
    _OWN.update(_ip=_ip, _OWN=_OWN)
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _guard(_fn, _OWN, globals()))
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 서울 지하철 100개 역 준비

자료는 **서울 지하철 일별 승하차 기록**(서울열린데이터광장 OA-12914)입니다. 기간은 2015-01-01～2026-06-30 의 4,199일입니다. 기록이 끊기지 않은 역 가운데 2024년까지 평균 승차가 많은 100개 역을 씁니다.

예측 원점은 수요일 5개입니다. 원점 전날까지 512일(컨텍스트)을 Chronos-2 에 입력하고, 원점 당일부터 7일을 채점합니다. 원점마다 채점 7일이 한 주이므로, 원점 2개는 「두 주」, 원점 3개는 「세 주」라 부릅니다.

| 채점한 주 | 예측 원점 | 채점 7일 |
|---|---|---|
| **공휴일 없는 두 주** | 2025-12-10 · 2026-06-24 | 평일 공휴일이 없다 |
| **공휴일 있는 세 주** | 2025-08-13 · 2025-10-01 · 2026-02-11 | 평일 공휴일이 있다 |

「오늘의 준비」 셀은 두 모델로 원점 5개를 예측해 분위수 아홉 열을 얻습니다. 이 분위수는 파일로도 남기고, 「미션 1 준비 셀」과 「미션 2 준비 셀」이 이 파일을 읽습니다. 출력은 아래 넷입니다.

- 표 1 · 채점 7일의 평일 공휴일 (원점 5개)
- 표 2 · 강남역 원점 2026-06-24 Chronos-2 + 공휴일 분위수 아홉 열과 실제 (명)
- 그림 1 · 강남역 원점 2026-02-11 Chronos-2 + 공휴일 80% 구간과 실제
- 표 3 · 제로샷 80% 구간의 포함률, 공휴일이 있는 주와 없는 주 (%)

In [ ]:
# 오늘의 준비 — 읽고 실행만 합니다
#  서울 지하철 100개 역에 Chronos-2 두 모델로 원점 5개를 예측해 표 1 · 표 2 · 그림 1 · 표 3 을 만듭니다 (T4 약 20초, CPU 약 100초)
#  Chronos-2 를 불러오다 내려받기 오류(Hugging Face)가 나면 맨 위 설치 셀부터 다시 실행합니다
import os, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

RAW_URL = "https://github.com/studio-js/seoul-subway-ridership/raw/main/data/seoul_subway_daily_2015_2026.parquet"
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)

# 지금까지와 같은 손질 — 역 이름의 괄호 별칭을 떼고 9호선 이름을 맞춘 뒤 (날짜, 호선, 역) 으로 합칩니다
_raw = pd.read_parquet(CACHE)
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) &            # 기록이 끊기지 않은 역만 남긴 뒤
              (_span["max"] >= pd.Timestamp("2026-06-30")) & (_span["count"] >= 4190)]
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")]             # 2024년까지 평균 승차 상위 100개 역
        .groupby(["line", "station"])["board"].mean()
        .loc[_full.index].sort_values(ascending=False).index[:100])
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]
MAT = _wide.to_numpy(dtype=float)             # 100개 역 x 4,199일 승차 (명)
DATES = pd.DatetimeIndex(_wide.columns)       # 행렬 열의 날짜
STATIONS = [f"{_line} {_name}" for _line, _name in _wide.index]   # 행렬 행의 역 이름

H, CTX = 7, 512                               # 채점 7일 · Chronos-2 에 주는 과거 512일
QS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # 분위수 아홉 열 (80% 구간의 두 끝 = 0.1 열 · 0.9 열)
ORIGINS = ["2025-08-13", "2025-10-01", "2025-12-10", "2026-02-11", "2026-06-24"]   # 예측 원점 5개 (수요일)
PLAIN_ORIGINS = ["2025-12-10", "2026-06-24"]              # 공휴일 없는 두 주: 채점 7일에 평일 공휴일이 없다
HOL_ORIGINS = ["2025-08-13", "2025-10-01", "2026-02-11"]  # 공휴일 있는 세 주: 채점 7일에 평일 공휴일이 있다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 맨 위 설치 셀과 같은 규칙 — 이 셀만 실행해도 정해집니다


def hol_flags(dates):
    """날짜마다 한국 공휴일이면 1, 아니면 0 (holidays 0.103 KR, 주말 공휴일 포함) — 달력이 정하는 값입니다"""
    import holidays
    di = pd.DatetimeIndex(dates)
    kr = holidays.KR(years=sorted(set(di.year)))
    return np.array([1.0 if d in kr else 0.0 for d in di])


HOL = hol_flags(DATES)                        # DATES 의 날마다 공휴일 1 / 0


def origin_ctx(origin):
    """원점 하나의 자료 — 원점의 열 번호(o), 원점 전날까지(hist), 채점 7일 실제(test, 명), 역별 MASE 분모(qden), 채점 7일 날짜(test_dates)"""
    o = int(DATES.get_loc(pd.Timestamp(origin)))
    hist = MAT[:, :o]
    qden = np.nanmean(np.abs(hist[:, 7:] - hist[:, :-7]), axis=1)
    return dict(o=o, hist=hist, test=MAT[:, o:o + H], qden=qden, test_dates=DATES[o:o + H])


def chronos_frames(origin, holiday=True):
    """Chronos-2 의 predict_df 에 전달할 두 표 (df, future_df)

    df        : 100개 역 x 원점 전날까지 512일 — item_id(역 번호 0 ~ 99) · timestamp · target(승차, 명)
                · holiday(공휴일 1 / 0). holiday=False 면 holiday 열이 없습니다
    future_df : 100개 역 x 채점 7일 — item_id · timestamp · holiday(달력이 정한 공휴일). holiday=False 면 None
    """
    ctx = origin_ctx(origin)
    o, S = ctx["o"], MAT.shape[0]
    df = pd.DataFrame({"item_id": np.repeat(np.arange(S), CTX),
                       "timestamp": np.tile(DATES[o - CTX:o].values, S),
                       "target": MAT[:, o - CTX:o].ravel()})
    future_df = None
    if holiday:
        df["holiday"] = np.tile(HOL[o - CTX:o], S)
        future_df = pd.DataFrame({"item_id": np.repeat(np.arange(S), H),
                                  "timestamp": np.tile(ctx["test_dates"].values, S),
                                  "holiday": np.tile(HOL[o:o + H], S)})
    assert df["timestamp"].max() < pd.Timestamp(origin)   # 과거 표에는 원점 전날까지만
    return df, future_df


# Chronos-2 를 DEVICE 에 올립니다. 끊기면 세 번까지 다시 시도합니다
from chronos import BaseChronosPipeline
PIPE = None
for _try in range(1, 4):
    try:
        PIPE = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE, torch_dtype=torch.float32)
        break
    except Exception as _e:
        print(f"Chronos-2 불러오기 {_try}번째 실패 : {type(_e).__name__}")
        time.sleep(5)
if PIPE is None:
    raise RuntimeError("Chronos-2 를 불러오지 못했습니다. 네트워크를 확인하고 이 셀을 다시 실행합니다")

# 원점 5개 x 두 모델(제로샷 · Chronos-2 + 공휴일)의 분위수 아홉 열을 얻어 파일로 남깁니다. 미션 준비 셀이 이 파일을 읽습니다
Y_TEST = {o: origin_ctx(o)["test"] for o in ORIGINS}      # 원점 → 100개 역 x 7일 실제 승차 (명)
Q_ZS, Q_HOL = {}, {}                                       # 원점 → 100개 역 x 7일 x 분위수 아홉 열 (명)
for _holiday, _path, _store in ((False, "c2_zs_q.parquet", Q_ZS), (True, "c2_hol_q.parquet", Q_HOL)):
    _frames = []
    for o in ORIGINS:
        _df, _future = chronos_frames(o, holiday=_holiday)
        _fc = PIPE.predict_df(_df, future_df=_future, prediction_length=H, quantile_levels=QS)
        _fc = _fc.sort_values(["item_id", "timestamp"])
        _store[o] = np.stack([_fc[str(t)].to_numpy().reshape(-1, H) for t in QS], axis=-1)
        _frames.append(_fc.assign(origin=o)[["origin", "item_id", "timestamp"] + [str(t) for t in QS]])
    pd.concat(_frames, ignore_index=True).to_parquet(_path)

# 표 1: 원점마다 채점 7일에 있는 평일(월~금) 공휴일
DAY_KO = "월화수목금토일"
WDH_TABLE = {}
for o in ORIGINS:
    _c = origin_ctx(o)
    WDH_TABLE[o] = [f"{d:%m-%d}({DAY_KO[d.dayofweek]})" for d, h in zip(_c["test_dates"], HOL[_c["o"]:_c["o"] + H])
                    if h == 1 and d.dayofweek < 5]
print("표 1 · 채점 7일의 평일 공휴일 (원점 5개)")
for o in ORIGINS:
    _group = "공휴일 없는 주" if o in PLAIN_ORIGINS else "공휴일 있는 주"
    print(f"  {o} · {_group} : {' · '.join(WDH_TABLE[o]) or '없음'}")
print("  주말 공휴일 2025-10-05(일, 추석 연휴)는 평일 공휴일로 세지 않습니다")

# 표 2: 강남역, 원점 2026-06-24 의 첫 사흘. Chronos-2 + 공휴일 분위수 아홉 열과 실제 승차
GN = STATIONS.index("2호선 강남")
_d = origin_ctx("2026-06-24")["test_dates"][:3]
GN_Q = pd.DataFrame(Q_HOL["2026-06-24"][GN, :3], index=[f"{d:%Y-%m-%d}" for d in _d], columns=[str(t) for t in QS])
GN_Q["실제"] = Y_TEST["2026-06-24"][GN, :3]
print("\n표 2 · 강남역 원점 2026-06-24 Chronos-2 + 공휴일 분위수 아홉 열과 실제 (명)")
print(GN_Q.round(0).astype(int).to_string())

# 그림 1: 강남역, 원점 2026-02-11 (설 연휴가 있는 주). 0.1 열과 0.9 열 사이를 칠하고 실제 승차를 점으로
_d = origin_ctx("2026-02-11")["test_dates"]
GN_SEOL = pd.DataFrame(Q_HOL["2026-02-11"][GN][:, [0, 4, 8]], columns=["0.1", "0.5", "0.9"],
                       index=[f"{d:%m-%d}({DAY_KO[d.dayofweek]})" for d in _d])
GN_SEOL["실제"] = Y_TEST["2026-02-11"][GN]
fig, ax = plt.subplots(figsize=(10, 3.6))
_x = np.arange(H)
ax.fill_between(_x, GN_SEOL["0.1"], GN_SEOL["0.9"], color="#B3B3B3", alpha=0.6, label="80% 구간 (0.1 열과 0.9 열 사이)")
ax.plot(_x, GN_SEOL["0.5"], color="#051C2C", lw=2, label="0.5 열 (점 예측)")
ax.scatter(_x, GN_SEOL["실제"], color="#00A9F4", s=40, zorder=3, label="실제 승차")
ax.set_xticks(_x, GN_SEOL.index)
ax.set_ylabel("승차 (명)")
ax.set_title("그림 1 · 강남역 원점 2026-02-11 Chronos-2 + 공휴일 80% 구간과 실제")
ax.legend(loc="lower left", frameon=False)
plt.show()

# 표 3: 제로샷 80% 구간의 포함률을 원점마다 구해 공휴일 없는 두 주와 공휴일 있는 세 주로 평균
_cov = {o: ((Y_TEST[o] >= Q_ZS[o][..., 0]) & (Y_TEST[o] <= Q_ZS[o][..., -1])).mean() * 100 for o in ORIGINS}
COV_ZS = {"공휴일 없는 두 주": np.mean([_cov[o] for o in PLAIN_ORIGINS]),
          "공휴일 있는 세 주": np.mean([_cov[o] for o in HOL_ORIGINS]),
          "원점 5개": np.mean([_cov[o] for o in ORIGINS])}
print("표 3 · 제로샷 80% 구간의 포함률, 공휴일이 있는 주와 없는 주 (%)")
for _k, _v in COV_ZS.items():
    print(f"  {_k:<9} {_v:5.1f}")

## 분위수와 포함률 확인

> **[문제] 0.9 열에 맞춘 인력이 모자랄 확률**
>
> 예를 들어 다음 주 수요일 어느 역의 0.1 열이 8만 명, 0.9 열이 10만 명일 때 10만 명에 맞춰 인력을 두면 인력이 모자랄 확률은 얼마일까요?
>
> 1. 약 10%
> 2. 약 20%
> 3. 약 90%

> **[문제] 인력 계획에 전달할 80% 구간**
>
> 예를 들어 같은 주에 구간 A 는 700점에서 구한 포함률이 90% 이고 평균 길이가 4,000명입니다. 구간 B 는 포함률이 80% 이고 평균 길이가 2,500명입니다. 「80% 구간」이라는 이름으로 인력 계획에 전달할 구간은 어느 쪽일까요?
>
> 1. A. 더 많이 담았으니 80% 구간으로 더 믿을 만하다
> 2. B. 명목 80% 에 맞게 담으면서 구간이 짧다
> 3. 어느 쪽이든 같다. 포함률 10%p 차이는 흔들림이다

## 서울 지하철 100개 역으로 판단하기

아래 세 문항은 「오늘의 준비」 셀 출력을 읽고 답합니다. 표 2 와 그림 1 은 강남역 한 역의 두 주입니다. 표 3 은 100개 역 700점의 포함률을 원점마다 구해 공휴일 없는 두 주와 공휴일 있는 세 주로 평균한 제로샷의 값입니다.

> **[문제] 강남역 6월 25일 80% 구간 길이**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 2 · 강남역 원점 2026-06-24 Chronos-2 + 공휴일 분위수 아홉 열과 실제 (명)」을 봅니다. 2026-06-25 행의 80% 구간 길이는 약 몇 명일까요?
>
> 1. 약 3,600명
> 2. 약 93,000명
> 3. 약 5,700명

> **[문제] 설 연휴가 있는 주 구간 밖으로 나간 날**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「그림 1 · 강남역 원점 2026-02-11 Chronos-2 + 공휴일 80% 구간과 실제」를 봅니다. 구간 밖으로 나간 날의 실제 승차는 구간의 어느 쪽에 있나요?
>
> 1. 모두 아래 끝보다 적었다
> 2. 모두 위 끝보다 많았다
> 3. 아래와 위 양쪽으로 나갔다

> **[문제] 제로샷 구간을 평일 공휴일이 있는 주 인력에**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 3 · 제로샷 80% 구간의 포함률, 공휴일이 있는 주와 없는 주 (%)」을 봅니다. 평일 공휴일이 있는 주의 인력 계획에 제로샷 80% 구간을 넘겨도 될까요?
>
> 1. 넘긴다. 공휴일 없는 두 주에서 80% 를 넘게 담았다
> 2. 넘기지 않는다. 공휴일 있는 세 주에서 절반도 담지 못했다
> 3. 넘긴다. 원점 5개를 합치면 명목 80% 에 가깝다

## 미션 1 — Chronos-2 + 공휴일의 80% 포함률

표 3 에서 제로샷 구간은 공휴일 있는 세 주 2,100점에서 포함률이 41.6% 였습니다. 공휴일 열을 입력한 Chronos-2 + 공휴일 구간의 포함률을 같은 방법으로 구해, 두 주와 세 주의 값을 명목 80% 와 비교합니다.

#### 할 일 — 미션 1 (25분)

1. 아래 「미션 1 준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 시킬 일을 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 내 말로 적습니다. 예시 프롬프트도 따옴표 안에 붙여 넣어야 채점됩니다.
3. 셀을 선택한 채로 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. LMS 밖이면 받은 코드를 AI 코드 표시선(`# --- 여기부터 AI 코드 ---`) 아래에 둡니다.
4. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: Chronos-2 + 공휴일의 80% 구간은 공휴일 없는 두 주와 공휴일 있는 세 주에서 포함률이 각각 몇 % 일까요?

**쓸 수 있는 것**: 「미션 1 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `Q_HOL` | 원점 → Chronos-2 + 공휴일 분위수 배열(100개 역 × 7일 × 9열, 명). `[..., 0]` 이 0.1 열, `[..., -1]` 이 0.9 열 |
| `Y_TEST` | 원점 → 채점 7일 실제 승차(100개 역 × 7일, 명) |
| `ORIGINS` · `PLAIN_ORIGINS` · `HOL_ORIGINS` | 원점 5개 · 공휴일 없는 두 주의 원점 2개 · 공휴일 있는 세 주의 원점 3개(날짜 문자열 목록) |
| `QS` | 분위수 아홉 개 `[0.1, 0.2, …, 0.9]` |
| `COL` · `FIG_W` | 그래프 색 사전과 그림 가로 크기(설치 셀) |

#### 만들 것

1·2번의 굵은 이름은 채점이 출력에서 찾으니 글자 그대로 쓰고, 3·4번의 굵은 말은 만들 것의 이름이라 출력에 쓰지 않아도 됩니다. 모두 「PROMPT 셀」에서 출력합니다.

1. **공휴일 없는 두 주 포함률**: 단위 % · 소수 첫째 자리. 원점마다 700점 가운데 실제가 0.1 열 이상 0.9 열 이하인 점의 비율을 구해 `PLAIN_ORIGINS` 두 원점으로 평균합니다
2. **공휴일 있는 세 주 포함률**: 단위 % · 소수 첫째 자리. 같은 방법으로 `HOL_ORIGINS` 세 원점을 평균합니다
3. **원점별 대조 행**: 원점마다 구간 안·아래·위의 점 수를 한 행씩 찍습니다. 세 수를 더해 700 이 되는지 대조하는 행이고, 채점하지 않습니다
4. **그래프 한 장**: 종류는 자유이고 가로축 「예측 원점」, 세로축 「포함률 (%)」 이름을 답니다. 권장은 원점 5개의 포함률 막대에 두 주와 세 주를 다른 색으로, 80% 높이에 가로 점선입니다

**통과 기준**: 1·2번 값이 「이름 = 값 %」 한 행씩 있고 기준 범위 안이며, 그래프에 가로축·세로축 이름이 있으면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름**: 위 표의 이름은 다시 만들지 말라고 적습니다.
- **구간의 두 끝**: 아래 끝은 0.1 열 `Q_HOL[원점][..., 0]`, 위 끝은 0.9 열 `Q_HOL[원점][..., -1]` 입니다. 끝과 같은 값도 안으로 세도록 `>=`·`<=` 두 조건을 `&` 로 묶으라고 적습니다.
- **평균하는 순서**: 원점 5개의 포함률을 먼저 구한 뒤, 두 주와 세 주를 따로 평균하라고 적습니다.
- **출력 형식**: 1·2번의 이름·단위·자릿수를 그대로 옮겨 적습니다.

<details class="lms-extra"><summary>자세히 — 예시 프롬프트</summary>

Q_HOL, Y_TEST, ORIGINS, PLAIN_ORIGINS, HOL_ORIGINS 와 설치 셀의 FIG_W, COL 은 이미 있으니 다시 만들지 마. ORIGINS 원점 5개마다 Y_TEST 가 Q_HOL 의 [..., 0] 이상이고 [..., -1] 이하인 점을 세어, 구간 안·아래·위 점 수와 포함률(%)을 원점마다 한 행씩 찍어 줘. 원점별 포함률을 PLAIN_ORIGINS 와 HOL_ORIGINS 로 나눠 평균하고 "공휴일 없는 두 주 포함률 = 값 %" 와 "공휴일 있는 세 주 포함률 = 값 %" 를 소수 첫째 자리로 한 행씩 출력해 줘. 원점 5개의 포함률 막대를 공휴일이 있는 주와 없는 주가 다른 색이 되게 그리고, 80% 에 가로 점선과 가로축 "예측 원점", 세로축 "포함률 (%)" 을 달아 줘.

</details>

<details class="lms-extra"><summary>배경 — 구간 안의 점과 밖의 점</summary>

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_quantile_regression_002.png" alt="분위수 선 세 개와 점을 그린 산점도. 두 바깥 선 사이의 점은 회색 원, 밖의 점은 더하기 표시" width="480" height="360"/>
<p align="center"><em>▲ 분위수 두 선 사이에 있는 점과 벗어난 점. 가로축은 입력 x, 세로축은 예측하려는 값 y 다. 파란 선이 0.05 분위수, 초록 선이 0.95 분위수이고 두 선 사이(90% 구간)에 있는 점은 회색 원, 벗어난 점은 + 로 찍었다. 포함률은 회색 원의 수를 전체 점 수로 나눈 값이다 (출처: scikit-learn 공식 문서, Quantile regression 예제)</em></p>

</details>

#### 예상 적기

> **[예측] 공휴일 있는 세 주 포함률은 공휴일 없는 두 주보다 높을까요, 낮을까요?**
>
> 1. 높다. 공휴일 열을 입력해 평일 공휴일이 있는 주의 포함률이 더 높다
> 2. 비슷하다. 공휴일이 있는 주와 없는 주 모두 포함률이 명목 80% 에 가깝다
> 3. 낮다. 평일 공휴일이 있는 주에는 여전히 벗어나는 점이 많다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>예상 확인 — 미션 1 결과를 본 뒤 펼칩니다</summary>

답은 셋째 보기입니다. 포함률은 공휴일 없는 두 주 79.5%, 공휴일 있는 세 주 66.4% 이고, 공휴일 있는 세 주는 명목 80% 보다 80 − 66.4 = 13.6%p 모자랍니다. 원점별로는 2025-08-13 78.9%, 2025-10-01 61.4%, 2026-02-11 59.0% 라, 평일 공휴일이 하루인 원점 2025-08-13(공휴일 있는 주)만 명목 80% 에 가깝습니다.

- 「높다」: 공휴일 열을 입력하자 제로샷의 41.6% 가 66.4% 로 높아졌지만, 공휴일 없는 두 주의 79.5% 에는 못 미칩니다.
- 「비슷하다」: 포함률이 명목 80% 에 가까운 것은 공휴일 없는 두 주뿐입니다.

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
#  「오늘의 준비」 셀이 남긴 분위수 파일을 읽고, 파일이 없으면 Chronos-2 로 다시 예측합니다 (T4 약 10초)
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

RAW_URL = "https://github.com/studio-js/seoul-subway-ridership/raw/main/data/seoul_subway_daily_2015_2026.parquet"
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)

# 지금까지와 같은 손질 — 역 이름의 괄호 별칭을 떼고 9호선 이름을 맞춘 뒤 (날짜, 호선, 역) 으로 합칩니다
_raw = pd.read_parquet(CACHE)
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) &            # 기록이 끊기지 않은 역만 남긴 뒤
              (_span["max"] >= pd.Timestamp("2026-06-30")) & (_span["count"] >= 4190)]
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")]             # 2024년까지 평균 승차 상위 100개 역
        .groupby(["line", "station"])["board"].mean()
        .loc[_full.index].sort_values(ascending=False).index[:100])
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]
MAT = _wide.to_numpy(dtype=float)             # 100개 역 x 4,199일 승차 (명)
DATES = pd.DatetimeIndex(_wide.columns)       # 행렬 열의 날짜
STATIONS = [f"{_line} {_name}" for _line, _name in _wide.index]   # 행렬 행의 역 이름

H, CTX = 7, 512                               # 채점 7일 · Chronos-2 에 주는 과거 512일
QS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # 분위수 아홉 열 (80% 구간의 두 끝 = 0.1 열 · 0.9 열)
ORIGINS = ["2025-08-13", "2025-10-01", "2025-12-10", "2026-02-11", "2026-06-24"]   # 예측 원점 5개 (수요일)
PLAIN_ORIGINS = ["2025-12-10", "2026-06-24"]              # 공휴일 없는 두 주: 채점 7일에 평일 공휴일이 없다
HOL_ORIGINS = ["2025-08-13", "2025-10-01", "2026-02-11"]  # 공휴일 있는 세 주: 채점 7일에 평일 공휴일이 있다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 맨 위 설치 셀과 같은 규칙 — 이 셀만 실행해도 정해집니다


def hol_flags(dates):
    """날짜마다 한국 공휴일이면 1, 아니면 0 (holidays 0.103 KR, 주말 공휴일 포함) — 달력이 정하는 값입니다"""
    import holidays
    di = pd.DatetimeIndex(dates)
    kr = holidays.KR(years=sorted(set(di.year)))
    return np.array([1.0 if d in kr else 0.0 for d in di])


HOL = hol_flags(DATES)                        # DATES 의 날마다 공휴일 1 / 0


def origin_ctx(origin):
    """원점 하나의 자료 — 원점의 열 번호(o), 원점 전날까지(hist), 채점 7일 실제(test, 명), 역별 MASE 분모(qden), 채점 7일 날짜(test_dates)"""
    o = int(DATES.get_loc(pd.Timestamp(origin)))
    hist = MAT[:, :o]
    qden = np.nanmean(np.abs(hist[:, 7:] - hist[:, :-7]), axis=1)
    return dict(o=o, hist=hist, test=MAT[:, o:o + H], qden=qden, test_dates=DATES[o:o + H])


def chronos_frames(origin, holiday=True):
    """Chronos-2 의 predict_df 에 전달할 두 표 (df, future_df)

    df        : 100개 역 x 원점 전날까지 512일 — item_id(역 번호 0 ~ 99) · timestamp · target(승차, 명)
                · holiday(공휴일 1 / 0). holiday=False 면 holiday 열이 없습니다
    future_df : 100개 역 x 채점 7일 — item_id · timestamp · holiday(달력이 정한 공휴일). holiday=False 면 None
    """
    ctx = origin_ctx(origin)
    o, S = ctx["o"], MAT.shape[0]
    df = pd.DataFrame({"item_id": np.repeat(np.arange(S), CTX),
                       "timestamp": np.tile(DATES[o - CTX:o].values, S),
                       "target": MAT[:, o - CTX:o].ravel()})
    future_df = None
    if holiday:
        df["holiday"] = np.tile(HOL[o - CTX:o], S)
        future_df = pd.DataFrame({"item_id": np.repeat(np.arange(S), H),
                                  "timestamp": np.tile(ctx["test_dates"].values, S),
                                  "holiday": np.tile(HOL[o:o + H], S)})
    assert df["timestamp"].max() < pd.Timestamp(origin)   # 과거 표에는 원점 전날까지만
    return df, future_df


def load_quantiles(path, holiday):
    """분위수 파일(origin · item_id · timestamp · "0.1"~"0.9")을 원점 → 100 x 7 x 9 배열로 읽습니다.
    「오늘의 준비」 셀이 남긴 파일이 없으면 Chronos-2 로 원점 5개를 예측해 그 파일을 만듭니다"""
    if not os.path.exists(path):
        from chronos import BaseChronosPipeline
        pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE, torch_dtype=torch.float32)
        frames = []
        for o in ORIGINS:
            df, future_df = chronos_frames(o, holiday=holiday)
            fc = pipe.predict_df(df, future_df=future_df, prediction_length=H, quantile_levels=QS)
            frames.append(fc.assign(origin=o)[["origin", "item_id", "timestamp"] + [str(t) for t in QS]])
        pd.concat(frames, ignore_index=True).to_parquet(path)
    d = pd.read_parquet(path).sort_values(["origin", "item_id", "timestamp"])
    return {o: np.stack([g[str(t)].to_numpy().reshape(-1, H) for t in QS], axis=-1)
            for o, g in d.groupby("origin")}


Y_TEST = {o: origin_ctx(o)["test"] for o in ORIGINS}       # 원점 → 100개 역 x 7일 실제 승차 (명)
Q_HOL = load_quantiles("c2_hol_q.parquet", holiday=True)   # 원점 → Chronos-2 + 공휴일 100개 역 x 7일 x 9열 (명)

print(f"자료 : 100개 역 x {MAT.shape[1]:,}일 승차 · 원점 5개 = 공휴일 없는 두 주 {' · '.join(PLAIN_ORIGINS)}"
      f" + 공휴일 있는 세 주 {' · '.join(HOL_ORIGINS)}")
print(f"Q_HOL : 원점마다 {Q_HOL[ORIGINS[0]].shape} (100개 역 x 7일 x 분위수 아홉 열, 명) · "
      f"Y_TEST : 원점마다 {Y_TEST[ORIGINS[0]].shape} (100개 역 x 7일, 명)")
print("쓸 수 있는 이름 : Q_HOL · Y_TEST · ORIGINS · PLAIN_ORIGINS · HOL_ORIGINS · QS (설치 셀의 COL · FIG_W 도 씁니다)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 예시 프롬프트도 따옴표 안에 붙여 넣어야 채점됩니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 미션 2 — 제로샷과 Chronos-2 + 공휴일의 구간 길이

포함률은 구간을 넓히기만 해도 오릅니다. Chronos-2 + 공휴일의 공휴일 있는 세 주 포함률이 제로샷보다 높은 것(41.6% 대 66.4%)이 구간을 넓혀서인지 평균 구간 길이로 확인합니다. 미션 1 과 같은 순서로 합니다.

시킬 일은 미션 1 처럼 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다. 따옴표 안이 비어 있으면 채점과 완성본 풀이가 잠깁니다.

**이 미션의 질문**: Chronos-2 + 공휴일의 공휴일 있는 세 주 포함률이 제로샷보다 높은 것은 구간을 넓혀서일까요?

**쓸 수 있는 것**: 「미션 2 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `Q_ZS` · `Q_HOL` | 원점 → 제로샷 · Chronos-2 + 공휴일의 분위수 배열(100개 역 × 7일 × 9열, 명) |
| `Y_TEST` | 원점 → 채점 7일 실제 승차(100개 역 × 7일, 명) |
| `ORIGINS` · `PLAIN_ORIGINS` · `HOL_ORIGINS` · `QS` | 미션 1 과 같은 원점 목록과 분위수 아홉 개 |
| `COL` · `FIG_W` | 그래프 색 사전과 그림 가로 크기(설치 셀) |

#### 만들 것

굵은 이름은 채점이 찾으니 글자 그대로 쓰고, 모두 「PROMPT 셀」에서 출력합니다.

1. **Chronos-2 + 공휴일의 공휴일 있는 세 주 평균 구간 길이**: 단위 명 · 정수. 구간 길이는 역·날마다 0.9 열에서 0.1 열을 뺀 값입니다. 원점마다 700점으로 평균한 뒤 `HOL_ORIGINS` 세 원점을 평균합니다. 천 명으로 나누지 않고 명 단위 그대로 적습니다
2. **공휴일 있는 세 주 구간 길이 비**: 단위 배 · 소수 둘째 자리. 공휴일 있는 세 주에서 Chronos-2 + 공휴일의 평균 구간 길이를 제로샷의 값으로 나눕니다
3. **그래프 한 장**: 종류는 자유이고 가로축 「채점한 주」, 세로축 「평균 구간 길이 (명)」 이름을 답니다. 권장은 공휴일 유무 × 두 모델의 막대 4개에 범례 두 모델입니다

**통과 기준**: 1·2번 값이 「이름 = 값 단위」 한 행씩 있고 기준 범위 안이며, 그래프에 가로축·세로축 이름이 있으면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **구간 길이**: `q[..., -1] - q[..., 0]`(0.9 열 − 0.1 열)을 원점마다 평균한 뒤 공휴일 없는 두 주와 공휴일 있는 세 주로 나눠 평균하라고 적습니다. 빼는 순서가 거꾸로면 음수가 나옵니다.
- **비의 방향**: Chronos-2 + 공휴일을 제로샷으로 나눈다고 분자와 분모를 적습니다.
- **평균할 원점**: 두 값 모두 공휴일 있는 세 주의 원점 3개로만 평균하라고 적습니다.
- **막대 차례**: 두 주와 세 주마다 두 모델의 막대를 나란히 두라고 적습니다.

<details class="lms-extra"><summary>배경 — 80% 구간과 95% 구간의 길이</summary>

<img src="https://otexts.com/fpp3/toolbox_files/figure-html/fig-googforecasts2-1.png" alt="구글 일별 종가 선 그래프 끝에 오른쪽으로 갈수록 넓어지는 두 겹의 예측구간을 칠한 그림" width="600" height="330"/>
<p align="center"><em>▲ 구글 일별 종가의 나이브 예측과 두 예측구간. 진한 파란 영역이 80% 구간, 연한 영역이 95% 구간이다. 명목 포함률을 높이면 구간이 길어지고, 며칠 뒤를 예측할수록 두 구간 모두 길어진다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3판, 그림 5.14)</em></p>

</details>

#### 예상 적기

> **[예측] 공휴일 있는 세 주에서 Chronos-2 + 공휴일 구간은 제로샷보다 길까요?**
>
> 1. 길다. 평일 공휴일이 있는 주에는 승차가 얼마나 줄지 몰라 구간을 넓힌다
> 2. 비슷하다. 같은 Chronos-2 라 구간 길이도 비슷하다
> 3. 짧다. 공휴일 열을 받아 예측이 더 정확해진다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>예상 확인 — 미션 2 결과를 본 뒤 펼칩니다</summary>

답은 첫째 보기입니다. 공휴일 있는 세 주에서 Chronos-2 + 공휴일의 평균 구간 길이는 5,734명으로, 제로샷 3,768명의 5,734 ÷ 3,768 = 1.52배입니다. 원점별로는 2025-08-13 1.13배, 2025-10-01 1.63배, 2026-02-11 1.92배입니다. 1.52배는 세 주 평균끼리 나눈 값이라, 원점별 비 셋의 평균 1.56 과 조금 다릅니다.

- 「비슷하다」: 두 모델의 구간 길이가 비슷한 것은 공휴일 없는 두 주(2,839명 대 3,082명, 0.92배)입니다.
- 「짧다」: 0.92배로 조금 짧아진 것도 공휴일 없는 두 주입니다. 평일 공휴일에 승차가 얼마나 줄지는 날과 역마다 달라, 공휴일 있는 세 주에서는 구간이 길어졌습니다.

그래서 공휴일 있는 세 주의 66.4% 는 구간을 1.52배로 넓혀 얻은 값이고, 넓힌 구간으로도 명목 80% 에는 못 미칩니다.

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다
#  「오늘의 준비」 셀이 남긴 두 모델의 분위수 파일을 읽고, 파일이 없으면 Chronos-2 로 다시 예측합니다 (T4 약 20초)
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

RAW_URL = "https://github.com/studio-js/seoul-subway-ridership/raw/main/data/seoul_subway_daily_2015_2026.parquet"
CACHE = "seoul_subway_daily.parquet"          # 한 번 받아 두면 다시 받지 않습니다
if not os.path.exists(CACHE):
    pd.read_parquet(RAW_URL).to_parquet(CACHE)

# 지금까지와 같은 손질 — 역 이름의 괄호 별칭을 떼고 9호선 이름을 맞춘 뒤 (날짜, 호선, 역) 으로 합칩니다
_raw = pd.read_parquet(CACHE)
_raw["date"] = pd.to_datetime(_raw["date"])
_raw["station"] = _raw["station"].astype(str).str.replace(r"\(.*\)$", "", regex=True)
_raw["line"] = _raw["line"].astype(str).replace({"9호선2단계": "9호선(2~3단계)",
                                                "9호선2~3단계": "9호선(2~3단계)"})
_clean = _raw.groupby(["date", "line", "station"], as_index=False,
                      observed=True)[["board", "alight"]].sum()
_span = _clean.groupby(["line", "station"])["date"].agg(["min", "max", "count"])
_full = _span[(_span["min"] <= pd.Timestamp("2015-01-01")) &            # 기록이 끊기지 않은 역만 남긴 뒤
              (_span["max"] >= pd.Timestamp("2026-06-30")) & (_span["count"] >= 4190)]
_top = (_clean[_clean["date"] <= pd.Timestamp("2024-12-31")]             # 2024년까지 평균 승차 상위 100개 역
        .groupby(["line", "station"])["board"].mean()
        .loc[_full.index].sort_values(ascending=False).index[:100])
_wide = _clean.pivot_table(index=["line", "station"], columns="date", values="board").loc[_top]
MAT = _wide.to_numpy(dtype=float)             # 100개 역 x 4,199일 승차 (명)
DATES = pd.DatetimeIndex(_wide.columns)       # 행렬 열의 날짜
STATIONS = [f"{_line} {_name}" for _line, _name in _wide.index]   # 행렬 행의 역 이름

H, CTX = 7, 512                               # 채점 7일 · Chronos-2 에 주는 과거 512일
QS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # 분위수 아홉 열 (80% 구간의 두 끝 = 0.1 열 · 0.9 열)
ORIGINS = ["2025-08-13", "2025-10-01", "2025-12-10", "2026-02-11", "2026-06-24"]   # 예측 원점 5개 (수요일)
PLAIN_ORIGINS = ["2025-12-10", "2026-06-24"]              # 공휴일 없는 두 주: 채점 7일에 평일 공휴일이 없다
HOL_ORIGINS = ["2025-08-13", "2025-10-01", "2026-02-11"]  # 공휴일 있는 세 주: 채점 7일에 평일 공휴일이 있다
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # 맨 위 설치 셀과 같은 규칙 — 이 셀만 실행해도 정해집니다


def hol_flags(dates):
    """날짜마다 한국 공휴일이면 1, 아니면 0 (holidays 0.103 KR, 주말 공휴일 포함) — 달력이 정하는 값입니다"""
    import holidays
    di = pd.DatetimeIndex(dates)
    kr = holidays.KR(years=sorted(set(di.year)))
    return np.array([1.0 if d in kr else 0.0 for d in di])


HOL = hol_flags(DATES)                        # DATES 의 날마다 공휴일 1 / 0


def origin_ctx(origin):
    """원점 하나의 자료 — 원점의 열 번호(o), 원점 전날까지(hist), 채점 7일 실제(test, 명), 역별 MASE 분모(qden), 채점 7일 날짜(test_dates)"""
    o = int(DATES.get_loc(pd.Timestamp(origin)))
    hist = MAT[:, :o]
    qden = np.nanmean(np.abs(hist[:, 7:] - hist[:, :-7]), axis=1)
    return dict(o=o, hist=hist, test=MAT[:, o:o + H], qden=qden, test_dates=DATES[o:o + H])


def chronos_frames(origin, holiday=True):
    """Chronos-2 의 predict_df 에 전달할 두 표 (df, future_df)

    df        : 100개 역 x 원점 전날까지 512일 — item_id(역 번호 0 ~ 99) · timestamp · target(승차, 명)
                · holiday(공휴일 1 / 0). holiday=False 면 holiday 열이 없습니다
    future_df : 100개 역 x 채점 7일 — item_id · timestamp · holiday(달력이 정한 공휴일). holiday=False 면 None
    """
    ctx = origin_ctx(origin)
    o, S = ctx["o"], MAT.shape[0]
    df = pd.DataFrame({"item_id": np.repeat(np.arange(S), CTX),
                       "timestamp": np.tile(DATES[o - CTX:o].values, S),
                       "target": MAT[:, o - CTX:o].ravel()})
    future_df = None
    if holiday:
        df["holiday"] = np.tile(HOL[o - CTX:o], S)
        future_df = pd.DataFrame({"item_id": np.repeat(np.arange(S), H),
                                  "timestamp": np.tile(ctx["test_dates"].values, S),
                                  "holiday": np.tile(HOL[o:o + H], S)})
    assert df["timestamp"].max() < pd.Timestamp(origin)   # 과거 표에는 원점 전날까지만
    return df, future_df


def load_quantiles(path, holiday):
    """분위수 파일(origin · item_id · timestamp · "0.1"~"0.9")을 원점 → 100 x 7 x 9 배열로 읽습니다.
    「오늘의 준비」 셀이 남긴 파일이 없으면 Chronos-2 로 원점 5개를 예측해 그 파일을 만듭니다"""
    if not os.path.exists(path):
        from chronos import BaseChronosPipeline
        pipe = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE, torch_dtype=torch.float32)
        frames = []
        for o in ORIGINS:
            df, future_df = chronos_frames(o, holiday=holiday)
            fc = pipe.predict_df(df, future_df=future_df, prediction_length=H, quantile_levels=QS)
            frames.append(fc.assign(origin=o)[["origin", "item_id", "timestamp"] + [str(t) for t in QS]])
        pd.concat(frames, ignore_index=True).to_parquet(path)
    d = pd.read_parquet(path).sort_values(["origin", "item_id", "timestamp"])
    return {o: np.stack([g[str(t)].to_numpy().reshape(-1, H) for t in QS], axis=-1)
            for o, g in d.groupby("origin")}


Y_TEST = {o: origin_ctx(o)["test"] for o in ORIGINS}       # 원점 → 100개 역 x 7일 실제 승차 (명)
Q_ZS = load_quantiles("c2_zs_q.parquet", holiday=False)    # 원점 → 제로샷 100개 역 x 7일 x 9열 (명)
Q_HOL = load_quantiles("c2_hol_q.parquet", holiday=True)   # 원점 → Chronos-2 + 공휴일 100개 역 x 7일 x 9열 (명)

print(f"자료 : 100개 역 x {MAT.shape[1]:,}일 승차 · 원점 5개 = 공휴일 없는 두 주 {' · '.join(PLAIN_ORIGINS)}"
      f" + 공휴일 있는 세 주 {' · '.join(HOL_ORIGINS)}")
print(f"Q_ZS · Q_HOL : 원점마다 {Q_HOL[ORIGINS[0]].shape} (100개 역 x 7일 x 분위수 아홉 열, 명) · "
      f"Y_TEST : 원점마다 {Y_TEST[ORIGINS[0]].shape} (100개 역 x 7일, 명)")
print("쓸 수 있는 이름 : Q_ZS · Q_HOL · Y_TEST · ORIGINS · PLAIN_ORIGINS · HOL_ORIGINS · QS (설치 셀의 COL · FIG_W 도 씁니다)")

In [ ]:
# 미션 2 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 따옴표 안이 비어 있으면 채점과 완성본 풀이가 잠깁니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 오늘의 결론

<div class="lms-summary">

**정리**

- 포함률: Chronos-2 + 공휴일은 공휴일 없는 두 주 79.5%, 공휴일 있는 세 주 66.4% 입니다. 제로샷은 85.1%·41.6% 입니다. 66.4% 는 명목 80% 보다 13.6%p 낮고 41.6% 는 그보다 더 낮아, 둘 다 명목 80% 로 보지 않습니다. 전달할 구간은 쓸 수 있는 두 구간끼리 비교해 정합니다
- 평균 구간 길이: Chronos-2 + 공휴일은 공휴일 있는 세 주 5,734명으로 제로샷 3,768명의 1.52배입니다. 공휴일 없는 두 주는 2,839명으로 제로샷의 0.92배입니다
- 할 일 · 평가표: `Q_HOL` 로 원점마다 포함률과 평균 구간 길이를 구해 공휴일 없는 두 주와 공휴일 있는 세 주로 평균해 적습니다. 100개 역 평균 구간 길이는 두 모델을 비교할 때만 씁니다
- 할 일 · 계획표: 역·날마다 두 끝과 그날 구간 길이를 적습니다(강남역 2월 12일 98,148 − 88,339 = 9,809명). 인력을 위 끝(0.9 열)에 맞춘다고 가정하면, 아래 끝까지는 줄여도 된다고 보는 범위가 그날 구간 길이입니다. 어느 열에 맞출지는 임계비율을 배울 때 정합니다
- 할 일 · 두 끝 옆의 포함률: 구간을 쓰는 날의 포함률을 적습니다. 공휴일 없는 주는 79.5% 이고, 위 끝보다 많은 점 18.9% 도 구간 옆에 적습니다. 평일 공휴일이 있는 주는 74.4% 입니다. 공휴일 있는 세 주 2,100점에서 당일 600점(6일 × 100개 역)을 뺀 1,500점이 남습니다. 그 가운데 1,116점이 구간 안입니다. 당일 인력은 구간과 따로 정합니다

Chronos-2 + 공휴일 80% 구간은 평일 공휴일이 없는 주 인력 계획에 그대로 씁니다. 다만 위 끝을 넘는 일이 다섯 번에 한 번(18.9%)이었다고 구간 옆에 함께 적습니다. 평일 공휴일이 있는 주에는 이 구간의 포함률이 제로샷 구간보다 높아(66.4% 대 41.6%), 실제 포함률과 함께 전달합니다.

</div>

> **[과제] 실습 제출 — 80% 구간 포함률과 구간 길이**
>
> 미션 1·2 의 「PROMPT 셀」이 채점을 통과한 상태로 노트북을 제출합니다. PROMPT 는 내가 쓴 그대로 둡니다.
>
> 마감: 2026-10-22T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 미션 1 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
> | 미션 2 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 딥러닝 시계열 넷째 날 완료</h2>
<div style="color:#656565">Chronos-2 두 모델의 80% 구간에서 포함률과 평균 구간 길이를 공휴일 유무로 나눠 구하고, 인력 계획에 쓸지 정했습니다.</div>
</div>